# Full Fine-tuning Notebook (CV Scoring)

Дообучение `unsloth/Llama-3.2-3B-Instruct` на полном датасете,
который генерируется прямо в ноутбуке (без чтения файлов по путям).


In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install trl peft accelerate bitsandbytes datasets

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-vav8mgu8/unsloth_0c9b2aa945d542b7b58e1ec0d80281ac
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-vav8mgu8/unsloth_0c9b2aa945d542b7b58e1ec0d80281ac
  Resolved https://github.com/unslothai/unsloth.git to commit efed5c37394a144349cd9b1ea525e132e04584e5
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 116.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 123.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 21.6 MB/s eta 0:00:0

In [2]:
import os
import json
import random
from pathlib import Path

import torch
from datasets import Dataset

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("GPU not found. Use Colab T4/A100 for training.")

random.seed(42)

GPU: Tesla T4 | VRAM: 15.6 GB


In [3]:
DATA_DIR = {"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 2 лет, навыки AsyncIO, Django, FastAPI, PostgreSQL, зарплата 114935-199320, локация Казань, формат Нет, занятость Полная. Вакансия: Python Backend Developer, опыт 2-4 лет, навыки FastAPI, Kafka, Python, Redis, зарплата 228810-308262, локация Казань, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/4): FastAPI. Недостающие навыки: Kafka, Python, Redis.\", \"experience_match\": \"Опыт кандидата 2 лет находится в диапазоне вакансии (2-4 лет).\", \"salary_match\": \"Ожидания кандидата (114935-199320 руб.) не пересекаются с вилкой вакансии (228810-308262 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: FastAPI\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Kafka, Python, Redis\"], \"recommendation\": \"consider\"}, \"confidence\": 0.77, \"match_score\": 56}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 4 лет, навыки A/B testing, Power BI, Python, SQL, Statistics, Tableau, зарплата 185980-218705, локация Регионы, формат Гибрид, занятость Частичная. Вакансия: Data Analyst, опыт 4-5 лет, навыки A/B testing, Product metrics, Python, SQL, зарплата 203931-251614, локация Москва, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): A/B testing, Python, SQL. Недостающие навыки: Product metrics.\", \"experience_match\": \"Опыт кандидата 4 лет находится в диапазоне вакансии (4-5 лет).\", \"salary_match\": \"Ожидания кандидата (185980-218705 руб.) пересекаются с вилкой вакансии (203931-251614 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: A/B testing, Python, SQL\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Product metrics\"], \"recommendation\": \"hire\"}, \"confidence\": 0.9, \"match_score\": 88}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 6 лет, навыки Pandas, Python, SQL, Tableau, зарплата 155515-190415, локация Казань, формат Гибрид, занятость Полная. Вакансия: Python Backend Developer, опыт 1-2 лет, навыки Docker, Kafka, PostgreSQL, Redis, зарплата 173281-281144, локация Москва, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: Docker, Kafka, PostgreSQL, Redis.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (2 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (155515-190415 руб.) пересекаются с вилкой вакансии (173281-281144 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Docker, Kafka, PostgreSQL, Redis\"], \"recommendation\": \"consider\"}, \"confidence\": 0.75, \"match_score\": 51}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 2 лет, навыки Airflow, Docker, MLflow, Pandas, PyTorch, Python, TensorFlow, зарплата 151750-197797, локация Москва, формат Нет, занятость Частичная. Вакансия: Data Analyst, опыт 5-7 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 261166-310320, локация Москва, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/5): Python. Недостающие навыки: A/B testing, Product metrics, SQL, Tableau.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (5-7 лет).\", \"salary_match\": \"Ожидания кандидата (151750-197797 руб.) не пересекаются с вилкой вакансии (261166-310320 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: Python\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, SQL, Tableau\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.71, \"match_score\": 40}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 8 лет, навыки Airflow, Docker, MLflow, NumPy, Pandas, Python, TensorFlow, зарплата 141531-214670, локация Москва, формат Да, занятость Частичная. Вакансия: Python Backend Developer, опыт 5-6 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 279423-342982, локация Санкт-Петербург, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/6): Docker, Python. Недостающие навыки: FastAPI, Kafka, PostgreSQL, Redis.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (141531-214670 руб.) не пересекаются с вилкой вакансии (279423-342982 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker, Python\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: FastAPI, Kafka, PostgreSQL, Redis\"], \"recommendation\": \"consider\"}, \"confidence\": 0.78, \"match_score\": 58}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 4 лет, навыки AWS, Docker, GitLab CI, Linux, Terraform, зарплата 292365-361722, локация Регионы, формат Гибрид, занятость Полная. Вакансия: DevOps Engineer, опыт 2-4 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Terraform, зарплата 308901-391085, локация Екатеринбург, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): AWS, Docker, GitLab CI, Terraform. Недостающие навыки: Kubernetes.\", \"experience_match\": \"Опыт кандидата 4 лет находится в диапазоне вакансии (2-4 лет).\", \"salary_match\": \"Ожидания кандидата (292365-361722 руб.) пересекаются с вилкой вакансии (308901-391085 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: AWS, Docker, GitLab CI, Terraform\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Kubernetes\"], \"recommendation\": \"hire\"}, \"confidence\": 0.91, \"match_score\": 89}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 8 лет, навыки CSS, JavaScript, Next.js, React, Redux, TypeScript, Vue, зарплата 132541-203230, локация Санкт-Петербург, формат Да, занятость Полная. Вакансия: Frontend Developer, опыт 4-6 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 163882-275702, локация Регионы, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (5/6): CSS, Next.js, React, Redux, TypeScript. Недостающие навыки: REST API.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (132541-203230 руб.) пересекаются с вилкой вакансии (163882-275702 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: CSS, Next.js, React, Redux, TypeScript\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: REST API\"], \"recommendation\": \"hire\"}, \"confidence\": 0.91, \"match_score\": 89}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 2 лет, навыки AsyncIO, Django, Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 241929-288584, локация Москва, формат Да, занятость Частичная. Вакансия: Python Backend Developer, опыт 3-6 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Redis, зарплата 223497-298279, локация Москва, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): Docker, FastAPI, PostgreSQL, Redis. Недостающие навыки: Kafka.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (3-6 лет).\", \"salary_match\": \"Ожидания кандидата (241929-288584 руб.) пересекаются с вилкой вакансии (223497-298279 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: Docker, FastAPI, PostgreSQL, Redis\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Kafka\", \"Недостаточный опыт\"], \"recommendation\": \"hire\"}, \"confidence\": 0.87, \"match_score\": 80}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 9 лет, навыки A/B testing, Pandas, Python, Statistics, зарплата 223425-274236, локация Москва, формат Гибрид, занятость Полная. Вакансия: Data Analyst, опыт 2-3 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 124867-179463, локация Москва, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): A/B testing, Python. Недостающие навыки: Product metrics, SQL, Tableau.\", \"experience_match\": \"Опыт кандидата 9 лет выше верхней границы (3 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (223425-274236 руб.) не пересекаются с вилкой вакансии (124867-179463 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: A/B testing, Python\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Product metrics, SQL, Tableau\"], \"recommendation\": \"consider\"}, \"confidence\": 0.79, \"match_score\": 61}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 6 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 125683-188275, локация Новосибирск, формат Да, занятость Частичная. Вакансия: ML Engineer, опыт 4-5 лет, навыки Docker, MLOps, OpenCV, Python, зарплата 252821-313571, локация Новосибирск, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/4): Docker. Недостающие навыки: MLOps, OpenCV, Python.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (125683-188275 руб.) не пересекаются с вилкой вакансии (252821-313571 руб.).\", \"location_match\": \"Локация совпадает: Новосибирск.\", \"strengths\": [\"Совпавшие навыки: Docker\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: MLOps, OpenCV, Python\"], \"recommendation\": \"consider\"}, \"confidence\": 0.77, \"match_score\": 56}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 8 лет, навыки HTML, Next.js, React, TypeScript, Vue, зарплата 104448-140354, локация Регионы, формат Да, занятость Частичная. Вакансия: Frontend Developer, опыт 1-3 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 177361-260184, локация Регионы, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): Next.js, React, TypeScript. Недостающие навыки: CSS, REST API, Redux.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (3 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (104448-140354 руб.) не пересекаются с вилкой вакансии (177361-260184 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: Next.js, React, TypeScript\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: CSS, REST API, Redux\"], \"recommendation\": \"consider\"}, \"confidence\": 0.82, \"match_score\": 68}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 6 лет, навыки CSS, HTML, Next.js, React, Redux, TypeScript, Vue, зарплата 188166-267560, локация Казань, формат Да, занятость Частичная. Вакансия: Data Analyst, опыт 6-8 лет, навыки A/B testing, Python, SQL, Tableau, зарплата 143064-188668, локация Регионы, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: A/B testing, Python, SQL, Tableau.\", \"experience_match\": \"Опыт кандидата 6 лет находится в диапазоне вакансии (6-8 лет).\", \"salary_match\": \"Ожидания кандидата (188166-267560 руб.) пересекаются с вилкой вакансии (143064-188668 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Python, SQL, Tableau\"], \"recommendation\": \"consider\"}, \"confidence\": 0.76, \"match_score\": 53}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 8 лет, навыки Airflow, Docker, MLflow, NumPy, Pandas, TensorFlow, зарплата 278641-352638, локация Казань, формат Да, занятость Частичная. Вакансия: Data Analyst, опыт 1-3 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 239917-348659, локация Санкт-Петербург, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/5): нет. Недостающие навыки: A/B testing, Product metrics, Python, SQL, Tableau.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (3 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (278641-352638 руб.) пересекаются с вилкой вакансии (239917-348659 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, Python, SQL, Tableau\"], \"recommendation\": \"consider\"}, \"confidence\": 0.75, \"match_score\": 51}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 5 лет, навыки AsyncIO, Django, FastAPI, PostgreSQL, Python, Redis, зарплата 277743-356227, локация Москва, формат Нет, занятость Частичная. Вакансия: Frontend Developer, опыт 6-7 лет, навыки CSS, Next.js, REST API, Redux, TypeScript, зарплата 262377-336604, локация Москва, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/5): нет. Недостающие навыки: CSS, Next.js, REST API, Redux, TypeScript.\", \"experience_match\": \"Опыт кандидата 5 лет ниже диапазона вакансии (6-7 лет).\", \"salary_match\": \"Ожидания кандидата (277743-356227 руб.) пересекаются с вилкой вакансии (262377-336604 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: CSS, Next.js, REST API, Redux, TypeScript\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.74, \"match_score\": 48}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 4 лет, навыки A/B testing, Pandas, Power BI, Python, SQL, Statistics, Tableau, зарплата 141442-197387, локация Регионы, формат Нет, занятость Полная. Вакансия: Data Analyst, опыт 2-4 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 167303-247410, локация Санкт-Петербург, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): A/B testing, Python, SQL, Tableau. Недостающие навыки: Product metrics.\", \"experience_match\": \"Опыт кандидата 4 лет находится в диапазоне вакансии (2-4 лет).\", \"salary_match\": \"Ожидания кандидата (141442-197387 руб.) пересекаются с вилкой вакансии (167303-247410 руб.).\", \"location_match\": \"Локация не совпадает (Регионы vs Санкт-Петербург).\", \"strengths\": [\"Совпавшие навыки: A/B testing, Python, SQL, Tableau\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Product metrics\", \"Несовпадение по локации/формату\"], \"recommendation\": \"hire\"}, \"confidence\": 0.89, \"match_score\": 85}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 2 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Terraform, зарплата 144794-189678, локация Москва, формат Да, занятость Частичная. Вакансия: Python Backend Developer, опыт 6-7 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 269913-324479, локация Екатеринбург, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/6): Docker. Недостающие навыки: FastAPI, Kafka, PostgreSQL, Python, Redis.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (6-7 лет).\", \"salary_match\": \"Ожидания кандидата (144794-189678 руб.) не пересекаются с вилкой вакансии (269913-324479 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker\"], \"weaknesses\": [\"Недостающие навыки: FastAPI, Kafka, PostgreSQL, Python, Redis\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.7, \"match_score\": 37}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 4 лет, навыки CSS, HTML, JavaScript, Next.js, React, TypeScript, Vue, зарплата 151218-216502, локация Санкт-Петербург, формат Гибрид, занятость Полная. Вакансия: Frontend Developer, опыт 6-8 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 179865-263188, локация Санкт-Петербург, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/6): CSS, Next.js, React, TypeScript. Недостающие навыки: REST API, Redux.\", \"experience_match\": \"Опыт кандидата 4 лет ниже диапазона вакансии (6-8 лет).\", \"salary_match\": \"Ожидания кандидата (151218-216502 руб.) пересекаются с вилкой вакансии (179865-263188 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: CSS, Next.js, React, TypeScript\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: REST API, Redux\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 74}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 1 лет, навыки CSS, HTML, JavaScript, React, Redux, TypeScript, Vue, зарплата 117254-161291, локация Санкт-Петербург, формат Да, занятость Полная. Вакансия: Frontend Developer, опыт 3-5 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 189583-293319, локация Санкт-Петербург, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/6): CSS, React, Redux, TypeScript. Недостающие навыки: Next.js, REST API.\", \"experience_match\": \"Опыт кандидата 1 лет ниже диапазона вакансии (3-5 лет).\", \"salary_match\": \"Ожидания кандидата (117254-161291 руб.) не пересекаются с вилкой вакансии (189583-293319 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: CSS, React, Redux, TypeScript\"], \"weaknesses\": [\"Недостающие навыки: Next.js, REST API\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.78, \"match_score\": 58}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 1 лет, навыки CSS, Next.js, React, Redux, Vue, зарплата 88575-177234, локация Казань, формат Гибрид, занятость Частичная. Вакансия: Frontend Developer, опыт 4-5 лет, навыки CSS, REST API, React, TypeScript, зарплата 161682-259985, локация Казань, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): CSS, React. Недостающие навыки: REST API, TypeScript.\", \"experience_match\": \"Опыт кандидата 1 лет ниже диапазона вакансии (4-5 лет).\", \"salary_match\": \"Ожидания кандидата (88575-177234 руб.) пересекаются с вилкой вакансии (161682-259985 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: CSS, React\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: REST API, TypeScript\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.77, \"match_score\": 56}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 7 лет, навыки CSS, HTML, JavaScript, Next.js, React, TypeScript, Vue, зарплата 90397-167761, локация Казань, формат Да, занятость Частичная. Вакансия: Data Analyst, опыт 1-2 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 137910-220342, локация Санкт-Петербург, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/5): нет. Недостающие навыки: A/B testing, Product metrics, Python, SQL, Tableau.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (2 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (90397-167761 руб.) пересекаются с вилкой вакансии (137910-220342 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, Python, SQL, Tableau\"], \"recommendation\": \"consider\"}, \"confidence\": 0.75, \"match_score\": 51}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 8 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 269145-343604, локация Регионы, формат Нет, занятость Полная. Вакансия: DevOps Engineer, опыт 6-8 лет, навыки ArgoCD, Docker, GitLab CI, Kubernetes, зарплата 264861-328619, локация Регионы, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): Docker, GitLab CI, Kubernetes. Недостающие навыки: ArgoCD.\", \"experience_match\": \"Опыт кандидата 8 лет находится в диапазоне вакансии (6-8 лет).\", \"salary_match\": \"Ожидания кандидата (269145-343604 руб.) пересекаются с вилкой вакансии (264861-328619 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: Docker, GitLab CI, Kubernetes\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD\"], \"recommendation\": \"hire\"}, \"confidence\": 0.9, \"match_score\": 88}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 4 лет, навыки AWS, Docker, Linux, Terraform, зарплата 198996-283545, локация Екатеринбург, формат Нет, занятость Частичная. Вакансия: DevOps Engineer, опыт 5-6 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 326439-438364, локация Екатеринбург, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): AWS, Docker, Terraform. Недостающие навыки: ArgoCD, GitLab CI, Kubernetes.\", \"experience_match\": \"Опыт кандидата 4 лет ниже диапазона вакансии (5-6 лет).\", \"salary_match\": \"Ожидания кандидата (198996-283545 руб.) не пересекаются с вилкой вакансии (326439-438364 руб.).\", \"location_match\": \"Локация совпадает: Екатеринбург.\", \"strengths\": [\"Совпавшие навыки: AWS, Docker, Terraform\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, GitLab CI, Kubernetes\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.8, \"match_score\": 62}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 1 лет, навыки A/B testing, Pandas, Power BI, Python, SQL, Statistics, зарплата 233253-307451, локация Санкт-Петербург, формат Гибрид, занятость Полная. Вакансия: Frontend Developer, опыт 1-4 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 243908-290857, локация Регионы, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/6): нет. Недостающие навыки: CSS, Next.js, REST API, React, Redux, TypeScript.\", \"experience_match\": \"Опыт кандидата 1 лет находится в диапазоне вакансии (1-4 лет).\", \"salary_match\": \"Ожидания кандидата (233253-307451 руб.) пересекаются с вилкой вакансии (243908-290857 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: CSS, Next.js, REST API, React, Redux\"], \"recommendation\": \"consider\"}, \"confidence\": 0.77, \"match_score\": 55}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 8 лет, навыки AsyncIO, Django, FastAPI, PostgreSQL, Python, Redis, зарплата 214781-248458, локация Санкт-Петербург, формат Нет, занятость Частичная. Вакансия: Python Backend Developer, опыт 5-8 лет, навыки Docker, FastAPI, Kafka, Python, Redis, зарплата 286919-328157, локация Санкт-Петербург, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): FastAPI, Python, Redis. Недостающие навыки: Docker, Kafka.\", \"experience_match\": \"Опыт кандидата 8 лет находится в диапазоне вакансии (5-8 лет).\", \"salary_match\": \"Ожидания кандидата (214781-248458 руб.) не пересекаются с вилкой вакансии (286919-328157 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: FastAPI, Python, Redis\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Docker, Kafka\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 71}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 8 лет, навыки AWS, Docker, Linux, Terraform, зарплата 282752-341890, локация Санкт-Петербург, формат Нет, занятость Частичная. Вакансия: Python Backend Developer, опыт 5-8 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 198738-270608, локация Санкт-Петербург, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/6): Docker. Недостающие навыки: FastAPI, Kafka, PostgreSQL, Python, Redis.\", \"experience_match\": \"Опыт кандидата 8 лет находится в диапазоне вакансии (5-8 лет).\", \"salary_match\": \"Ожидания кандидата (282752-341890 руб.) не пересекаются с вилкой вакансии (198738-270608 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: Docker\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: FastAPI, Kafka, PostgreSQL, Python, Redis\"], \"recommendation\": \"consider\"}, \"confidence\": 0.77, \"match_score\": 54}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 5 лет, навыки JavaScript, React, Redux, Vue, зарплата 185879-258496, локация Регионы, формат Нет, занятость Полная. Вакансия: DevOps Engineer, опыт 4-6 лет, навыки AWS, ArgoCD, Docker, Kubernetes, Terraform, зарплата 136817-193065, локация Санкт-Петербург, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/5): нет. Недостающие навыки: AWS, ArgoCD, Docker, Kubernetes, Terraform.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (4-6 лет).\", \"salary_match\": \"Ожидания кандидата (185879-258496 руб.) пересекаются с вилкой вакансии (136817-193065 руб.).\", \"location_match\": \"Локация не совпадает (Регионы vs Санкт-Петербург).\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: AWS, ArgoCD, Docker, Kubernetes, Terraform\", \"Несовпадение по локации/формату\"], \"recommendation\": \"reject\"}, \"confidence\": 0.74, \"match_score\": 47}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 6 лет, навыки CSS, HTML, Next.js, React, Redux, TypeScript, Vue, зарплата 80553-159982, локация Санкт-Петербург, формат Да, занятость Частичная. Вакансия: Frontend Developer, опыт 2-3 лет, навыки CSS, Next.js, REST API, React, Redux, зарплата 108134-180545, локация Санкт-Петербург, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): CSS, Next.js, React, Redux. Недостающие навыки: REST API.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (3 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (80553-159982 руб.) пересекаются с вилкой вакансии (108134-180545 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: CSS, Next.js, React, Redux\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: REST API\"], \"recommendation\": \"hire\"}, \"confidence\": 0.9, \"match_score\": 87}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 9 лет, навыки A/B testing, Pandas, Power BI, Python, SQL, Statistics, Tableau, зарплата 148945-210263, локация Регионы, формат Да, занятость Полная. Вакансия: ML Engineer, опыт 3-6 лет, навыки Docker, MLOps, OpenCV, PyTorch, зарплата 127516-191831, локация Москва, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: Docker, MLOps, OpenCV, PyTorch.\", \"experience_match\": \"Опыт кандидата 9 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (148945-210263 руб.) пересекаются с вилкой вакансии (127516-191831 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Docker, MLOps, OpenCV, PyTorch\"], \"recommendation\": \"consider\"}, \"confidence\": 0.75, \"match_score\": 51}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 4 лет, навыки GitLab CI, Kubernetes, Prometheus, Terraform, зарплата 123584-189776, локация Санкт-Петербург, формат Нет, занятость Полная. Вакансия: DevOps Engineer, опыт 4-7 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Terraform, зарплата 256952-309169, локация Москва, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): GitLab CI, Terraform. Недостающие навыки: AWS, ArgoCD, Docker.\", \"experience_match\": \"Опыт кандидата 4 лет находится в диапазоне вакансии (4-7 лет).\", \"salary_match\": \"Ожидания кандидата (123584-189776 руб.) не пересекаются с вилкой вакансии (256952-309169 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: GitLab CI, Terraform\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: AWS, ArgoCD, Docker\"], \"recommendation\": \"consider\"}, \"confidence\": 0.8, \"match_score\": 62}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 7 лет, навыки HTML, JavaScript, Next.js, React, TypeScript, зарплата 182970-224152, локация Санкт-Петербург, формат Нет, занятость Полная. Вакансия: Frontend Developer, опыт 6-8 лет, навыки CSS, REST API, React, Redux, TypeScript, зарплата 146242-235348, локация Санкт-Петербург, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): React, TypeScript. Недостающие навыки: CSS, REST API, Redux.\", \"experience_match\": \"Опыт кандидата 7 лет находится в диапазоне вакансии (6-8 лет).\", \"salary_match\": \"Ожидания кандидата (182970-224152 руб.) пересекаются с вилкой вакансии (146242-235348 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: React, TypeScript\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: CSS, REST API, Redux\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 71}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 2 лет, навыки Docker, MLflow, NumPy, Pandas, PyTorch, Python, TensorFlow, зарплата 111774-167706, локация Казань, формат Нет, занятость Полная. Вакансия: ML Engineer, опыт 2-4 лет, навыки Docker, MLOps, OpenCV, PyTorch, зарплата 227088-316880, локация Казань, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): Docker, PyTorch. Недостающие навыки: MLOps, OpenCV.\", \"experience_match\": \"Опыт кандидата 2 лет находится в диапазоне вакансии (2-4 лет).\", \"salary_match\": \"Ожидания кандидата (111774-167706 руб.) не пересекаются с вилкой вакансии (227088-316880 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: Docker, PyTorch\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: MLOps, OpenCV\"], \"recommendation\": \"consider\"}, \"confidence\": 0.82, \"match_score\": 67}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 9 лет, навыки AsyncIO, Django, FastAPI, PostgreSQL, Python, Redis, зарплата 240756-291092, локация Екатеринбург, формат Да, занятость Полная. Вакансия: ML Engineer, опыт 3-6 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, зарплата 274007-340611, локация Екатеринбург, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/5): нет. Недостающие навыки: Albumentations, Docker, MLOps, OpenCV, PyTorch.\", \"experience_match\": \"Опыт кандидата 9 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (240756-291092 руб.) пересекаются с вилкой вакансии (274007-340611 руб.).\", \"location_match\": \"Локация совпадает: Екатеринбург.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, Docker, MLOps, OpenCV, PyTorch\"], \"recommendation\": \"consider\"}, \"confidence\": 0.75, \"match_score\": 51}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 6 лет, навыки JavaScript, React, Redux, Vue, зарплата 92545-169285, локация Санкт-Петербург, формат Да, занятость Полная. Вакансия: Frontend Developer, опыт 3-4 лет, навыки CSS, Next.js, React, Redux, TypeScript, зарплата 170954-229650, локация Санкт-Петербург, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): React, Redux. Недостающие навыки: CSS, Next.js, TypeScript.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (92545-169285 руб.) не пересекаются с вилкой вакансии (170954-229650 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: React, Redux\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: CSS, Next.js, TypeScript\"], \"recommendation\": \"consider\"}, \"confidence\": 0.8, \"match_score\": 63}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 3 лет, навыки CSS, HTML, JavaScript, Next.js, Redux, TypeScript, Vue, зарплата 201822-291506, локация Санкт-Петербург, формат Да, занятость Полная. Вакансия: Frontend Developer, опыт 2-4 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 114526-170800, локация Санкт-Петербург, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/6): CSS, Next.js, Redux, TypeScript. Недостающие навыки: REST API, React.\", \"experience_match\": \"Опыт кандидата 3 лет находится в диапазоне вакансии (2-4 лет).\", \"salary_match\": \"Ожидания кандидата (201822-291506 руб.) не пересекаются с вилкой вакансии (114526-170800 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: CSS, Next.js, Redux, TypeScript\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: REST API, React\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 74}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 2 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 310371-352967, локация Екатеринбург, формат Гибрид, занятость Частичная. Вакансия: Data Analyst, опыт 3-5 лет, навыки A/B testing, Product metrics, Python, Tableau, зарплата 273008-358076, локация Санкт-Петербург, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: A/B testing, Product metrics, Python, Tableau.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (3-5 лет).\", \"salary_match\": \"Ожидания кандидата (310371-352967 руб.) пересекаются с вилкой вакансии (273008-358076 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, Python, Tableau\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.73, \"match_score\": 46}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 6 лет, навыки Airflow, Docker, MLflow, Python, зарплата 284193-340615, локация Москва, формат Да, занятость Частичная. Вакансия: Frontend Developer, опыт 5-6 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 249151-297209, локация Регионы, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/6): нет. Недостающие навыки: CSS, Next.js, REST API, React, Redux, TypeScript.\", \"experience_match\": \"Опыт кандидата 6 лет находится в диапазоне вакансии (5-6 лет).\", \"salary_match\": \"Ожидания кандидата (284193-340615 руб.) пересекаются с вилкой вакансии (249151-297209 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: CSS, Next.js, REST API, React, Redux\"], \"recommendation\": \"consider\"}, \"confidence\": 0.76, \"match_score\": 53}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 1 лет, навыки A/B testing, Power BI, Python, SQL, Statistics, Tableau, зарплата 203670-257523, локация Санкт-Петербург, формат Гибрид, занятость Частичная. Вакансия: Data Analyst, опыт 2-3 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 210395-270066, локация Москва, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): A/B testing, Python, SQL, Tableau. Недостающие навыки: Product metrics.\", \"experience_match\": \"Опыт кандидата 1 лет ниже диапазона вакансии (2-3 лет).\", \"salary_match\": \"Ожидания кандидата (203670-257523 руб.) пересекаются с вилкой вакансии (210395-270066 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: A/B testing, Python, SQL, Tableau\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Product metrics\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 76}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 2 лет, навыки Airflow, Docker, MLflow, NumPy, PyTorch, Python, зарплата 299297-375400, локация Москва, формат Да, занятость Частичная. Вакансия: ML Engineer, опыт 1-4 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, Python, зарплата 248889-291530, локация Санкт-Петербург, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): Docker, PyTorch, Python. Недостающие навыки: Albumentations, MLOps, OpenCV.\", \"experience_match\": \"Опыт кандидата 2 лет находится в диапазоне вакансии (1-4 лет).\", \"salary_match\": \"Ожидания кандидата (299297-375400 руб.) не пересекаются с вилкой вакансии (248889-291530 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker, PyTorch, Python\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV\"], \"recommendation\": \"consider\"}, \"confidence\": 0.82, \"match_score\": 67}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 7 лет, навыки CSS, Next.js, TypeScript, Vue, зарплата 92093-124515, локация Санкт-Петербург, формат Гибрид, занятость Полная. Вакансия: Frontend Developer, опыт 2-4 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 187685-278802, локация Санкт-Петербург, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): CSS, Next.js, TypeScript. Недостающие навыки: REST API, React, Redux.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (92093-124515 руб.) не пересекаются с вилкой вакансии (187685-278802 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: CSS, Next.js, TypeScript\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: REST API, React, Redux\"], \"recommendation\": \"consider\"}, \"confidence\": 0.82, \"match_score\": 68}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 1 лет, навыки Pandas, Power BI, Python, SQL, Statistics, зарплата 93096-182383, локация Регионы, формат Гибрид, занятость Полная. Вакансия: Data Analyst, опыт 2-5 лет, навыки A/B testing, Product metrics, SQL, Tableau, зарплата 190566-243318, локация Регионы, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/4): SQL. Недостающие навыки: A/B testing, Product metrics, Tableau.\", \"experience_match\": \"Опыт кандидата 1 лет ниже диапазона вакансии (2-5 лет).\", \"salary_match\": \"Ожидания кандидата (93096-182383 руб.) не пересекаются с вилкой вакансии (190566-243318 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: SQL\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, Tableau\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.73, \"match_score\": 45}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 3 лет, навыки Docker, MLflow, PyTorch, Python, зарплата 273376-309689, локация Регионы, формат Нет, занятость Частичная. Вакансия: ML Engineer, опыт 3-6 лет, навыки Albumentations, Docker, OpenCV, PyTorch, Python, зарплата 160058-268208, локация Санкт-Петербург, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): Docker, PyTorch, Python. Недостающие навыки: Albumentations, OpenCV.\", \"experience_match\": \"Опыт кандидата 3 лет находится в диапазоне вакансии (3-6 лет).\", \"salary_match\": \"Ожидания кандидата (273376-309689 руб.) не пересекаются с вилкой вакансии (160058-268208 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker, PyTorch, Python\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, OpenCV\"], \"recommendation\": \"consider\"}, \"confidence\": 0.84, \"match_score\": 73}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 10 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 229142-291287, локация Санкт-Петербург, формат Нет, занятость Частичная. Вакансия: DevOps Engineer, опыт 5-6 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Terraform, зарплата 153399-233770, локация Санкт-Петербург, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): AWS, Docker, GitLab CI, Terraform. Недостающие навыки: ArgoCD.\", \"experience_match\": \"Опыт кандидата 10 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (229142-291287 руб.) пересекаются с вилкой вакансии (153399-233770 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: AWS, Docker, GitLab CI, Terraform\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD\"], \"recommendation\": \"hire\"}, \"confidence\": 0.9, \"match_score\": 87}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 4 лет, навыки AsyncIO, Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 189337-221035, локация Москва, формат Гибрид, занятость Частичная. Вакансия: Python Backend Developer, опыт 4-6 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 278232-350672, локация Москва, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (5/6): Docker, FastAPI, PostgreSQL, Python, Redis. Недостающие навыки: Kafka.\", \"experience_match\": \"Опыт кандидата 4 лет находится в диапазоне вакансии (4-6 лет).\", \"salary_match\": \"Ожидания кандидата (189337-221035 руб.) не пересекаются с вилкой вакансии (278232-350672 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: Docker, FastAPI, PostgreSQL, Python, Redis\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Kafka\"], \"recommendation\": \"hire\"}, \"confidence\": 0.88, \"match_score\": 82}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 5 лет, навыки CSS, JavaScript, TypeScript, Vue, зарплата 91924-128175, локация Регионы, формат Да, занятость Частичная. Вакансия: Frontend Developer, опыт 3-4 лет, навыки Next.js, REST API, React, Redux, TypeScript, зарплата 144369-211590, локация Регионы, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/5): TypeScript. Недостающие навыки: Next.js, REST API, React, Redux.\", \"experience_match\": \"Опыт кандидата 5 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (91924-128175 руб.) не пересекаются с вилкой вакансии (144369-211590 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: TypeScript\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Next.js, REST API, React, Redux\"], \"recommendation\": \"consider\"}, \"confidence\": 0.77, \"match_score\": 54}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 6 лет, навыки Airflow, Docker, MLflow, TensorFlow, зарплата 235562-324555, локация Москва, формат Нет, занятость Полная. Вакансия: ML Engineer, опыт 1-4 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, зарплата 150506-208780, локация Москва, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/5): Docker. Недостающие навыки: Albumentations, MLOps, OpenCV, PyTorch.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (235562-324555 руб.) не пересекаются с вилкой вакансии (150506-208780 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: Docker\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV, PyTorch\"], \"recommendation\": \"consider\"}, \"confidence\": 0.76, \"match_score\": 52}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 2 лет, навыки AsyncIO, Django, Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 195298-231578, локация Санкт-Петербург, формат Да, занятость Частичная. Вакансия: Python Backend Developer, опыт 4-6 лет, навыки Docker, Kafka, PostgreSQL, Python, Redis, зарплата 281822-375706, локация Екатеринбург, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): Docker, PostgreSQL, Python, Redis. Недостающие навыки: Kafka.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (4-6 лет).\", \"salary_match\": \"Ожидания кандидата (195298-231578 руб.) не пересекаются с вилкой вакансии (281822-375706 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker, PostgreSQL, Python, Redis\"], \"weaknesses\": [\"Недостающие навыки: Kafka\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 70}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 1 лет, навыки CSS, HTML, Next.js, React, Redux, зарплата 116982-156835, локация Москва, формат Да, занятость Полная. Вакансия: Frontend Developer, опыт 3-5 лет, навыки Next.js, REST API, React, Redux, TypeScript, зарплата 223337-272125, локация Москва, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): Next.js, React, Redux. Недостающие навыки: REST API, TypeScript.\", \"experience_match\": \"Опыт кандидата 1 лет ниже диапазона вакансии (3-5 лет).\", \"salary_match\": \"Ожидания кандидата (116982-156835 руб.) не пересекаются с вилкой вакансии (223337-272125 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: Next.js, React, Redux\"], \"weaknesses\": [\"Недостающие навыки: REST API, TypeScript\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.77, \"match_score\": 55}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 3 лет, навыки MLflow, NumPy, PyTorch, Python, TensorFlow, зарплата 296242-356529, локация Казань, формат Нет, занятость Частичная. Вакансия: ML Engineer, опыт 3-5 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, Python, зарплата 294870-403016, локация Санкт-Петербург, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/6): PyTorch, Python. Недостающие навыки: Albumentations, Docker, MLOps, OpenCV.\", \"experience_match\": \"Опыт кандидата 3 лет находится в диапазоне вакансии (3-5 лет).\", \"salary_match\": \"Ожидания кандидата (296242-356529 руб.) пересекаются с вилкой вакансии (294870-403016 руб.).\", \"location_match\": \"Локация не совпадает (Казань vs Санкт-Петербург).\", \"strengths\": [\"Совпавшие навыки: PyTorch, Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, Docker, MLOps, OpenCV\", \"Несовпадение по локации/формату\"], \"recommendation\": \"consider\"}, \"confidence\": 0.8, \"match_score\": 62}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 5 лет, навыки Airflow, Docker, MLflow, Pandas, PyTorch, Python, зарплата 189392-273474, локация Москва, формат Да, занятость Полная. Вакансия: ML Engineer, опыт 2-4 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, Python, зарплата 194852-305888, локация Санкт-Петербург, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): Docker, PyTorch, Python. Недостающие навыки: Albumentations, MLOps, OpenCV.\", \"experience_match\": \"Опыт кандидата 5 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (189392-273474 руб.) пересекаются с вилкой вакансии (194852-305888 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker, PyTorch, Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 74}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 5 лет, навыки A/B testing, Power BI, SQL, Statistics, зарплата 159205-233585, локация Москва, формат Нет, занятость Частичная. Вакансия: Frontend Developer, опыт 2-4 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 246482-338537, локация Регионы, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/6): нет. Недостающие навыки: CSS, Next.js, REST API, React, Redux, TypeScript.\", \"experience_match\": \"Опыт кандидата 5 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (159205-233585 руб.) не пересекаются с вилкой вакансии (246482-338537 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: CSS, Next.js, REST API, React, Redux\"], \"recommendation\": \"reject\"}, \"confidence\": 0.73, \"match_score\": 45}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 7 лет, навыки CSS, Redux, TypeScript, Vue, зарплата 184016-264459, локация Казань, формат Нет, занятость Частичная. Вакансия: Frontend Developer, опыт 5-6 лет, навыки CSS, Next.js, Redux, TypeScript, зарплата 110335-209966, локация Казань, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): CSS, Redux, TypeScript. Недостающие навыки: Next.js.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (184016-264459 руб.) пересекаются с вилкой вакансии (110335-209966 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: CSS, Redux, TypeScript\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Next.js\"], \"recommendation\": \"hire\"}, \"confidence\": 0.9, \"match_score\": 87}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 3 лет, навыки AsyncIO, FastAPI, PostgreSQL, Redis, зарплата 121427-209772, локация Екатеринбург, формат Гибрид, занятость Частичная. Вакансия: Python Backend Developer, опыт 3-5 лет, навыки FastAPI, Kafka, Python, Redis, зарплата 269336-341607, локация Екатеринбург, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): FastAPI, Redis. Недостающие навыки: Kafka, Python.\", \"experience_match\": \"Опыт кандидата 3 лет находится в диапазоне вакансии (3-5 лет).\", \"salary_match\": \"Ожидания кандидата (121427-209772 руб.) не пересекаются с вилкой вакансии (269336-341607 руб.).\", \"location_match\": \"Локация совпадает: Екатеринбург.\", \"strengths\": [\"Совпавшие навыки: FastAPI, Redis\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Kafka, Python\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 69}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 7 лет, навыки AsyncIO, PostgreSQL, Python, Redis, зарплата 244560-322612, локация Новосибирск, формат Да, занятость Частичная. Вакансия: Python Backend Developer, опыт 5-6 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 196283-253831, локация Новосибирск, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): PostgreSQL, Python, Redis. Недостающие навыки: Docker, FastAPI, Kafka.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (244560-322612 руб.) пересекаются с вилкой вакансии (196283-253831 руб.).\", \"location_match\": \"Локация совпадает: Новосибирск.\", \"strengths\": [\"Совпавшие навыки: PostgreSQL, Python, Redis\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Docker, FastAPI, Kafka\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 74}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 6 лет, навыки CSS, Redux, TypeScript, Vue, зарплата 183371-241005, локация Казань, формат Нет, занятость Частичная. Вакансия: Frontend Developer, опыт 2-4 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 114851-168638, локация Казань, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): CSS, Redux, TypeScript. Недостающие навыки: Next.js, REST API, React.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (183371-241005 руб.) не пересекаются с вилкой вакансии (114851-168638 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: CSS, Redux, TypeScript\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Next.js, REST API, React\"], \"recommendation\": \"consider\"}, \"confidence\": 0.82, \"match_score\": 68}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 4 лет, навыки CSS, HTML, JavaScript, Next.js, Redux, TypeScript, Vue, зарплата 147409-222576, локация Регионы, формат Гибрид, занятость Полная. Вакансия: Frontend Developer, опыт 3-5 лет, навыки Next.js, REST API, React, Redux, TypeScript, зарплата 216568-299908, локация Регионы, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): Next.js, Redux, TypeScript. Недостающие навыки: REST API, React.\", \"experience_match\": \"Опыт кандидата 4 лет находится в диапазоне вакансии (3-5 лет).\", \"salary_match\": \"Ожидания кандидата (147409-222576 руб.) пересекаются с вилкой вакансии (216568-299908 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: Next.js, Redux, TypeScript\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: REST API, React\"], \"recommendation\": \"hire\"}, \"confidence\": 0.88, \"match_score\": 82}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 5 лет, навыки Airflow, Docker, Pandas, Python, зарплата 177334-215735, локация Казань, формат Нет, занятость Полная. Вакансия: ML Engineer, опыт 4-6 лет, навыки Albumentations, Docker, MLOps, OpenCV, Python, зарплата 224640-335417, локация Москва, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): Docker, Python. Недостающие навыки: Albumentations, MLOps, OpenCV.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (4-6 лет).\", \"salary_match\": \"Ожидания кандидата (177334-215735 руб.) не пересекаются с вилкой вакансии (224640-335417 руб.).\", \"location_match\": \"Локация не совпадает (Казань vs Москва).\", \"strengths\": [\"Совпавшие навыки: Docker, Python\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV\", \"Несовпадение по локации/формату\"], \"recommendation\": \"consider\"}, \"confidence\": 0.78, \"match_score\": 58}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 6 лет, навыки AWS, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 217491-266374, локация Новосибирск, формат Да, занятость Полная. Вакансия: DevOps Engineer, опыт 3-5 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Terraform, зарплата 162404-221847, локация Новосибирск, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): AWS, GitLab CI, Kubernetes, Terraform. Недостающие навыки: Docker.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (217491-266374 руб.) пересекаются с вилкой вакансии (162404-221847 руб.).\", \"location_match\": \"Локация совпадает: Новосибирск.\", \"strengths\": [\"Совпавшие навыки: AWS, GitLab CI, Kubernetes, Terraform\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Docker\"], \"recommendation\": \"hire\"}, \"confidence\": 0.9, \"match_score\": 87}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 3 лет, навыки CSS, HTML, JavaScript, Next.js, React, TypeScript, Vue, зарплата 166932-249874, локация Санкт-Петербург, формат Гибрид, занятость Частичная. Вакансия: Frontend Developer, опыт 6-8 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 145979-220221, локация Санкт-Петербург, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/6): CSS, Next.js, React, TypeScript. Недостающие навыки: REST API, Redux.\", \"experience_match\": \"Опыт кандидата 3 лет ниже диапазона вакансии (6-8 лет).\", \"salary_match\": \"Ожидания кандидата (166932-249874 руб.) пересекаются с вилкой вакансии (145979-220221 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: CSS, Next.js, React, TypeScript\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: REST API, Redux\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 70}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 4 лет, навыки Airflow, Docker, Pandas, PyTorch, зарплата 246778-308839, локация Регионы, формат Да, занятость Частичная. Вакансия: ML Engineer, опыт 1-3 лет, навыки Albumentations, Docker, MLOps, PyTorch, Python, зарплата 196739-314278, локация Казань, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): Docker, PyTorch. Недостающие навыки: Albumentations, MLOps, Python.\", \"experience_match\": \"Опыт кандидата 4 лет выше верхней границы (3 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (246778-308839 руб.) пересекаются с вилкой вакансии (196739-314278 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker, PyTorch\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, Python\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 69}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 1 лет, навыки A/B testing, Pandas, Power BI, SQL, Statistics, Tableau, зарплата 234705-281602, локация Регионы, формат Нет, занятость Полная. Вакансия: Data Analyst, опыт 3-6 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 161803-269291, локация Регионы, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): A/B testing, SQL, Tableau. Недостающие навыки: Product metrics, Python.\", \"experience_match\": \"Опыт кандидата 1 лет ниже диапазона вакансии (3-6 лет).\", \"salary_match\": \"Ожидания кандидата (234705-281602 руб.) пересекаются с вилкой вакансии (161803-269291 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: A/B testing, SQL, Tableau\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Product metrics, Python\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.81, \"match_score\": 65}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 8 лет, навыки AsyncIO, Docker, FastAPI, PostgreSQL, Redis, зарплата 225679-285267, локация Санкт-Петербург, формат Да, занятость Полная. Вакансия: Python Backend Developer, опыт 4-6 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 151720-224213, локация Санкт-Петербург, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/6): Docker, FastAPI, PostgreSQL, Redis. Недостающие навыки: Kafka, Python.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (225679-285267 руб.) не пересекаются с вилкой вакансии (151720-224213 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: Docker, FastAPI, PostgreSQL, Redis\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Kafka, Python\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 75}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 8 лет, навыки AsyncIO, Django, Python, Redis, зарплата 272421-327986, локация Новосибирск, формат Гибрид, занятость Полная. Вакансия: Python Backend Developer, опыт 3-5 лет, навыки FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 147988-235077, локация Новосибирск, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): Python, Redis. Недостающие навыки: FastAPI, Kafka, PostgreSQL.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (272421-327986 руб.) не пересекаются с вилкой вакансии (147988-235077 руб.).\", \"location_match\": \"Локация совпадает: Новосибирск.\", \"strengths\": [\"Совпавшие навыки: Python, Redis\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: FastAPI, Kafka, PostgreSQL\"], \"recommendation\": \"consider\"}, \"confidence\": 0.8, \"match_score\": 63}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 6 лет, навыки AsyncIO, Docker, FastAPI, PostgreSQL, Python, зарплата 254931-332656, локация Екатеринбург, формат Нет, занятость Частичная. Вакансия: Python Backend Developer, опыт 4-5 лет, навыки Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 131058-243092, локация Екатеринбург, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): Docker, FastAPI, PostgreSQL, Python. Недостающие навыки: Redis.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (254931-332656 руб.) не пересекаются с вилкой вакансии (131058-243092 руб.).\", \"location_match\": \"Локация совпадает: Екатеринбург.\", \"strengths\": [\"Совпавшие навыки: Docker, FastAPI, PostgreSQL, Python\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Redis\"], \"recommendation\": \"consider\"}, \"confidence\": 0.87, \"match_score\": 79}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 10 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 169334-238497, локация Новосибирск, формат Гибрид, занятость Частичная. Вакансия: Data Analyst, опыт 3-6 лет, навыки A/B testing, Product metrics, Python, Tableau, зарплата 308618-364394, локация Санкт-Петербург, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: A/B testing, Product metrics, Python, Tableau.\", \"experience_match\": \"Опыт кандидата 10 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (169334-238497 руб.) не пересекаются с вилкой вакансии (308618-364394 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, Python, Tableau\"], \"recommendation\": \"reject\"}, \"confidence\": 0.72, \"match_score\": 43}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 2 лет, навыки Docker, MLflow, NumPy, Python, TensorFlow, зарплата 276080-353620, локация Казань, формат Гибрид, занятость Частичная. Вакансия: DevOps Engineer, опыт 2-5 лет, навыки AWS, ArgoCD, GitLab CI, Terraform, зарплата 268964-320818, локация Москва, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: AWS, ArgoCD, GitLab CI, Terraform.\", \"experience_match\": \"Опыт кандидата 2 лет находится в диапазоне вакансии (2-5 лет).\", \"salary_match\": \"Ожидания кандидата (276080-353620 руб.) пересекаются с вилкой вакансии (268964-320818 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: AWS, ArgoCD, GitLab CI, Terraform\"], \"recommendation\": \"consider\"}, \"confidence\": 0.77, \"match_score\": 55}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 4 лет, навыки CSS, HTML, JavaScript, Next.js, TypeScript, Vue, зарплата 120782-158166, локация Регионы, формат Гибрид, занятость Полная. Вакансия: Frontend Developer, опыт 3-4 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 139116-241298, локация Регионы, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): CSS, Next.js, TypeScript. Недостающие навыки: REST API, React, Redux.\", \"experience_match\": \"Опыт кандидата 4 лет находится в диапазоне вакансии (3-4 лет).\", \"salary_match\": \"Ожидания кандидата (120782-158166 руб.) пересекаются с вилкой вакансии (139116-241298 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: CSS, Next.js, TypeScript\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: REST API, React, Redux\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 75}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 3 лет, навыки AsyncIO, Django, FastAPI, Python, зарплата 148077-218388, локация Казань, формат Да, занятость Частичная. Вакансия: Data Analyst, опыт 5-8 лет, навыки A/B testing, Product metrics, Python, SQL, зарплата 131367-239714, локация Москва, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/4): Python. Недостающие навыки: A/B testing, Product metrics, SQL.\", \"experience_match\": \"Опыт кандидата 3 лет ниже диапазона вакансии (5-8 лет).\", \"salary_match\": \"Ожидания кандидата (148077-218388 руб.) пересекаются с вилкой вакансии (131367-239714 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Python\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, SQL\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.77, \"match_score\": 56}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 4 лет, навыки CSS, JavaScript, Next.js, React, TypeScript, Vue, зарплата 196121-268629, локация Регионы, формат Да, занятость Частичная. Вакансия: Frontend Developer, опыт 1-2 лет, навыки CSS, React, Redux, TypeScript, зарплата 138051-209741, локация Санкт-Петербург, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): CSS, React, TypeScript. Недостающие навыки: Redux.\", \"experience_match\": \"Опыт кандидата 4 лет выше верхней границы (2 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (196121-268629 руб.) пересекаются с вилкой вакансии (138051-209741 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: CSS, React, TypeScript\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Redux\"], \"recommendation\": \"hire\"}, \"confidence\": 0.89, \"match_score\": 85}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 3 лет, навыки Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 202544-250368, локация Казань, формат Гибрид, занятость Частичная. Вакансия: Python Backend Developer, опыт 2-4 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 257325-305696, локация Екатеринбург, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (5/6): Docker, FastAPI, PostgreSQL, Python, Redis. Недостающие навыки: Kafka.\", \"experience_match\": \"Опыт кандидата 3 лет находится в диапазоне вакансии (2-4 лет).\", \"salary_match\": \"Ожидания кандидата (202544-250368 руб.) не пересекаются с вилкой вакансии (257325-305696 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker, FastAPI, PostgreSQL, Python, Redis\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Kafka\"], \"recommendation\": \"hire\"}, \"confidence\": 0.89, \"match_score\": 84}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 8 лет, навыки A/B testing, Pandas, Power BI, Statistics, Tableau, зарплата 144516-188242, локация Казань, формат Нет, занятость Частичная. Вакансия: Data Analyst, опыт 3-5 лет, навыки A/B testing, Product metrics, SQL, Tableau, зарплата 195511-266108, локация Казань, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): A/B testing, Tableau. Недостающие навыки: Product metrics, SQL.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (144516-188242 руб.) не пересекаются с вилкой вакансии (195511-266108 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: A/B testing, Tableau\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Product metrics, SQL\"], \"recommendation\": \"consider\"}, \"confidence\": 0.82, \"match_score\": 68}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 3 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 165564-231739, локация Москва, формат Гибрид, занятость Частичная. Вакансия: DevOps Engineer, опыт 3-4 лет, навыки AWS, ArgoCD, Kubernetes, Terraform, зарплата 169677-230194, локация Москва, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): AWS, Kubernetes, Terraform. Недостающие навыки: ArgoCD.\", \"experience_match\": \"Опыт кандидата 3 лет находится в диапазоне вакансии (3-4 лет).\", \"salary_match\": \"Ожидания кандидата (165564-231739 руб.) пересекаются с вилкой вакансии (169677-230194 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: AWS, Kubernetes, Terraform\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD\"], \"recommendation\": \"hire\"}, \"confidence\": 0.9, \"match_score\": 88}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 8 лет, навыки A/B testing, Pandas, Power BI, Python, SQL, Statistics, зарплата 176401-255887, локация Казань, формат Гибрид, занятость Полная. Вакансия: Data Analyst, опыт 2-5 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 161503-226865, локация Казань, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): A/B testing, Python, SQL. Недостающие навыки: Product metrics, Tableau.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (176401-255887 руб.) пересекаются с вилкой вакансии (161503-226865 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: A/B testing, Python, SQL\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Product metrics, Tableau\"], \"recommendation\": \"hire\"}, \"confidence\": 0.87, \"match_score\": 80}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 7 лет, навыки AsyncIO, Django, Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 224102-295341, локация Москва, формат Нет, занятость Полная. Вакансия: Python Backend Developer, опыт 5-8 лет, навыки FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 235355-331530, локация Москва, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): FastAPI, PostgreSQL, Python, Redis. Недостающие навыки: Kafka.\", \"experience_match\": \"Опыт кандидата 7 лет находится в диапазоне вакансии (5-8 лет).\", \"salary_match\": \"Ожидания кандидата (224102-295341 руб.) пересекаются с вилкой вакансии (235355-331530 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: FastAPI, PostgreSQL, Python, Redis\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Kafka\"], \"recommendation\": \"hire\"}, \"confidence\": 0.91, \"match_score\": 91}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 2 лет, навыки HTML, JavaScript, React, TypeScript, зарплата 217754-283509, локация Регионы, формат Нет, занятость Частичная. Вакансия: Frontend Developer, опыт 2-3 лет, навыки CSS, Next.js, React, TypeScript, зарплата 119378-161972, локация Регионы, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): React, TypeScript. Недостающие навыки: CSS, Next.js.\", \"experience_match\": \"Опыт кандидата 2 лет находится в диапазоне вакансии (2-3 лет).\", \"salary_match\": \"Ожидания кандидата (217754-283509 руб.) не пересекаются с вилкой вакансии (119378-161972 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: React, TypeScript\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: CSS, Next.js\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 69}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 9 лет, навыки AsyncIO, Django, Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 235836-301961, локация Новосибирск, формат Гибрид, занятость Частичная. Вакансия: Python Backend Developer, опыт 5-7 лет, навыки FastAPI, PostgreSQL, Python, Redis, зарплата 268920-334117, локация Новосибирск, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/4): FastAPI, PostgreSQL, Python, Redis. Недостающие навыки: нет.\", \"experience_match\": \"Опыт кандидата 9 лет выше верхней границы (7 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (235836-301961 руб.) пересекаются с вилкой вакансии (268920-334117 руб.).\", \"location_match\": \"Локация совпадает: Новосибирск.\", \"strengths\": [\"Совпавшие навыки: FastAPI, PostgreSQL, Python, Redis\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [], \"recommendation\": \"hire\"}, \"confidence\": 0.93, \"match_score\": 96}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 6 лет, навыки CSS, HTML, React, TypeScript, зарплата 179914-247044, локация Москва, формат Гибрид, занятость Полная. Вакансия: Frontend Developer, опыт 4-6 лет, навыки Next.js, React, Redux, TypeScript, зарплата 186825-266026, локация Москва, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): React, TypeScript. Недостающие навыки: Next.js, Redux.\", \"experience_match\": \"Опыт кандидата 6 лет находится в диапазоне вакансии (4-6 лет).\", \"salary_match\": \"Ожидания кандидата (179914-247044 руб.) пересекаются с вилкой вакансии (186825-266026 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: React, TypeScript\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Next.js, Redux\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 75}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 6 лет, навыки Pandas, Power BI, SQL, Statistics, зарплата 120581-153198, локация Москва, формат Нет, занятость Частичная. Вакансия: Data Analyst, опыт 4-7 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 150292-232579, локация Казань, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/5): SQL. Недостающие навыки: A/B testing, Product metrics, Python, Tableau.\", \"experience_match\": \"Опыт кандидата 6 лет находится в диапазоне вакансии (4-7 лет).\", \"salary_match\": \"Ожидания кандидата (120581-153198 руб.) пересекаются с вилкой вакансии (150292-232579 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: SQL\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, Python, Tableau\"], \"recommendation\": \"consider\"}, \"confidence\": 0.8, \"match_score\": 62}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 6 лет, навыки AsyncIO, Django, Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 213560-283739, локация Санкт-Петербург, формат Да, занятость Частичная. Вакансия: Python Backend Developer, опыт 3-5 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 209940-297902, локация Санкт-Петербург, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (5/6): Docker, FastAPI, PostgreSQL, Python, Redis. Недостающие навыки: Kafka.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (213560-283739 руб.) пересекаются с вилкой вакансии (209940-297902 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: Docker, FastAPI, PostgreSQL, Python, Redis\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Kafka\"], \"recommendation\": \"hire\"}, \"confidence\": 0.91, \"match_score\": 91}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 4 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Prometheus, зарплата 309928-373777, локация Санкт-Петербург, формат Нет, занятость Частичная. Вакансия: DevOps Engineer, опыт 6-9 лет, навыки AWS, Docker, GitLab CI, Kubernetes, зарплата 181903-243252, локация Санкт-Петербург, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/4): AWS, Docker, GitLab CI, Kubernetes. Недостающие навыки: нет.\", \"experience_match\": \"Опыт кандидата 4 лет ниже диапазона вакансии (6-9 лет).\", \"salary_match\": \"Ожидания кандидата (309928-373777 руб.) не пересекаются с вилкой вакансии (181903-243252 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: AWS, Docker, GitLab CI, Kubernetes\"], \"weaknesses\": [\"Недостаточный опыт\"], \"recommendation\": \"hire\"}, \"confidence\": 0.88, \"match_score\": 83}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 5 лет, навыки Airflow, Docker, NumPy, Pandas, PyTorch, зарплата 115559-199621, локация Казань, формат Нет, занятость Полная. Вакансия: ML Engineer, опыт 5-6 лет, навыки Albumentations, Docker, OpenCV, PyTorch, Python, зарплата 151672-205148, локация Казань, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): Docker, PyTorch. Недостающие навыки: Albumentations, OpenCV, Python.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (5-6 лет).\", \"salary_match\": \"Ожидания кандидата (115559-199621 руб.) пересекаются с вилкой вакансии (151672-205148 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: Docker, PyTorch\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, OpenCV, Python\"], \"recommendation\": \"consider\"}, \"confidence\": 0.84, \"match_score\": 73}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 8 лет, навыки CSS, HTML, JavaScript, Next.js, React, Redux, зарплата 101664-159099, локация Регионы, формат Да, занятость Полная. Вакансия: Frontend Developer, опыт 1-2 лет, навыки CSS, Next.js, React, TypeScript, зарплата 169042-281731, локация Санкт-Петербург, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): CSS, Next.js, React. Недостающие навыки: TypeScript.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (2 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (101664-159099 руб.) не пересекаются с вилкой вакансии (169042-281731 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: CSS, Next.js, React\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: TypeScript\"], \"recommendation\": \"consider\"}, \"confidence\": 0.87, \"match_score\": 79}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 4 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Prometheus, зарплата 149223-191661, локация Регионы, формат Нет, занятость Полная. Вакансия: DevOps Engineer, опыт 4-5 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 302527-383228, локация Регионы, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/6): AWS, Docker, GitLab CI, Kubernetes. Недостающие навыки: ArgoCD, Terraform.\", \"experience_match\": \"Опыт кандидата 4 лет находится в диапазоне вакансии (4-5 лет).\", \"salary_match\": \"Ожидания кандидата (149223-191661 руб.) не пересекаются с вилкой вакансии (302527-383228 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: AWS, Docker, GitLab CI, Kubernetes\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, Terraform\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 74}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 6 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Prometheus, Terraform, зарплата 298252-358122, локация Новосибирск, формат Гибрид, занятость Полная. Вакансия: DevOps Engineer, опыт 3-5 лет, навыки AWS, Docker, GitLab CI, Terraform, зарплата 188646-307481, локация Новосибирск, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/4): AWS, Docker, GitLab CI, Terraform. Недостающие навыки: нет.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (298252-358122 руб.) пересекаются с вилкой вакансии (188646-307481 руб.).\", \"location_match\": \"Локация совпадает: Новосибирск.\", \"strengths\": [\"Совпавшие навыки: AWS, Docker, GitLab CI, Terraform\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [], \"recommendation\": \"hire\"}, \"confidence\": 0.93, \"match_score\": 96}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 9 лет, навыки A/B testing, SQL, Statistics, Tableau, зарплата 155996-241813, локация Санкт-Петербург, формат Да, занятость Частичная. Вакансия: Data Analyst, опыт 5-6 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 220268-288038, локация Регионы, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): A/B testing, SQL, Tableau. Недостающие навыки: Product metrics, Python.\", \"experience_match\": \"Опыт кандидата 9 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (155996-241813 руб.) пересекаются с вилкой вакансии (220268-288038 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: A/B testing, SQL, Tableau\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Product metrics, Python\"], \"recommendation\": \"consider\"}, \"confidence\": 0.86, \"match_score\": 78}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 7 лет, навыки Docker, GitLab CI, Linux, Prometheus, Terraform, зарплата 233206-275378, локация Екатеринбург, формат Да, занятость Полная. Вакансия: Data Analyst, опыт 6-8 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 328177-444968, локация Новосибирск, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/5): нет. Недостающие навыки: A/B testing, Product metrics, Python, SQL, Tableau.\", \"experience_match\": \"Опыт кандидата 7 лет находится в диапазоне вакансии (6-8 лет).\", \"salary_match\": \"Ожидания кандидата (233206-275378 руб.) не пересекаются с вилкой вакансии (328177-444968 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, Python, SQL, Tableau\"], \"recommendation\": \"reject\"}, \"confidence\": 0.73, \"match_score\": 46}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 8 лет, навыки A/B testing, Pandas, Power BI, SQL, Statistics, Tableau, зарплата 188901-248837, локация Регионы, формат Гибрид, занятость Частичная. Вакансия: Python Backend Developer, опыт 2-5 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 125135-238377, локация Санкт-Петербург, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/6): нет. Недостающие навыки: Docker, FastAPI, Kafka, PostgreSQL, Python, Redis.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (188901-248837 руб.) пересекаются с вилкой вакансии (125135-238377 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Docker, FastAPI, Kafka, PostgreSQL, Python\"], \"recommendation\": \"consider\"}, \"confidence\": 0.76, \"match_score\": 53}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 4 лет, навыки HTML, Next.js, React, TypeScript, зарплата 102375-143891, локация Регионы, формат Нет, занятость Частичная. Вакансия: Frontend Developer, опыт 4-5 лет, навыки CSS, React, Redux, TypeScript, зарплата 238012-318192, локация Регионы, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): React, TypeScript. Недостающие навыки: CSS, Redux.\", \"experience_match\": \"Опыт кандидата 4 лет находится в диапазоне вакансии (4-5 лет).\", \"salary_match\": \"Ожидания кандидата (102375-143891 руб.) не пересекаются с вилкой вакансии (238012-318192 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: React, TypeScript\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: CSS, Redux\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 69}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 2 лет, навыки AsyncIO, Django, Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 136081-168462, локация Санкт-Петербург, формат Да, занятость Частичная. Вакансия: DevOps Engineer, опыт 3-5 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Terraform, зарплата 224069-328999, локация Казань, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/5): Docker. Недостающие навыки: AWS, ArgoCD, GitLab CI, Terraform.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (3-5 лет).\", \"salary_match\": \"Ожидания кандидата (136081-168462 руб.) не пересекаются с вилкой вакансии (224069-328999 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker\"], \"weaknesses\": [\"Недостающие навыки: AWS, ArgoCD, GitLab CI, Terraform\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.73, \"match_score\": 45}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 8 лет, навыки Kubernetes, Linux, Prometheus, Terraform, зарплата 156499-237934, локация Москва, формат Гибрид, занятость Полная. Вакансия: Data Analyst, опыт 3-6 лет, навыки A/B testing, Product metrics, Python, Tableau, зарплата 204164-271288, локация Екатеринбург, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: A/B testing, Product metrics, Python, Tableau.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (156499-237934 руб.) пересекаются с вилкой вакансии (204164-271288 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, Python, Tableau\"], \"recommendation\": \"consider\"}, \"confidence\": 0.75, \"match_score\": 51}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 7 лет, навыки CSS, JavaScript, Next.js, React, Redux, TypeScript, Vue, зарплата 110406-185279, локация Регионы, формат Да, занятость Частичная. Вакансия: Frontend Developer, опыт 1-2 лет, навыки CSS, REST API, React, Redux, TypeScript, зарплата 198109-238757, локация Регионы, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): CSS, React, Redux, TypeScript. Недостающие навыки: REST API.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (2 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (110406-185279 руб.) не пересекаются с вилкой вакансии (198109-238757 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: CSS, React, Redux, TypeScript\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: REST API\"], \"recommendation\": \"consider\"}, \"confidence\": 0.87, \"match_score\": 79}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 6 лет, навыки Airflow, MLflow, NumPy, Pandas, PyTorch, Python, TensorFlow, зарплата 229944-267412, локация Казань, формат Нет, занятость Полная. Вакансия: Frontend Developer, опыт 3-4 лет, навыки CSS, Next.js, REST API, React, Redux, зарплата 208774-312931, локация Регионы, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/5): нет. Недостающие навыки: CSS, Next.js, REST API, React, Redux.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (229944-267412 руб.) пересекаются с вилкой вакансии (208774-312931 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: CSS, Next.js, REST API, React, Redux\"], \"recommendation\": \"consider\"}, \"confidence\": 0.75, \"match_score\": 51}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 5 лет, навыки CSS, JavaScript, Next.js, React, TypeScript, зарплата 217150-301131, локация Казань, формат Гибрид, занятость Полная. Вакансия: ML Engineer, опыт 2-4 лет, навыки Albumentations, Docker, MLOps, PyTorch, зарплата 102307-157343, локация Регионы, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: Albumentations, Docker, MLOps, PyTorch.\", \"experience_match\": \"Опыт кандидата 5 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (217150-301131 руб.) не пересекаются с вилкой вакансии (102307-157343 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, Docker, MLOps, PyTorch\"], \"recommendation\": \"reject\"}, \"confidence\": 0.73, \"match_score\": 45}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 5 лет, навыки AWS, GitLab CI, Kubernetes, Linux, Terraform, зарплата 129045-212334, локация Санкт-Петербург, формат Нет, занятость Полная. Вакансия: DevOps Engineer, опыт 3-5 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 244742-288445, локация Санкт-Петербург, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/6): AWS, GitLab CI, Kubernetes, Terraform. Недостающие навыки: ArgoCD, Docker.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (3-5 лет).\", \"salary_match\": \"Ожидания кандидата (129045-212334 руб.) не пересекаются с вилкой вакансии (244742-288445 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: AWS, GitLab CI, Kubernetes, Terraform\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, Docker\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 74}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 5 лет, навыки CSS, HTML, Next.js, React, TypeScript, Vue, зарплата 106770-144581, локация Регионы, формат Гибрид, занятость Частичная. Вакансия: Frontend Developer, опыт 3-4 лет, навыки CSS, Next.js, React, Redux, зарплата 207683-303694, локация Москва, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): CSS, Next.js, React. Недостающие навыки: Redux.\", \"experience_match\": \"Опыт кандидата 5 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (106770-144581 руб.) не пересекаются с вилкой вакансии (207683-303694 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: CSS, Next.js, React\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Redux\"], \"recommendation\": \"consider\"}, \"confidence\": 0.86, \"match_score\": 77}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 7 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 241920-331517, локация Москва, формат Нет, занятость Частичная. Вакансия: DevOps Engineer, опыт 2-4 лет, навыки ArgoCD, Docker, GitLab CI, Kubernetes, зарплата 251333-348137, локация Москва, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): Docker, GitLab CI, Kubernetes. Недостающие навыки: ArgoCD.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (241920-331517 руб.) пересекаются с вилкой вакансии (251333-348137 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: Docker, GitLab CI, Kubernetes\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD\"], \"recommendation\": \"hire\"}, \"confidence\": 0.89, \"match_score\": 85}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 1 лет, навыки HTML, JavaScript, Next.js, Redux, зарплата 114428-188600, локация Санкт-Петербург, формат Да, занятость Частичная. Вакансия: Data Analyst, опыт 5-6 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 109453-196604, локация Москва, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/5): нет. Недостающие навыки: A/B testing, Product metrics, Python, SQL, Tableau.\", \"experience_match\": \"Опыт кандидата 1 лет ниже диапазона вакансии (5-6 лет).\", \"salary_match\": \"Ожидания кандидата (114428-188600 руб.) пересекаются с вилкой вакансии (109453-196604 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, Python, SQL, Tableau\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.69, \"match_score\": 35}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 9 лет, навыки A/B testing, Pandas, Power BI, Python, SQL, Statistics, Tableau, зарплата 239926-275722, локация Москва, формат Гибрид, занятость Полная. Вакансия: Python Backend Developer, опыт 1-4 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 174285-276210, локация Регионы, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/6): Python. Недостающие навыки: Docker, FastAPI, Kafka, PostgreSQL, Redis.\", \"experience_match\": \"Опыт кандидата 9 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (239926-275722 руб.) пересекаются с вилкой вакансии (174285-276210 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Docker, FastAPI, Kafka, PostgreSQL, Redis\"], \"recommendation\": \"consider\"}, \"confidence\": 0.79, \"match_score\": 59}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 3 лет, навыки CSS, HTML, Next.js, React, Vue, зарплата 167591-234538, локация Москва, формат Гибрид, занятость Полная. Вакансия: Data Analyst, опыт 4-6 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 199066-316187, локация Санкт-Петербург, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/5): нет. Недостающие навыки: A/B testing, Product metrics, Python, SQL, Tableau.\", \"experience_match\": \"Опыт кандидата 3 лет ниже диапазона вакансии (4-6 лет).\", \"salary_match\": \"Ожидания кандидата (167591-234538 руб.) пересекаются с вилкой вакансии (199066-316187 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, Python, SQL, Tableau\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.74, \"match_score\": 48}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 1 лет, навыки A/B testing, Pandas, Python, SQL, Statistics, зарплата 222723-280518, локация Санкт-Петербург, формат Нет, занятость Частичная. Вакансия: Data Analyst, опыт 3-5 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 249443-339155, локация Санкт-Петербург, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): A/B testing, Python, SQL. Недостающие навыки: Product metrics, Tableau.\", \"experience_match\": \"Опыт кандидата 1 лет ниже диапазона вакансии (3-5 лет).\", \"salary_match\": \"Ожидания кандидата (222723-280518 руб.) пересекаются с вилкой вакансии (249443-339155 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: A/B testing, Python, SQL\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Product metrics, Tableau\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.8, \"match_score\": 63}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 1 лет, навыки Airflow, Docker, MLflow, Pandas, PyTorch, Python, TensorFlow, зарплата 222094-257836, локация Регионы, формат Да, занятость Частичная. Вакансия: Frontend Developer, опыт 3-4 лет, навыки CSS, Next.js, REST API, React, зарплата 217214-335885, локация Москва, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: CSS, Next.js, REST API, React.\", \"experience_match\": \"Опыт кандидата 1 лет ниже диапазона вакансии (3-4 лет).\", \"salary_match\": \"Ожидания кандидата (222094-257836 руб.) пересекаются с вилкой вакансии (217214-335885 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: CSS, Next.js, REST API, React\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.69, \"match_score\": 36}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 2 лет, навыки Airflow, Docker, MLflow, NumPy, Pandas, Python, TensorFlow, зарплата 168665-243821, локация Регионы, формат Гибрид, занятость Частичная. Вакансия: ML Engineer, опыт 4-7 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, Python, зарплата 214369-258567, локация Санкт-Петербург, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/6): Docker, Python. Недостающие навыки: Albumentations, MLOps, OpenCV, PyTorch.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (4-7 лет).\", \"salary_match\": \"Ожидания кандидата (168665-243821 руб.) пересекаются с вилкой вакансии (214369-258567 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker, Python\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV, PyTorch\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.78, \"match_score\": 57}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 2 лет, навыки Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 104694-156826, локация Москва, формат Гибрид, занятость Частичная. Вакансия: Python Backend Developer, опыт 2-4 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, зарплата 131223-224373, локация Москва, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): Docker, FastAPI, PostgreSQL, Python. Недостающие навыки: Kafka.\", \"experience_match\": \"Опыт кандидата 2 лет находится в диапазоне вакансии (2-4 лет).\", \"salary_match\": \"Ожидания кандидата (104694-156826 руб.) пересекаются с вилкой вакансии (131223-224373 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: Docker, FastAPI, PostgreSQL, Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Kafka\"], \"recommendation\": \"hire\"}, \"confidence\": 0.91, \"match_score\": 89}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 2 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Prometheus, зарплата 151263-189395, локация Регионы, формат Да, занятость Полная. Вакансия: Frontend Developer, опыт 4-6 лет, навыки Next.js, REST API, React, TypeScript, зарплата 268694-331125, локация Екатеринбург, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: Next.js, REST API, React, TypeScript.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (4-6 лет).\", \"salary_match\": \"Ожидания кандидата (151263-189395 руб.) не пересекаются с вилкой вакансии (268694-331125 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [], \"weaknesses\": [\"Недостающие навыки: Next.js, REST API, React, TypeScript\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.68, \"match_score\": 32}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 3 лет, навыки Airflow, Docker, MLflow, NumPy, Pandas, PyTorch, зарплата 294740-343632, локация Москва, формат Да, занятость Частичная. Вакансия: ML Engineer, опыт 4-6 лет, навыки Albumentations, MLOps, OpenCV, Python, зарплата 286048-331370, локация Москва, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: Albumentations, MLOps, OpenCV, Python.\", \"experience_match\": \"Опыт кандидата 3 лет ниже диапазона вакансии (4-6 лет).\", \"salary_match\": \"Ожидания кандидата (294740-343632 руб.) пересекаются с вилкой вакансии (286048-331370 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV, Python\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.74, \"match_score\": 48}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 6 лет, навыки AWS, Linux, Prometheus, Terraform, зарплата 225952-312634, локация Регионы, формат Да, занятость Частичная. Вакансия: Data Analyst, опыт 4-5 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 284898-325048, локация Москва, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/5): нет. Недостающие навыки: A/B testing, Product metrics, Python, SQL, Tableau.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (225952-312634 руб.) пересекаются с вилкой вакансии (284898-325048 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, Python, SQL, Tableau\"], \"recommendation\": \"consider\"}, \"confidence\": 0.76, \"match_score\": 53}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 6 лет, навыки Docker, GitLab CI, Kubernetes, Linux, Prometheus, зарплата 137566-173269, локация Екатеринбург, формат Нет, занятость Частичная. Вакансия: DevOps Engineer, опыт 4-7 лет, навыки AWS, ArgoCD, Docker, Kubernetes, Terraform, зарплата 251737-364660, локация Екатеринбург, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): Docker, Kubernetes. Недостающие навыки: AWS, ArgoCD, Terraform.\", \"experience_match\": \"Опыт кандидата 6 лет находится в диапазоне вакансии (4-7 лет).\", \"salary_match\": \"Ожидания кандидата (137566-173269 руб.) не пересекаются с вилкой вакансии (251737-364660 руб.).\", \"location_match\": \"Локация совпадает: Екатеринбург.\", \"strengths\": [\"Совпавшие навыки: Docker, Kubernetes\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: AWS, ArgoCD, Terraform\"], \"recommendation\": \"consider\"}, \"confidence\": 0.8, \"match_score\": 62}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 4 лет, навыки CSS, JavaScript, Next.js, React, Vue, зарплата 202489-264505, локация Москва, формат Да, занятость Частичная. Вакансия: Python Backend Developer, опыт 6-8 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 165829-212487, локация Казань, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/6): нет. Недостающие навыки: Docker, FastAPI, Kafka, PostgreSQL, Python, Redis.\", \"experience_match\": \"Опыт кандидата 4 лет ниже диапазона вакансии (6-8 лет).\", \"salary_match\": \"Ожидания кандидата (202489-264505 руб.) пересекаются с вилкой вакансии (165829-212487 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Docker, FastAPI, Kafka, PostgreSQL, Python\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.73, \"match_score\": 44}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 5 лет, навыки Airflow, Docker, MLflow, NumPy, PyTorch, Python, TensorFlow, зарплата 260722-344567, локация Регионы, формат Гибрид, занятость Частичная. Вакансия: ML Engineer, опыт 5-6 лет, навыки Docker, MLOps, OpenCV, Python, зарплата 199675-273576, локация Регионы, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): Docker, Python. Недостающие навыки: MLOps, OpenCV.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (5-6 лет).\", \"salary_match\": \"Ожидания кандидата (260722-344567 руб.) пересекаются с вилкой вакансии (199675-273576 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: Docker, Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: MLOps, OpenCV\"], \"recommendation\": \"consider\"}, \"confidence\": 0.86, \"match_score\": 77}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 2 лет, навыки AsyncIO, Docker, FastAPI, Python, Redis, зарплата 247756-291217, локация Санкт-Петербург, формат Да, занятость Полная. Вакансия: Python Backend Developer, опыт 2-3 лет, навыки FastAPI, Kafka, PostgreSQL, Python, зарплата 274036-340253, локация Санкт-Петербург, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): FastAPI, Python. Недостающие навыки: Kafka, PostgreSQL.\", \"experience_match\": \"Опыт кандидата 2 лет находится в диапазоне вакансии (2-3 лет).\", \"salary_match\": \"Ожидания кандидата (247756-291217 руб.) пересекаются с вилкой вакансии (274036-340253 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: FastAPI, Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Kafka, PostgreSQL\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 75}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 8 лет, навыки Airflow, NumPy, Pandas, TensorFlow, зарплата 111492-166874, локация Москва, формат Да, занятость Частичная. Вакансия: ML Engineer, опыт 4-5 лет, навыки Docker, OpenCV, PyTorch, Python, зарплата 186302-275679, локация Москва, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: Docker, OpenCV, PyTorch, Python.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (111492-166874 руб.) не пересекаются с вилкой вакансии (186302-275679 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Docker, OpenCV, PyTorch, Python\"], \"recommendation\": \"reject\"}, \"confidence\": 0.72, \"match_score\": 43}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 5 лет, навыки Pandas, Python, SQL, Statistics, зарплата 88147-144649, локация Санкт-Петербург, формат Да, занятость Полная. Вакансия: Data Analyst, опыт 3-4 лет, навыки A/B testing, Product metrics, SQL, Tableau, зарплата 228541-341410, локация Санкт-Петербург, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/4): SQL. Недостающие навыки: A/B testing, Product metrics, Tableau.\", \"experience_match\": \"Опыт кандидата 5 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (88147-144649 руб.) не пересекаются с вилкой вакансии (228541-341410 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: SQL\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, Tableau\"], \"recommendation\": \"consider\"}, \"confidence\": 0.77, \"match_score\": 56}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 7 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 128501-202213, локация Новосибирск, формат Да, занятость Полная. Вакансия: DevOps Engineer, опыт 2-3 лет, навыки ArgoCD, Docker, Kubernetes, Terraform, зарплата 301543-420616, локация Новосибирск, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): Docker, Kubernetes, Terraform. Недостающие навыки: ArgoCD.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (3 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (128501-202213 руб.) не пересекаются с вилкой вакансии (301543-420616 руб.).\", \"location_match\": \"Локация совпадает: Новосибирск.\", \"strengths\": [\"Совпавшие навыки: Docker, Kubernetes, Terraform\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD\"], \"recommendation\": \"consider\"}, \"confidence\": 0.86, \"match_score\": 77}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 10 лет, навыки AWS, Kubernetes, Prometheus, Terraform, зарплата 164483-237661, локация Регионы, формат Гибрид, занятость Полная. Вакансия: DevOps Engineer, опыт 5-8 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 172228-249424, локация Регионы, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): AWS, Kubernetes, Terraform. Недостающие навыки: ArgoCD, Docker, GitLab CI.\", \"experience_match\": \"Опыт кандидата 10 лет выше верхней границы (8 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (164483-237661 руб.) пересекаются с вилкой вакансии (172228-249424 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: AWS, Kubernetes, Terraform\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, Docker, GitLab CI\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 76}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 2 лет, навыки A/B testing, Pandas, Power BI, Python, Statistics, зарплата 147804-225329, локация Москва, формат Да, занятость Частичная. Вакансия: Data Analyst, опыт 4-5 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 157990-241982, локация Москва, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): A/B testing, Python. Недостающие навыки: Product metrics, SQL, Tableau.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (4-5 лет).\", \"salary_match\": \"Ожидания кандидата (147804-225329 руб.) пересекаются с вилкой вакансии (157990-241982 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: A/B testing, Python\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Product metrics, SQL, Tableau\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.79, \"match_score\": 60}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 6 лет, навыки NumPy, PyTorch, Python, TensorFlow, зарплата 238829-299883, локация Санкт-Петербург, формат Да, занятость Частичная. Вакансия: Python Backend Developer, опыт 2-4 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, зарплата 176048-237515, локация Казань, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/5): Python. Недостающие навыки: Docker, FastAPI, Kafka, PostgreSQL.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (238829-299883 руб.) не пересекаются с вилкой вакансии (176048-237515 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Python\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Docker, FastAPI, Kafka, PostgreSQL\"], \"recommendation\": \"consider\"}, \"confidence\": 0.76, \"match_score\": 52}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 2 лет, навыки CSS, HTML, Next.js, Redux, TypeScript, Vue, зарплата 182298-236750, локация Санкт-Петербург, формат Нет, занятость Частичная. Вакансия: Frontend Developer, опыт 2-4 лет, навыки CSS, Next.js, REST API, Redux, TypeScript, зарплата 146372-258822, локация Санкт-Петербург, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): CSS, Next.js, Redux, TypeScript. Недостающие навыки: REST API.\", \"experience_match\": \"Опыт кандидата 2 лет находится в диапазоне вакансии (2-4 лет).\", \"salary_match\": \"Ожидания кандидата (182298-236750 руб.) пересекаются с вилкой вакансии (146372-258822 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: CSS, Next.js, Redux, TypeScript\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: REST API\"], \"recommendation\": \"hire\"}, \"confidence\": 0.91, \"match_score\": 91}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 3 лет, навыки Airflow, Docker, NumPy, Pandas, Python, зарплата 293403-359230, локация Регионы, формат Гибрид, занятость Частичная. Вакансия: ML Engineer, опыт 5-8 лет, навыки Albumentations, Docker, MLOps, PyTorch, Python, зарплата 192520-294931, локация Регионы, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): Docker, Python. Недостающие навыки: Albumentations, MLOps, PyTorch.\", \"experience_match\": \"Опыт кандидата 3 лет ниже диапазона вакансии (5-8 лет).\", \"salary_match\": \"Ожидания кандидата (293403-359230 руб.) пересекаются с вилкой вакансии (192520-294931 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: Docker, Python\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, PyTorch\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.8, \"match_score\": 63}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 5 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 260512-304025, локация Регионы, формат Да, занятость Полная. Вакансия: DevOps Engineer, опыт 6-7 лет, навыки AWS, ArgoCD, Docker, Kubernetes, Terraform, зарплата 313915-368374, локация Регионы, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): AWS, Docker, Kubernetes, Terraform. Недостающие навыки: ArgoCD.\", \"experience_match\": \"Опыт кандидата 5 лет ниже диапазона вакансии (6-7 лет).\", \"salary_match\": \"Ожидания кандидата (260512-304025 руб.) не пересекаются с вилкой вакансии (313915-368374 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: AWS, Docker, Kubernetes, Terraform\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.86, \"match_score\": 78}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 5 лет, навыки Airflow, Docker, MLflow, Pandas, PyTorch, зарплата 220888-296539, локация Санкт-Петербург, формат Да, занятость Полная. Вакансия: ML Engineer, опыт 4-6 лет, навыки Docker, OpenCV, PyTorch, Python, зарплата 285691-389479, локация Санкт-Петербург, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): Docker, PyTorch. Недостающие навыки: OpenCV, Python.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (4-6 лет).\", \"salary_match\": \"Ожидания кандидата (220888-296539 руб.) пересекаются с вилкой вакансии (285691-389479 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: Docker, PyTorch\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: OpenCV, Python\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 75}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 3 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Terraform, зарплата 318587-375794, локация Регионы, формат Гибрид, занятость Полная. Вакансия: ML Engineer, опыт 4-5 лет, навыки Albumentations, MLOps, PyTorch, Python, зарплата 194053-282540, локация Санкт-Петербург, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: Albumentations, MLOps, PyTorch, Python.\", \"experience_match\": \"Опыт кандидата 3 лет ниже диапазона вакансии (4-5 лет).\", \"salary_match\": \"Ожидания кандидата (318587-375794 руб.) не пересекаются с вилкой вакансии (194053-282540 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, PyTorch, Python\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.71, \"match_score\": 40}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 2 лет, навыки Django, Docker, PostgreSQL, Python, Redis, зарплата 176840-229301, локация Екатеринбург, формат Гибрид, занятость Полная. Вакансия: Python Backend Developer, опыт 3-6 лет, навыки Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 183504-250480, локация Екатеринбург, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): Docker, PostgreSQL, Python, Redis. Недостающие навыки: FastAPI.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (3-6 лет).\", \"salary_match\": \"Ожидания кандидата (176840-229301 руб.) пересекаются с вилкой вакансии (183504-250480 руб.).\", \"location_match\": \"Локация совпадает: Екатеринбург.\", \"strengths\": [\"Совпавшие навыки: Docker, PostgreSQL, Python, Redis\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: FastAPI\", \"Недостаточный опыт\"], \"recommendation\": \"hire\"}, \"confidence\": 0.87, \"match_score\": 80}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 3 лет, навыки Docker, NumPy, PyTorch, Python, TensorFlow, зарплата 251522-292177, локация Москва, формат Нет, занятость Частичная. Вакансия: Frontend Developer, опыт 3-5 лет, навыки CSS, React, Redux, TypeScript, зарплата 279891-359417, локация Москва, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: CSS, React, Redux, TypeScript.\", \"experience_match\": \"Опыт кандидата 3 лет находится в диапазоне вакансии (3-5 лет).\", \"salary_match\": \"Ожидания кандидата (251522-292177 руб.) пересекаются с вилкой вакансии (279891-359417 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: CSS, React, Redux, TypeScript\"], \"recommendation\": \"consider\"}, \"confidence\": 0.77, \"match_score\": 55}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 8 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Terraform, зарплата 158232-240663, локация Москва, формат Нет, занятость Частичная. Вакансия: DevOps Engineer, опыт 5-6 лет, навыки ArgoCD, GitLab CI, Kubernetes, Terraform, зарплата 189203-291198, локация Москва, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): GitLab CI, Kubernetes, Terraform. Недостающие навыки: ArgoCD.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (158232-240663 руб.) пересекаются с вилкой вакансии (189203-291198 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: GitLab CI, Kubernetes, Terraform\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD\"], \"recommendation\": \"hire\"}, \"confidence\": 0.9, \"match_score\": 87}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 3 лет, навыки A/B testing, Pandas, Power BI, Python, SQL, Statistics, Tableau, зарплата 227682-308094, локация Санкт-Петербург, формат Нет, занятость Полная. Вакансия: Data Analyst, опыт 3-6 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 150856-243530, локация Санкт-Петербург, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): A/B testing, Python, SQL, Tableau. Недостающие навыки: Product metrics.\", \"experience_match\": \"Опыт кандидата 3 лет находится в диапазоне вакансии (3-6 лет).\", \"salary_match\": \"Ожидания кандидата (227682-308094 руб.) пересекаются с вилкой вакансии (150856-243530 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: A/B testing, Python, SQL, Tableau\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Product metrics\"], \"recommendation\": \"hire\"}, \"confidence\": 0.91, \"match_score\": 91}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 3 лет, навыки AWS, GitLab CI, Kubernetes, Terraform, зарплата 187634-267833, локация Екатеринбург, формат Гибрид, занятость Полная. Вакансия: DevOps Engineer, опыт 5-6 лет, навыки ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 222389-296407, локация Екатеринбург, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): GitLab CI, Kubernetes, Terraform. Недостающие навыки: ArgoCD, Docker.\", \"experience_match\": \"Опыт кандидата 3 лет ниже диапазона вакансии (5-6 лет).\", \"salary_match\": \"Ожидания кандидата (187634-267833 руб.) пересекаются с вилкой вакансии (222389-296407 руб.).\", \"location_match\": \"Локация совпадает: Екатеринбург.\", \"strengths\": [\"Совпавшие навыки: GitLab CI, Kubernetes, Terraform\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, Docker\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.84, \"match_score\": 72}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 3 лет, навыки Airflow, Docker, MLflow, NumPy, Pandas, Python, TensorFlow, зарплата 237219-313528, локация Казань, формат Нет, занятость Полная. Вакансия: ML Engineer, опыт 3-4 лет, навыки Albumentations, Docker, MLOps, OpenCV, зарплата 254688-367960, локация Казань, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/4): Docker. Недостающие навыки: Albumentations, MLOps, OpenCV.\", \"experience_match\": \"Опыт кандидата 3 лет находится в диапазоне вакансии (3-4 лет).\", \"salary_match\": \"Ожидания кандидата (237219-313528 руб.) пересекаются с вилкой вакансии (254688-367960 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: Docker\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV\"], \"recommendation\": \"consider\"}, \"confidence\": 0.81, \"match_score\": 64}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 7 лет, навыки JavaScript, React, TypeScript, Vue, зарплата 194231-246952, локация Москва, формат Да, занятость Полная. Вакансия: Frontend Developer, опыт 6-7 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 210134-308311, локация Москва, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/6): React, TypeScript. Недостающие навыки: CSS, Next.js, REST API, Redux.\", \"experience_match\": \"Опыт кандидата 7 лет находится в диапазоне вакансии (6-7 лет).\", \"salary_match\": \"Ожидания кандидата (194231-246952 руб.) пересекаются с вилкой вакансии (210134-308311 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: React, TypeScript\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: CSS, Next.js, REST API, Redux\"], \"recommendation\": \"consider\"}, \"confidence\": 0.82, \"match_score\": 68}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 6 лет, навыки CSS, HTML, JavaScript, Next.js, React, TypeScript, Vue, зарплата 141165-207425, локация Регионы, формат Да, занятость Частичная. Вакансия: Frontend Developer, опыт 5-6 лет, навыки CSS, Next.js, REST API, React, Redux, зарплата 120826-168162, локация Регионы, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): CSS, Next.js, React. Недостающие навыки: REST API, Redux.\", \"experience_match\": \"Опыт кандидата 6 лет находится в диапазоне вакансии (5-6 лет).\", \"salary_match\": \"Ожидания кандидата (141165-207425 руб.) пересекаются с вилкой вакансии (120826-168162 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: CSS, Next.js, React\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: REST API, Redux\"], \"recommendation\": \"hire\"}, \"confidence\": 0.87, \"match_score\": 80}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 3 лет, навыки CSS, Next.js, React, Redux, TypeScript, Vue, зарплата 195030-263457, локация Санкт-Петербург, формат Нет, занятость Полная. Вакансия: Frontend Developer, опыт 1-3 лет, навыки CSS, REST API, React, Redux, TypeScript, зарплата 109806-187832, локация Санкт-Петербург, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): CSS, React, Redux, TypeScript. Недостающие навыки: REST API.\", \"experience_match\": \"Опыт кандидата 3 лет находится в диапазоне вакансии (1-3 лет).\", \"salary_match\": \"Ожидания кандидата (195030-263457 руб.) не пересекаются с вилкой вакансии (109806-187832 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: CSS, React, Redux, TypeScript\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: REST API\"], \"recommendation\": \"hire\"}, \"confidence\": 0.88, \"match_score\": 82}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 4 лет, навыки Airflow, NumPy, PyTorch, Python, зарплата 259454-326468, локация Регионы, формат Нет, занятость Частичная. Вакансия: ML Engineer, опыт 3-4 лет, навыки Albumentations, Docker, MLOps, PyTorch, Python, зарплата 151763-265653, локация Регионы, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): PyTorch, Python. Недостающие навыки: Albumentations, Docker, MLOps.\", \"experience_match\": \"Опыт кандидата 4 лет находится в диапазоне вакансии (3-4 лет).\", \"salary_match\": \"Ожидания кандидата (259454-326468 руб.) пересекаются с вилкой вакансии (151763-265653 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: PyTorch, Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, Docker, MLOps\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 71}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 8 лет, навыки Airflow, Docker, NumPy, Pandas, PyTorch, Python, TensorFlow, зарплата 132569-199674, локация Москва, формат Гибрид, занятость Полная. Вакансия: ML Engineer, опыт 5-8 лет, навыки Albumentations, Docker, MLOps, PyTorch, Python, зарплата 261374-312929, локация Санкт-Петербург, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): Docker, PyTorch, Python. Недостающие навыки: Albumentations, MLOps.\", \"experience_match\": \"Опыт кандидата 8 лет находится в диапазоне вакансии (5-8 лет).\", \"salary_match\": \"Ожидания кандидата (132569-199674 руб.) не пересекаются с вилкой вакансии (261374-312929 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker, PyTorch, Python\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 71}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 5 лет, навыки HTML, JavaScript, Next.js, Redux, TypeScript, Vue, зарплата 125043-190078, локация Казань, формат Нет, занятость Полная. Вакансия: Frontend Developer, опыт 1-2 лет, навыки REST API, React, Redux, TypeScript, зарплата 130457-170640, локация Казань, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): Redux, TypeScript. Недостающие навыки: REST API, React.\", \"experience_match\": \"Опыт кандидата 5 лет выше верхней границы (2 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (125043-190078 руб.) пересекаются с вилкой вакансии (130457-170640 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: Redux, TypeScript\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: REST API, React\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 74}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 5 лет, навыки Airflow, Docker, Pandas, TensorFlow, зарплата 189075-232539, локация Санкт-Петербург, формат Гибрид, занятость Полная. Вакансия: ML Engineer, опыт 1-4 лет, навыки Albumentations, OpenCV, PyTorch, Python, зарплата 275724-395275, локация Казань, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: Albumentations, OpenCV, PyTorch, Python.\", \"experience_match\": \"Опыт кандидата 5 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (189075-232539 руб.) не пересекаются с вилкой вакансии (275724-395275 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, OpenCV, PyTorch, Python\"], \"recommendation\": \"reject\"}, \"confidence\": 0.72, \"match_score\": 43}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 2 лет, навыки A/B testing, Power BI, Python, SQL, Statistics, Tableau, зарплата 223876-308350, локация Москва, формат Нет, занятость Полная. Вакансия: Data Analyst, опыт 2-5 лет, навыки A/B testing, Python, SQL, Tableau, зарплата 163575-234548, локация Москва, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/4): A/B testing, Python, SQL, Tableau. Недостающие навыки: нет.\", \"experience_match\": \"Опыт кандидата 2 лет находится в диапазоне вакансии (2-5 лет).\", \"salary_match\": \"Ожидания кандидата (223876-308350 руб.) пересекаются с вилкой вакансии (163575-234548 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: A/B testing, Python, SQL, Tableau\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [], \"recommendation\": \"hire\"}, \"confidence\": 0.93, \"match_score\": 96}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 4 лет, навыки CSS, JavaScript, Next.js, React, Redux, TypeScript, Vue, зарплата 139082-215019, локация Санкт-Петербург, формат Да, занятость Полная. Вакансия: Frontend Developer, опыт 3-4 лет, навыки CSS, REST API, React, Redux, TypeScript, зарплата 164842-261092, локация Санкт-Петербург, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): CSS, React, Redux, TypeScript. Недостающие навыки: REST API.\", \"experience_match\": \"Опыт кандидата 4 лет находится в диапазоне вакансии (3-4 лет).\", \"salary_match\": \"Ожидания кандидата (139082-215019 руб.) пересекаются с вилкой вакансии (164842-261092 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: CSS, React, Redux, TypeScript\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: REST API\"], \"recommendation\": \"hire\"}, \"confidence\": 0.91, \"match_score\": 91}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 1 лет, навыки A/B testing, Pandas, Power BI, SQL, Statistics, Tableau, зарплата 101135-169249, локация Санкт-Петербург, формат Да, занятость Полная. Вакансия: Data Analyst, опыт 3-6 лет, навыки A/B testing, Product metrics, Python, Tableau, зарплата 206499-313234, локация Санкт-Петербург, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): A/B testing, Tableau. Недостающие навыки: Product metrics, Python.\", \"experience_match\": \"Опыт кандидата 1 лет ниже диапазона вакансии (3-6 лет).\", \"salary_match\": \"Ожидания кандидата (101135-169249 руб.) не пересекаются с вилкой вакансии (206499-313234 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: A/B testing, Tableau\"], \"weaknesses\": [\"Недостающие навыки: Product metrics, Python\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.75, \"match_score\": 50}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 6 лет, навыки HTML, React, Redux, TypeScript, Vue, зарплата 192471-279546, локация Регионы, формат Гибрид, занятость Полная. Вакансия: Frontend Developer, опыт 3-5 лет, навыки CSS, Next.js, React, TypeScript, зарплата 171411-283309, локация Регионы, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): React, TypeScript. Недостающие навыки: CSS, Next.js.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (192471-279546 руб.) пересекаются с вилкой вакансии (171411-283309 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: React, TypeScript\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: CSS, Next.js\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 74}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 7 лет, навыки MLflow, NumPy, PyTorch, Python, зарплата 230348-289458, локация Москва, формат Да, занятость Полная. Вакансия: Frontend Developer, опыт 2-3 лет, навыки Next.js, REST API, React, Redux, TypeScript, зарплата 290217-388950, локация Москва, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/5): нет. Недостающие навыки: Next.js, REST API, React, Redux, TypeScript.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (3 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (230348-289458 руб.) не пересекаются с вилкой вакансии (290217-388950 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Next.js, REST API, React, Redux, TypeScript\"], \"recommendation\": \"reject\"}, \"confidence\": 0.73, \"match_score\": 45}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 5 лет, навыки MLflow, NumPy, Pandas, PyTorch, Python, TensorFlow, зарплата 289072-378705, локация Санкт-Петербург, формат Да, занятость Полная. Вакансия: ML Engineer, опыт 2-5 лет, навыки Albumentations, MLOps, OpenCV, Python, зарплата 189851-295255, локация Санкт-Петербург, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/4): Python. Недостающие навыки: Albumentations, MLOps, OpenCV.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (2-5 лет).\", \"salary_match\": \"Ожидания кандидата (289072-378705 руб.) пересекаются с вилкой вакансии (189851-295255 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV\"], \"recommendation\": \"consider\"}, \"confidence\": 0.81, \"match_score\": 66}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 7 лет, навыки A/B testing, Python, SQL, Tableau, зарплата 218761-261385, локация Регионы, формат Нет, занятость Полная. Вакансия: Python Backend Developer, опыт 3-6 лет, навыки FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 234792-300652, локация Москва, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/5): Python. Недостающие навыки: FastAPI, Kafka, PostgreSQL, Redis.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (218761-261385 руб.) пересекаются с вилкой вакансии (234792-300652 руб.).\", \"location_match\": \"Локация не совпадает (Регионы vs Москва).\", \"strengths\": [\"Совпавшие навыки: Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: FastAPI, Kafka, PostgreSQL, Redis\", \"Несовпадение по локации/формату\"], \"recommendation\": \"consider\"}, \"confidence\": 0.77, \"match_score\": 54}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 7 лет, навыки A/B testing, Pandas, Python, SQL, Statistics, Tableau, зарплата 92673-153413, локация Санкт-Петербург, формат Гибрид, занятость Частичная. Вакансия: Data Analyst, опыт 2-3 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 150870-208209, локация Казань, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): A/B testing, Python, SQL, Tableau. Недостающие навыки: Product metrics.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (3 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (92673-153413 руб.) пересекаются с вилкой вакансии (150870-208209 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: A/B testing, Python, SQL, Tableau\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Product metrics\"], \"recommendation\": \"hire\"}, \"confidence\": 0.9, \"match_score\": 87}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 5 лет, навыки HTML, JavaScript, Redux, TypeScript, зарплата 184235-272737, локация Регионы, формат Гибрид, занятость Частичная. Вакансия: Frontend Developer, опыт 5-7 лет, навыки CSS, Next.js, REST API, Redux, TypeScript, зарплата 143249-224607, локация Санкт-Петербург, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): Redux, TypeScript. Недостающие навыки: CSS, Next.js, REST API.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (5-7 лет).\", \"salary_match\": \"Ожидания кандидата (184235-272737 руб.) пересекаются с вилкой вакансии (143249-224607 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Redux, TypeScript\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: CSS, Next.js, REST API\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 71}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 5 лет, навыки HTML, JavaScript, Next.js, Redux, TypeScript, Vue, зарплата 216892-283427, локация Москва, формат Нет, занятость Полная. Вакансия: Frontend Developer, опыт 3-5 лет, навыки Next.js, REST API, React, Redux, TypeScript, зарплата 167558-252665, локация Москва, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): Next.js, Redux, TypeScript. Недостающие навыки: REST API, React.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (3-5 лет).\", \"salary_match\": \"Ожидания кандидата (216892-283427 руб.) пересекаются с вилкой вакансии (167558-252665 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: Next.js, Redux, TypeScript\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: REST API, React\"], \"recommendation\": \"hire\"}, \"confidence\": 0.88, \"match_score\": 82}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 6 лет, навыки HTML, JavaScript, Next.js, Redux, зарплата 81089-139677, локация Москва, формат Гибрид, занятость Частичная. Вакансия: Frontend Developer, опыт 1-3 лет, навыки Next.js, REST API, Redux, TypeScript, зарплата 104912-205172, локация Регионы, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): Next.js, Redux. Недостающие навыки: REST API, TypeScript.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (3 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (81089-139677 руб.) пересекаются с вилкой вакансии (104912-205172 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Next.js, Redux\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: REST API, TypeScript\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 76}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 9 лет, навыки AsyncIO, Django, FastAPI, PostgreSQL, Redis, зарплата 240440-298827, локация Москва, формат Да, занятость Полная. Вакансия: Frontend Developer, опыт 2-3 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 223546-328574, локация Екатеринбург, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/6): нет. Недостающие навыки: CSS, Next.js, REST API, React, Redux, TypeScript.\", \"experience_match\": \"Опыт кандидата 9 лет выше верхней границы (3 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (240440-298827 руб.) пересекаются с вилкой вакансии (223546-328574 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: CSS, Next.js, REST API, React, Redux\"], \"recommendation\": \"consider\"}, \"confidence\": 0.76, \"match_score\": 53}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 2 лет, навыки GitLab CI, Linux, Prometheus, Terraform, зарплата 182060-216838, локация Москва, формат Гибрид, занятость Полная. Вакансия: DevOps Engineer, опыт 4-7 лет, навыки AWS, Docker, GitLab CI, Kubernetes, зарплата 307277-404755, локация Москва, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/4): GitLab CI. Недостающие навыки: AWS, Docker, Kubernetes.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (4-7 лет).\", \"salary_match\": \"Ожидания кандидата (182060-216838 руб.) не пересекаются с вилкой вакансии (307277-404755 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: GitLab CI\"], \"weaknesses\": [\"Недостающие навыки: AWS, Docker, Kubernetes\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.73, \"match_score\": 45}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 8 лет, навыки CSS, HTML, JavaScript, Next.js, React, TypeScript, Vue, зарплата 154490-240307, локация Казань, формат Нет, занятость Частичная. Вакансия: ML Engineer, опыт 4-6 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, зарплата 164355-215714, локация Казань, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/5): нет. Недостающие навыки: Albumentations, Docker, MLOps, OpenCV, PyTorch.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (154490-240307 руб.) пересекаются с вилкой вакансии (164355-215714 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, Docker, MLOps, OpenCV, PyTorch\"], \"recommendation\": \"consider\"}, \"confidence\": 0.76, \"match_score\": 53}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 3 лет, навыки Power BI, Python, Statistics, Tableau, зарплата 134305-202776, локация Москва, формат Гибрид, занятость Частичная. Вакансия: Data Analyst, опыт 3-4 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 147798-240003, локация Москва, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): Python, Tableau. Недостающие навыки: A/B testing, Product metrics, SQL.\", \"experience_match\": \"Опыт кандидата 3 лет находится в диапазоне вакансии (3-4 лет).\", \"salary_match\": \"Ожидания кандидата (134305-202776 руб.) пересекаются с вилкой вакансии (147798-240003 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: Python, Tableau\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, SQL\"], \"recommendation\": \"consider\"}, \"confidence\": 0.84, \"match_score\": 73}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 4 лет, навыки MLflow, PyTorch, Python, TensorFlow, зарплата 222166-260083, локация Москва, формат Да, занятость Частичная. Вакансия: ML Engineer, опыт 5-7 лет, навыки Albumentations, Docker, MLOps, PyTorch, Python, зарплата 200924-268944, локация Москва, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): PyTorch, Python. Недостающие навыки: Albumentations, Docker, MLOps.\", \"experience_match\": \"Опыт кандидата 4 лет ниже диапазона вакансии (5-7 лет).\", \"salary_match\": \"Ожидания кандидата (222166-260083 руб.) пересекаются с вилкой вакансии (200924-268944 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: PyTorch, Python\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, Docker, MLOps\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.82, \"match_score\": 68}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 8 лет, навыки A/B testing, Pandas, Power BI, SQL, Statistics, зарплата 214140-249058, локация Санкт-Петербург, формат Нет, занятость Полная. Вакансия: Frontend Developer, опыт 2-3 лет, навыки CSS, Next.js, REST API, React, зарплата 136552-210092, локация Казань, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: CSS, Next.js, REST API, React.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (3 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (214140-249058 руб.) не пересекаются с вилкой вакансии (136552-210092 руб.).\", \"location_match\": \"Локация не совпадает (Санкт-Петербург vs Казань).\", \"strengths\": [\"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: CSS, Next.js, REST API, React\", \"Несовпадение по локации/формату\"], \"recommendation\": \"reject\"}, \"confidence\": 0.71, \"match_score\": 39}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 3 лет, навыки Pandas, Power BI, Python, Statistics, Tableau, зарплата 227215-290339, локация Регионы, формат Нет, занятость Полная. Вакансия: Data Analyst, опыт 2-4 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 239753-333379, локация Регионы, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): Python, Tableau. Недостающие навыки: A/B testing, Product metrics, SQL.\", \"experience_match\": \"Опыт кандидата 3 лет находится в диапазоне вакансии (2-4 лет).\", \"salary_match\": \"Ожидания кандидата (227215-290339 руб.) пересекаются с вилкой вакансии (239753-333379 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: Python, Tableau\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, SQL\"], \"recommendation\": \"consider\"}, \"confidence\": 0.84, \"match_score\": 73}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 4 лет, навыки AsyncIO, Django, FastAPI, PostgreSQL, Python, Redis, зарплата 139342-205742, локация Новосибирск, формат Гибрид, занятость Полная. Вакансия: Python Backend Developer, опыт 5-6 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 220819-308042, локация Новосибирск, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/6): FastAPI, PostgreSQL, Python, Redis. Недостающие навыки: Docker, Kafka.\", \"experience_match\": \"Опыт кандидата 4 лет ниже диапазона вакансии (5-6 лет).\", \"salary_match\": \"Ожидания кандидата (139342-205742 руб.) не пересекаются с вилкой вакансии (220819-308042 руб.).\", \"location_match\": \"Локация совпадает: Новосибирск.\", \"strengths\": [\"Совпавшие навыки: FastAPI, PostgreSQL, Python, Redis\"], \"weaknesses\": [\"Недостающие навыки: Docker, Kafka\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 71}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 5 лет, навыки Docker, PostgreSQL, Python, Redis, зарплата 252371-288396, локация Санкт-Петербург, формат Гибрид, занятость Полная. Вакансия: Python Backend Developer, опыт 2-3 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 134911-252785, локация Москва, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/6): Docker, PostgreSQL, Python, Redis. Недостающие навыки: FastAPI, Kafka.\", \"experience_match\": \"Опыт кандидата 5 лет выше верхней границы (3 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (252371-288396 руб.) пересекаются с вилкой вакансии (134911-252785 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker, PostgreSQL, Python, Redis\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: FastAPI, Kafka\"], \"recommendation\": \"hire\"}, \"confidence\": 0.88, \"match_score\": 83}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 3 лет, навыки AWS, Docker, Linux, Prometheus, Terraform, зарплата 189601-267261, локация Санкт-Петербург, формат Гибрид, занятость Полная. Вакансия: ML Engineer, опыт 3-4 лет, навыки Albumentations, Docker, MLOps, OpenCV, Python, зарплата 276527-325189, локация Новосибирск, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/5): Docker. Недостающие навыки: Albumentations, MLOps, OpenCV, Python.\", \"experience_match\": \"Опыт кандидата 3 лет находится в диапазоне вакансии (3-4 лет).\", \"salary_match\": \"Ожидания кандидата (189601-267261 руб.) не пересекаются с вилкой вакансии (276527-325189 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV, Python\"], \"recommendation\": \"consider\"}, \"confidence\": 0.77, \"match_score\": 55}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 2 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 313947-387263, локация Регионы, формат Да, занятость Полная. Вакансия: DevOps Engineer, опыт 6-7 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 226830-315549, локация Регионы, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (5/6): AWS, Docker, GitLab CI, Kubernetes, Terraform. Недостающие навыки: ArgoCD.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (6-7 лет).\", \"salary_match\": \"Ожидания кандидата (313947-387263 руб.) пересекаются с вилкой вакансии (226830-315549 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: AWS, Docker, GitLab CI, Kubernetes, Terraform\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 75}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 7 лет, навыки Airflow, Docker, MLflow, NumPy, Pandas, PyTorch, TensorFlow, зарплата 225704-283405, локация Регионы, формат Да, занятость Частичная. Вакансия: ML Engineer, опыт 2-4 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, Python, зарплата 272459-392333, локация Регионы, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/6): Docker, PyTorch. Недостающие навыки: Albumentations, MLOps, OpenCV, Python.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (225704-283405 руб.) пересекаются с вилкой вакансии (272459-392333 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: Docker, PyTorch\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV, Python\"], \"recommendation\": \"consider\"}, \"confidence\": 0.81, \"match_score\": 66}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 5 лет, навыки A/B testing, Pandas, Power BI, Python, SQL, Statistics, зарплата 138262-194891, локация Казань, формат Нет, занятость Частичная. Вакансия: Data Analyst, опыт 4-5 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 183022-286375, локация Санкт-Петербург, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): A/B testing, Python, SQL. Недостающие навыки: Product metrics, Tableau.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (4-5 лет).\", \"salary_match\": \"Ожидания кандидата (138262-194891 руб.) пересекаются с вилкой вакансии (183022-286375 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: A/B testing, Python, SQL\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Product metrics, Tableau\"], \"recommendation\": \"hire\"}, \"confidence\": 0.88, \"match_score\": 82}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 3 лет, навыки CSS, HTML, JavaScript, Redux, TypeScript, Vue, зарплата 218124-273563, локация Москва, формат Нет, занятость Частичная. Вакансия: Frontend Developer, опыт 4-6 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 112770-185957, локация Москва, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): CSS, Redux, TypeScript. Недостающие навыки: Next.js, REST API, React.\", \"experience_match\": \"Опыт кандидата 3 лет ниже диапазона вакансии (4-6 лет).\", \"salary_match\": \"Ожидания кандидата (218124-273563 руб.) не пересекаются с вилкой вакансии (112770-185957 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: CSS, Redux, TypeScript\"], \"weaknesses\": [\"Недостающие навыки: Next.js, REST API, React\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.8, \"match_score\": 63}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 4 лет, навыки A/B testing, Pandas, Power BI, Python, SQL, Statistics, Tableau, зарплата 136658-197338, локация Регионы, формат Гибрид, занятость Частичная. Вакансия: Frontend Developer, опыт 1-4 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 210941-275571, локация Москва, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/6): нет. Недостающие навыки: CSS, Next.js, REST API, React, Redux, TypeScript.\", \"experience_match\": \"Опыт кандидата 4 лет находится в диапазоне вакансии (1-4 лет).\", \"salary_match\": \"Ожидания кандидата (136658-197338 руб.) не пересекаются с вилкой вакансии (210941-275571 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: CSS, Next.js, REST API, React, Redux\"], \"recommendation\": \"reject\"}, \"confidence\": 0.73, \"match_score\": 46}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 8 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 295349-338268, локация Новосибирск, формат Нет, занятость Частичная. Вакансия: Python Backend Developer, опыт 3-4 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, зарплата 316257-398291, локация Москва, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/5): Docker. Недостающие навыки: FastAPI, Kafka, PostgreSQL, Python.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (295349-338268 руб.) пересекаются с вилкой вакансии (316257-398291 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: FastAPI, Kafka, PostgreSQL, Python\"], \"recommendation\": \"consider\"}, \"confidence\": 0.79, \"match_score\": 60}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 5 лет, навыки Airflow, Docker, MLflow, Python, TensorFlow, зарплата 119951-155866, локация Санкт-Петербург, формат Нет, занятость Частичная. Вакансия: ML Engineer, опыт 2-4 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, Python, зарплата 154724-225214, локация Санкт-Петербург, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/6): Docker, Python. Недостающие навыки: Albumentations, MLOps, OpenCV, PyTorch.\", \"experience_match\": \"Опыт кандидата 5 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (119951-155866 руб.) пересекаются с вилкой вакансии (154724-225214 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: Docker, Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV, PyTorch\"], \"recommendation\": \"consider\"}, \"confidence\": 0.81, \"match_score\": 66}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 6 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 305899-384634, локация Регионы, формат Да, занятость Полная. Вакансия: DevOps Engineer, опыт 3-4 лет, навыки AWS, ArgoCD, Docker, Kubernetes, Terraform, зарплата 151864-267621, локация Регионы, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): AWS, Docker, Kubernetes, Terraform. Недостающие навыки: ArgoCD.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (305899-384634 руб.) не пересекаются с вилкой вакансии (151864-267621 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: AWS, Docker, Kubernetes, Terraform\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD\"], \"recommendation\": \"consider\"}, \"confidence\": 0.87, \"match_score\": 79}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 5 лет, навыки A/B testing, Pandas, Power BI, Python, SQL, Statistics, Tableau, зарплата 231818-309004, локация Санкт-Петербург, формат Да, занятость Полная. Вакансия: Data Analyst, опыт 5-8 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 219465-298156, локация Санкт-Петербург, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): A/B testing, Python, SQL, Tableau. Недостающие навыки: Product metrics.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (5-8 лет).\", \"salary_match\": \"Ожидания кандидата (231818-309004 руб.) пересекаются с вилкой вакансии (219465-298156 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: A/B testing, Python, SQL, Tableau\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Product metrics\"], \"recommendation\": \"hire\"}, \"confidence\": 0.91, \"match_score\": 89}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 7 лет, навыки CSS, Next.js, Redux, TypeScript, зарплата 217106-287475, локация Санкт-Петербург, формат Нет, занятость Полная. Вакансия: Frontend Developer, опыт 6-7 лет, навыки CSS, Next.js, REST API, TypeScript, зарплата 237428-309703, локация Санкт-Петербург, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): CSS, Next.js, TypeScript. Недостающие навыки: REST API.\", \"experience_match\": \"Опыт кандидата 7 лет находится в диапазоне вакансии (6-7 лет).\", \"salary_match\": \"Ожидания кандидата (217106-287475 руб.) пересекаются с вилкой вакансии (237428-309703 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: CSS, Next.js, TypeScript\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: REST API\"], \"recommendation\": \"hire\"}, \"confidence\": 0.9, \"match_score\": 88}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 8 лет, навыки HTML, JavaScript, Next.js, Redux, TypeScript, Vue, зарплата 163124-243545, локация Казань, формат Гибрид, занятость Частичная. Вакансия: Frontend Developer, опыт 3-5 лет, навыки CSS, Next.js, React, Redux, TypeScript, зарплата 112235-225245, локация Москва, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): Next.js, Redux, TypeScript. Недостающие навыки: CSS, React.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (163124-243545 руб.) пересекаются с вилкой вакансии (112235-225245 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Next.js, Redux, TypeScript\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: CSS, React\"], \"recommendation\": \"hire\"}, \"confidence\": 0.87, \"match_score\": 80}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 4 лет, навыки CSS, HTML, Next.js, React, Redux, TypeScript, зарплата 83946-144108, локация Регионы, формат Да, занятость Частичная. Вакансия: Python Backend Developer, опыт 4-6 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 214785-319945, локация Москва, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/6): нет. Недостающие навыки: Docker, FastAPI, Kafka, PostgreSQL, Python, Redis.\", \"experience_match\": \"Опыт кандидата 4 лет находится в диапазоне вакансии (4-6 лет).\", \"salary_match\": \"Ожидания кандидата (83946-144108 руб.) не пересекаются с вилкой вакансии (214785-319945 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Docker, FastAPI, Kafka, PostgreSQL, Python\"], \"recommendation\": \"reject\"}, \"confidence\": 0.73, \"match_score\": 44}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 2 лет, навыки AsyncIO, Docker, PostgreSQL, Python, зарплата 266166-342256, локация Москва, формат Да, занятость Полная. Вакансия: Python Backend Developer, опыт 5-8 лет, навыки FastAPI, Kafka, PostgreSQL, Python, зарплата 132760-204837, локация Москва, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): PostgreSQL, Python. Недостающие навыки: FastAPI, Kafka.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (5-8 лет).\", \"salary_match\": \"Ожидания кандидата (266166-342256 руб.) не пересекаются с вилкой вакансии (132760-204837 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: PostgreSQL, Python\"], \"weaknesses\": [\"Недостающие навыки: FastAPI, Kafka\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.77, \"match_score\": 54}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 4 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 192380-252098, локация Москва, формат Да, занятость Полная. Вакансия: DevOps Engineer, опыт 6-8 лет, навыки ArgoCD, Docker, GitLab CI, Kubernetes, зарплата 315314-422533, локация Новосибирск, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): Docker, GitLab CI, Kubernetes. Недостающие навыки: ArgoCD.\", \"experience_match\": \"Опыт кандидата 4 лет ниже диапазона вакансии (6-8 лет).\", \"salary_match\": \"Ожидания кандидата (192380-252098 руб.) не пересекаются с вилкой вакансии (315314-422533 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker, GitLab CI, Kubernetes\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 70}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 5 лет, навыки Airflow, MLflow, NumPy, Pandas, PyTorch, Python, TensorFlow, зарплата 115488-173782, локация Москва, формат Гибрид, занятость Полная. Вакансия: ML Engineer, опыт 2-4 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, Python, зарплата 218605-277477, локация Москва, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/6): PyTorch, Python. Недостающие навыки: Albumentations, Docker, MLOps, OpenCV.\", \"experience_match\": \"Опыт кандидата 5 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (115488-173782 руб.) не пересекаются с вилкой вакансии (218605-277477 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: PyTorch, Python\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, Docker, MLOps, OpenCV\"], \"recommendation\": \"consider\"}, \"confidence\": 0.78, \"match_score\": 58}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 4 лет, навыки A/B testing, Pandas, Power BI, SQL, Statistics, Tableau, зарплата 135872-181536, локация Регионы, формат Да, занятость Частичная. Вакансия: Data Analyst, опыт 4-7 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 236024-319476, локация Регионы, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): A/B testing, SQL, Tableau. Недостающие навыки: Product metrics, Python.\", \"experience_match\": \"Опыт кандидата 4 лет находится в диапазоне вакансии (4-7 лет).\", \"salary_match\": \"Ожидания кандидата (135872-181536 руб.) не пересекаются с вилкой вакансии (236024-319476 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: A/B testing, SQL, Tableau\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Product metrics, Python\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 71}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 7 лет, навыки A/B testing, Pandas, Power BI, Python, SQL, Statistics, Tableau, зарплата 81532-137414, локация Санкт-Петербург, формат Нет, занятость Частичная. Вакансия: Data Analyst, опыт 5-6 лет, навыки A/B testing, Product metrics, SQL, Tableau, зарплата 140586-182329, локация Санкт-Петербург, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): A/B testing, SQL, Tableau. Недостающие навыки: Product metrics.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (81532-137414 руб.) не пересекаются с вилкой вакансии (140586-182329 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: A/B testing, SQL, Tableau\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Product metrics\"], \"recommendation\": \"consider\"}, \"confidence\": 0.87, \"match_score\": 79}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 2 лет, навыки Kubernetes, Linux, Prometheus, Terraform, зарплата 285327-349477, локация Екатеринбург, формат Гибрид, занятость Частичная. Вакансия: DevOps Engineer, опыт 6-7 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 247325-290489, локация Екатеринбург, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/6): Kubernetes, Terraform. Недостающие навыки: AWS, ArgoCD, Docker, GitLab CI.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (6-7 лет).\", \"salary_match\": \"Ожидания кандидата (285327-349477 руб.) пересекаются с вилкой вакансии (247325-290489 руб.).\", \"location_match\": \"Локация совпадает: Екатеринбург.\", \"strengths\": [\"Совпавшие навыки: Kubernetes, Terraform\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: AWS, ArgoCD, Docker, GitLab CI\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.76, \"match_score\": 53}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 3 лет, навыки Airflow, Docker, MLflow, NumPy, PyTorch, Python, зарплата 271712-356418, локация Москва, формат Да, занятость Частичная. Вакансия: Python Backend Developer, опыт 2-3 лет, навыки Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 206254-271724, локация Регионы, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): Docker, Python. Недостающие навыки: FastAPI, PostgreSQL, Redis.\", \"experience_match\": \"Опыт кандидата 3 лет находится в диапазоне вакансии (2-3 лет).\", \"salary_match\": \"Ожидания кандидата (271712-356418 руб.) пересекаются с вилкой вакансии (206254-271724 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker, Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: FastAPI, PostgreSQL, Redis\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 71}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 2 лет, навыки HTML, JavaScript, Next.js, React, Redux, TypeScript, Vue, зарплата 183213-219541, локация Москва, формат Гибрид, занятость Частичная. Вакансия: Frontend Developer, опыт 1-2 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 197687-254741, локация Москва, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/6): Next.js, React, Redux, TypeScript. Недостающие навыки: CSS, REST API.\", \"experience_match\": \"Опыт кандидата 2 лет находится в диапазоне вакансии (1-2 лет).\", \"salary_match\": \"Ожидания кандидата (183213-219541 руб.) пересекаются с вилкой вакансии (197687-254741 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: Next.js, React, Redux, TypeScript\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: CSS, REST API\"], \"recommendation\": \"hire\"}, \"confidence\": 0.88, \"match_score\": 83}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 10 лет, навыки AWS, GitLab CI, Kubernetes, Prometheus, зарплата 280947-353108, локация Новосибирск, формат Да, занятость Частичная. Вакансия: DevOps Engineer, опыт 5-6 лет, навыки AWS, ArgoCD, Docker, GitLab CI, зарплата 177852-275730, локация Новосибирск, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): AWS, GitLab CI. Недостающие навыки: ArgoCD, Docker.\", \"experience_match\": \"Опыт кандидата 10 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (280947-353108 руб.) не пересекаются с вилкой вакансии (177852-275730 руб.).\", \"location_match\": \"Локация совпадает: Новосибирск.\", \"strengths\": [\"Совпавшие навыки: AWS, GitLab CI\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, Docker\"], \"recommendation\": \"consider\"}, \"confidence\": 0.82, \"match_score\": 68}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 1 лет, навыки Airflow, Docker, MLflow, NumPy, Pandas, PyTorch, Python, зарплата 117035-159270, локация Регионы, формат Да, занятость Полная. Вакансия: ML Engineer, опыт 5-7 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, Python, зарплата 192166-241274, локация Регионы, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): Docker, PyTorch, Python. Недостающие навыки: Albumentations, MLOps, OpenCV.\", \"experience_match\": \"Опыт кандидата 1 лет ниже диапазона вакансии (5-7 лет).\", \"salary_match\": \"Ожидания кандидата (117035-159270 руб.) не пересекаются с вилкой вакансии (192166-241274 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: Docker, PyTorch, Python\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.74, \"match_score\": 47}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 5 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Prometheus, Terraform, зарплата 209421-242585, локация Регионы, формат Нет, занятость Частичная. Вакансия: Data Analyst, опыт 4-6 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 297883-405736, локация Екатеринбург, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/5): нет. Недостающие навыки: A/B testing, Product metrics, Python, SQL, Tableau.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (4-6 лет).\", \"salary_match\": \"Ожидания кандидата (209421-242585 руб.) не пересекаются с вилкой вакансии (297883-405736 руб.).\", \"location_match\": \"Локация не совпадает (Регионы vs Екатеринбург).\", \"strengths\": [\"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, Python, SQL, Tableau\", \"Несовпадение по локации/формату\"], \"recommendation\": \"reject\"}, \"confidence\": 0.71, \"match_score\": 40}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 5 лет, навыки Airflow, Docker, MLflow, NumPy, PyTorch, Python, TensorFlow, зарплата 122736-191659, локация Москва, формат Да, занятость Частичная. Вакансия: ML Engineer, опыт 5-8 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, Python, зарплата 152572-206742, локация Москва, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): Docker, PyTorch, Python. Недостающие навыки: Albumentations, MLOps, OpenCV.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (5-8 лет).\", \"salary_match\": \"Ожидания кандидата (122736-191659 руб.) пересекаются с вилкой вакансии (152572-206742 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: Docker, PyTorch, Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV\"], \"recommendation\": \"consider\"}, \"confidence\": 0.86, \"match_score\": 77}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 6 лет, навыки CSS, HTML, React, TypeScript, Vue, зарплата 168311-234364, локация Казань, формат Нет, занятость Полная. Вакансия: Frontend Developer, опыт 2-4 лет, навыки CSS, REST API, React, Redux, TypeScript, зарплата 106675-203594, локация Казань, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): CSS, React, TypeScript. Недостающие навыки: REST API, Redux.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (168311-234364 руб.) пересекаются с вилкой вакансии (106675-203594 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: CSS, React, TypeScript\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: REST API, Redux\"], \"recommendation\": \"consider\"}, \"confidence\": 0.86, \"match_score\": 78}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 6 лет, навыки Docker, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 140989-192479, локация Новосибирск, формат Гибрид, занятость Полная. Вакансия: DevOps Engineer, опыт 4-5 лет, навыки AWS, GitLab CI, Kubernetes, Terraform, зарплата 323256-402006, локация Новосибирск, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): GitLab CI, Kubernetes, Terraform. Недостающие навыки: AWS.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (140989-192479 руб.) не пересекаются с вилкой вакансии (323256-402006 руб.).\", \"location_match\": \"Локация совпадает: Новосибирск.\", \"strengths\": [\"Совпавшие навыки: GitLab CI, Kubernetes, Terraform\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: AWS\"], \"recommendation\": \"consider\"}, \"confidence\": 0.86, \"match_score\": 77}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 3 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 261290-338066, локация Санкт-Петербург, формат Да, занятость Полная. Вакансия: DevOps Engineer, опыт 6-9 лет, навыки AWS, ArgoCD, Docker, Kubernetes, Terraform, зарплата 320839-377103, локация Москва, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): AWS, Docker, Kubernetes, Terraform. Недостающие навыки: ArgoCD.\", \"experience_match\": \"Опыт кандидата 3 лет ниже диапазона вакансии (6-9 лет).\", \"salary_match\": \"Ожидания кандидата (261290-338066 руб.) пересекаются с вилкой вакансии (320839-377103 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: AWS, Docker, Kubernetes, Terraform\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.86, \"match_score\": 78}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 4 лет, навыки Airflow, Docker, MLflow, NumPy, Pandas, TensorFlow, зарплата 231471-297398, локация Казань, формат Нет, занятость Частичная. Вакансия: DevOps Engineer, опыт 5-7 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 247009-336136, локация Москва, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/6): Docker. Недостающие навыки: AWS, ArgoCD, GitLab CI, Kubernetes, Terraform.\", \"experience_match\": \"Опыт кандидата 4 лет ниже диапазона вакансии (5-7 лет).\", \"salary_match\": \"Ожидания кандидата (231471-297398 руб.) пересекаются с вилкой вакансии (247009-336136 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: AWS, ArgoCD, GitLab CI, Kubernetes, Terraform\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.78, \"match_score\": 57}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 3 лет, навыки CSS, HTML, JavaScript, Next.js, React, TypeScript, Vue, зарплата 179290-260392, локация Казань, формат Нет, занятость Полная. Вакансия: Frontend Developer, опыт 3-5 лет, навыки Next.js, REST API, React, Redux, TypeScript, зарплата 170617-240596, локация Казань, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): Next.js, React, TypeScript. Недостающие навыки: REST API, Redux.\", \"experience_match\": \"Опыт кандидата 3 лет находится в диапазоне вакансии (3-5 лет).\", \"salary_match\": \"Ожидания кандидата (179290-260392 руб.) пересекаются с вилкой вакансии (170617-240596 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: Next.js, React, TypeScript\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: REST API, Redux\"], \"recommendation\": \"hire\"}, \"confidence\": 0.87, \"match_score\": 80}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 5 лет, навыки AsyncIO, Django, FastAPI, PostgreSQL, Redis, зарплата 169237-200164, локация Казань, формат Да, занятость Частичная. Вакансия: Python Backend Developer, опыт 4-5 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 260020-316611, локация Казань, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): FastAPI, PostgreSQL, Redis. Недостающие навыки: Docker, Kafka, Python.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (4-5 лет).\", \"salary_match\": \"Ожидания кандидата (169237-200164 руб.) не пересекаются с вилкой вакансии (260020-316611 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: FastAPI, PostgreSQL, Redis\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Docker, Kafka, Python\"], \"recommendation\": \"consider\"}, \"confidence\": 0.82, \"match_score\": 67}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 1 лет, навыки MLflow, NumPy, PyTorch, Python, зарплата 114225-188683, локация Москва, формат Нет, занятость Полная. Вакансия: ML Engineer, опыт 3-5 лет, навыки Albumentations, Docker, MLOps, OpenCV, Python, зарплата 232624-343460, локация Москва, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/5): Python. Недостающие навыки: Albumentations, Docker, MLOps, OpenCV.\", \"experience_match\": \"Опыт кандидата 1 лет ниже диапазона вакансии (3-5 лет).\", \"salary_match\": \"Ожидания кандидата (114225-188683 руб.) не пересекаются с вилкой вакансии (232624-343460 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: Python\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, Docker, MLOps, OpenCV\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.7, \"match_score\": 37}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 7 лет, навыки Docker, GitLab CI, Kubernetes, Linux, Terraform, зарплата 139397-187569, локация Екатеринбург, формат Да, занятость Частичная. Вакансия: DevOps Engineer, опыт 4-5 лет, навыки AWS, ArgoCD, Docker, Kubernetes, Terraform, зарплата 166354-256528, локация Екатеринбург, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): Docker, Kubernetes, Terraform. Недостающие навыки: AWS, ArgoCD.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (139397-187569 руб.) пересекаются с вилкой вакансии (166354-256528 руб.).\", \"location_match\": \"Локация совпадает: Екатеринбург.\", \"strengths\": [\"Совпавшие навыки: Docker, Kubernetes, Terraform\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: AWS, ArgoCD\"], \"recommendation\": \"consider\"}, \"confidence\": 0.86, \"match_score\": 78}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 6 лет, навыки A/B testing, Pandas, Python, Statistics, зарплата 109325-163113, локация Москва, формат Гибрид, занятость Полная. Вакансия: Data Analyst, опыт 4-5 лет, навыки A/B testing, Python, SQL, Tableau, зарплата 190612-261997, локация Москва, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): A/B testing, Python. Недостающие навыки: SQL, Tableau.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (109325-163113 руб.) не пересекаются с вилкой вакансии (190612-261997 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: A/B testing, Python\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: SQL, Tableau\"], \"recommendation\": \"consider\"}, \"confidence\": 0.81, \"match_score\": 66}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 6 лет, навыки AsyncIO, Docker, FastAPI, Python, зарплата 167967-211754, локация Казань, формат Да, занятость Полная. Вакансия: ML Engineer, опыт 2-3 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, Python, зарплата 162002-259522, локация Новосибирск, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/6): Docker, Python. Недостающие навыки: Albumentations, MLOps, OpenCV, PyTorch.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (3 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (167967-211754 руб.) пересекаются с вилкой вакансии (162002-259522 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker, Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV, PyTorch\"], \"recommendation\": \"consider\"}, \"confidence\": 0.81, \"match_score\": 66}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 4 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Terraform, зарплата 278790-333275, локация Екатеринбург, формат Да, занятость Полная. Вакансия: DevOps Engineer, опыт 5-8 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 259011-371000, локация Санкт-Петербург, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (5/6): AWS, Docker, GitLab CI, Kubernetes, Terraform. Недостающие навыки: ArgoCD.\", \"experience_match\": \"Опыт кандидата 4 лет ниже диапазона вакансии (5-8 лет).\", \"salary_match\": \"Ожидания кандидата (278790-333275 руб.) пересекаются с вилкой вакансии (259011-371000 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: AWS, Docker, GitLab CI, Kubernetes, Terraform\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD\", \"Недостаточный опыт\"], \"recommendation\": \"hire\"}, \"confidence\": 0.89, \"match_score\": 85}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 7 лет, навыки Airflow, Docker, MLflow, Pandas, Python, зарплата 140823-199362, локация Казань, формат Нет, занятость Полная. Вакансия: ML Engineer, опыт 1-3 лет, навыки Docker, MLOps, OpenCV, PyTorch, Python, зарплата 217128-292662, локация Казань, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): Docker, Python. Недостающие навыки: MLOps, OpenCV, PyTorch.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (3 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (140823-199362 руб.) не пересекаются с вилкой вакансии (217128-292662 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: Docker, Python\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: MLOps, OpenCV, PyTorch\"], \"recommendation\": \"consider\"}, \"confidence\": 0.8, \"match_score\": 63}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 2 лет, навыки HTML, JavaScript, Next.js, React, Redux, TypeScript, Vue, зарплата 104164-193276, локация Регионы, формат Гибрид, занятость Частичная. Вакансия: Python Backend Developer, опыт 1-3 лет, навыки Docker, Kafka, PostgreSQL, Python, зарплата 100314-196871, локация Москва, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: Docker, Kafka, PostgreSQL, Python.\", \"experience_match\": \"Опыт кандидата 2 лет находится в диапазоне вакансии (1-3 лет).\", \"salary_match\": \"Ожидания кандидата (104164-193276 руб.) пересекаются с вилкой вакансии (100314-196871 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Docker, Kafka, PostgreSQL, Python\"], \"recommendation\": \"consider\"}, \"confidence\": 0.76, \"match_score\": 53}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 6 лет, навыки Django, Docker, Python, Redis, зарплата 253301-319976, локация Санкт-Петербург, формат Нет, занятость Частичная. Вакансия: DevOps Engineer, опыт 2-3 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 254757-308175, локация Новосибирск, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/6): Docker. Недостающие навыки: AWS, ArgoCD, GitLab CI, Kubernetes, Terraform.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (3 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (253301-319976 руб.) пересекаются с вилкой вакансии (254757-308175 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: AWS, ArgoCD, GitLab CI, Kubernetes, Terraform\"], \"recommendation\": \"consider\"}, \"confidence\": 0.79, \"match_score\": 59}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 8 лет, навыки Airflow, MLflow, NumPy, Pandas, PyTorch, Python, зарплата 237017-300533, локация Регионы, формат Нет, занятость Частичная. Вакансия: ML Engineer, опыт 5-7 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, Python, зарплата 213109-272425, локация Регионы, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/6): PyTorch, Python. Недостающие навыки: Albumentations, Docker, MLOps, OpenCV.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (7 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (237017-300533 руб.) пересекаются с вилкой вакансии (213109-272425 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: PyTorch, Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, Docker, MLOps, OpenCV\"], \"recommendation\": \"consider\"}, \"confidence\": 0.81, \"match_score\": 66}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 4 лет, навыки A/B testing, Pandas, Python, SQL, Statistics, Tableau, зарплата 220804-251087, локация Регионы, формат Гибрид, занятость Частичная. Вакансия: Data Analyst, опыт 3-4 лет, навыки A/B testing, Product metrics, SQL, Tableau, зарплата 135869-203920, локация Регионы, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): A/B testing, SQL, Tableau. Недостающие навыки: Product metrics.\", \"experience_match\": \"Опыт кандидата 4 лет находится в диапазоне вакансии (3-4 лет).\", \"salary_match\": \"Ожидания кандидата (220804-251087 руб.) не пересекаются с вилкой вакансии (135869-203920 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: A/B testing, SQL, Tableau\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Product metrics\"], \"recommendation\": \"consider\"}, \"confidence\": 0.86, \"match_score\": 78}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 2 лет, навыки AsyncIO, Django, Docker, PostgreSQL, Python, Redis, зарплата 265687-322013, локация Казань, формат Гибрид, занятость Полная. Вакансия: Python Backend Developer, опыт 5-8 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Redis, зарплата 230744-277888, локация Казань, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): Docker, PostgreSQL, Redis. Недостающие навыки: FastAPI, Kafka.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (5-8 лет).\", \"salary_match\": \"Ожидания кандидата (265687-322013 руб.) пересекаются с вилкой вакансии (230744-277888 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: Docker, PostgreSQL, Redis\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: FastAPI, Kafka\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.81, \"match_score\": 65}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 8 лет, навыки HTML, Next.js, React, Vue, зарплата 179254-248901, локация Регионы, формат Нет, занятость Частичная. Вакансия: Frontend Developer, опыт 3-5 лет, навыки CSS, Next.js, REST API, React, TypeScript, зарплата 123406-188039, локация Регионы, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): Next.js, React. Недостающие навыки: CSS, REST API, TypeScript.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (179254-248901 руб.) пересекаются с вилкой вакансии (123406-188039 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: Next.js, React\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: CSS, REST API, TypeScript\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 71}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 1 лет, навыки Airflow, Docker, Pandas, PyTorch, Python, TensorFlow, зарплата 250033-319887, локация Регионы, формат Нет, занятость Полная. Вакансия: ML Engineer, опыт 1-2 лет, навыки Albumentations, Docker, MLOps, OpenCV, Python, зарплата 239454-281825, локация Регионы, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): Docker, Python. Недостающие навыки: Albumentations, MLOps, OpenCV.\", \"experience_match\": \"Опыт кандидата 1 лет находится в диапазоне вакансии (1-2 лет).\", \"salary_match\": \"Ожидания кандидата (250033-319887 руб.) пересекаются с вилкой вакансии (239454-281825 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: Docker, Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 71}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 4 лет, навыки AsyncIO, Django, Docker, FastAPI, Python, Redis, зарплата 131797-204138, локация Екатеринбург, формат Нет, занятость Полная. Вакансия: Python Backend Developer, опыт 3-5 лет, навыки Docker, FastAPI, Kafka, Redis, зарплата 229840-321089, локация Екатеринбург, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): Docker, FastAPI, Redis. Недостающие навыки: Kafka.\", \"experience_match\": \"Опыт кандидата 4 лет находится в диапазоне вакансии (3-5 лет).\", \"salary_match\": \"Ожидания кандидата (131797-204138 руб.) не пересекаются с вилкой вакансии (229840-321089 руб.).\", \"location_match\": \"Локация совпадает: Екатеринбург.\", \"strengths\": [\"Совпавшие навыки: Docker, FastAPI, Redis\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Kafka\"], \"recommendation\": \"consider\"}, \"confidence\": 0.86, \"match_score\": 78}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 8 лет, навыки Docker, MLflow, NumPy, PyTorch, зарплата 111870-154720, локация Москва, формат Да, занятость Полная. Вакансия: ML Engineer, опыт 5-6 лет, навыки Albumentations, OpenCV, PyTorch, Python, зарплата 307362-399277, локация Регионы, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/4): PyTorch. Недостающие навыки: Albumentations, OpenCV, Python.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (111870-154720 руб.) не пересекаются с вилкой вакансии (307362-399277 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: PyTorch\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, OpenCV, Python\"], \"recommendation\": \"consider\"}, \"confidence\": 0.77, \"match_score\": 56}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 3 лет, навыки AsyncIO, Django, Docker, FastAPI, PostgreSQL, зарплата 163875-217291, локация Москва, формат Да, занятость Полная. Вакансия: Python Backend Developer, опыт 3-4 лет, навыки Docker, FastAPI, Kafka, Python, Redis, зарплата 139998-223739, локация Москва, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): Docker, FastAPI. Недостающие навыки: Kafka, Python, Redis.\", \"experience_match\": \"Опыт кандидата 3 лет находится в диапазоне вакансии (3-4 лет).\", \"salary_match\": \"Ожидания кандидата (163875-217291 руб.) пересекаются с вилкой вакансии (139998-223739 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: Docker, FastAPI\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Kafka, Python, Redis\"], \"recommendation\": \"consider\"}, \"confidence\": 0.84, \"match_score\": 73}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 3 лет, навыки A/B testing, Power BI, Python, SQL, зарплата 157047-227248, локация Санкт-Петербург, формат Да, занятость Полная. Вакансия: Data Analyst, опыт 4-5 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 131551-213996, локация Санкт-Петербург, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): A/B testing, Python, SQL. Недостающие навыки: Product metrics, Tableau.\", \"experience_match\": \"Опыт кандидата 3 лет ниже диапазона вакансии (4-5 лет).\", \"salary_match\": \"Ожидания кандидата (157047-227248 руб.) пересекаются с вилкой вакансии (131551-213996 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: A/B testing, Python, SQL\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Product metrics, Tableau\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.84, \"match_score\": 73}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 8 лет, навыки Airflow, Docker, MLflow, NumPy, PyTorch, Python, зарплата 216262-258382, локация Москва, формат Нет, занятость Частичная. Вакансия: ML Engineer, опыт 2-5 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, Python, зарплата 237103-337959, локация Москва, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): Docker, PyTorch, Python. Недостающие навыки: Albumentations, MLOps, OpenCV.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (216262-258382 руб.) пересекаются с вилкой вакансии (237103-337959 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: Docker, PyTorch, Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 74}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 9 лет, навыки AsyncIO, Django, FastAPI, PostgreSQL, Redis, зарплата 111340-192763, локация Санкт-Петербург, формат Гибрид, занятость Частичная. Вакансия: Python Backend Developer, опыт 3-6 лет, навыки FastAPI, Kafka, PostgreSQL, Python, зарплата 243382-291885, локация Санкт-Петербург, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): FastAPI, PostgreSQL. Недостающие навыки: Kafka, Python.\", \"experience_match\": \"Опыт кандидата 9 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (111340-192763 руб.) не пересекаются с вилкой вакансии (243382-291885 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: FastAPI, PostgreSQL\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Kafka, Python\"], \"recommendation\": \"consider\"}, \"confidence\": 0.82, \"match_score\": 68}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 8 лет, навыки AWS, Docker, GitLab CI, Linux, Prometheus, зарплата 288156-328048, локация Регионы, формат Нет, занятость Частичная. Вакансия: DevOps Engineer, опыт 2-5 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Terraform, зарплата 300616-348193, локация Екатеринбург, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): AWS, Docker, GitLab CI. Недостающие навыки: Kubernetes, Terraform.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (288156-328048 руб.) пересекаются с вилкой вакансии (300616-348193 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: AWS, Docker, GitLab CI\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Kubernetes, Terraform\"], \"recommendation\": \"hire\"}, \"confidence\": 0.87, \"match_score\": 80}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 4 лет, навыки A/B testing, Pandas, Python, SQL, Statistics, зарплата 118401-157794, локация Москва, формат Да, занятость Частичная. Вакансия: Frontend Developer, опыт 1-4 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 153206-255526, локация Санкт-Петербург, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/6): нет. Недостающие навыки: CSS, Next.js, REST API, React, Redux, TypeScript.\", \"experience_match\": \"Опыт кандидата 4 лет находится в диапазоне вакансии (1-4 лет).\", \"salary_match\": \"Ожидания кандидата (118401-157794 руб.) пересекаются с вилкой вакансии (153206-255526 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: CSS, Next.js, REST API, React, Redux\"], \"recommendation\": \"consider\"}, \"confidence\": 0.77, \"match_score\": 55}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 8 лет, навыки AsyncIO, Django, Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 180455-226460, локация Москва, формат Да, занятость Полная. Вакансия: Python Backend Developer, опыт 6-9 лет, навыки FastAPI, Kafka, PostgreSQL, Python, зарплата 258441-378289, локация Москва, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): FastAPI, PostgreSQL, Python. Недостающие навыки: Kafka.\", \"experience_match\": \"Опыт кандидата 8 лет находится в диапазоне вакансии (6-9 лет).\", \"salary_match\": \"Ожидания кандидата (180455-226460 руб.) не пересекаются с вилкой вакансии (258441-378289 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: FastAPI, PostgreSQL, Python\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Kafka\"], \"recommendation\": \"hire\"}, \"confidence\": 0.87, \"match_score\": 80}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 3 лет, навыки A/B testing, Pandas, Power BI, Python, SQL, Statistics, Tableau, зарплата 159859-196056, локация Санкт-Петербург, формат Да, занятость Частичная. Вакансия: ML Engineer, опыт 1-3 лет, навыки Albumentations, Docker, OpenCV, PyTorch, Python, зарплата 226181-311064, локация Москва, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/5): Python. Недостающие навыки: Albumentations, Docker, OpenCV, PyTorch.\", \"experience_match\": \"Опыт кандидата 3 лет находится в диапазоне вакансии (1-3 лет).\", \"salary_match\": \"Ожидания кандидата (159859-196056 руб.) не пересекаются с вилкой вакансии (226181-311064 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Python\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, Docker, OpenCV, PyTorch\"], \"recommendation\": \"consider\"}, \"confidence\": 0.77, \"match_score\": 55}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 9 лет, навыки Docker, GitLab CI, Kubernetes, Linux, Terraform, зарплата 297609-377040, локация Регионы, формат Нет, занятость Частичная. Вакансия: DevOps Engineer, опыт 6-8 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 253551-365681, локация Регионы, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/6): Docker, GitLab CI, Kubernetes, Terraform. Недостающие навыки: AWS, ArgoCD.\", \"experience_match\": \"Опыт кандидата 9 лет выше верхней границы (8 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (297609-377040 руб.) пересекаются с вилкой вакансии (253551-365681 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: Docker, GitLab CI, Kubernetes, Terraform\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: AWS, ArgoCD\"], \"recommendation\": \"hire\"}, \"confidence\": 0.88, \"match_score\": 83}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 7 лет, навыки AsyncIO, Docker, PostgreSQL, Python, зарплата 211798-287371, локация Санкт-Петербург, формат Да, занятость Полная. Вакансия: Frontend Developer, опыт 3-6 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 245444-309335, локация Екатеринбург, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/6): нет. Недостающие навыки: CSS, Next.js, REST API, React, Redux, TypeScript.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (211798-287371 руб.) пересекаются с вилкой вакансии (245444-309335 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: CSS, Next.js, REST API, React, Redux\"], \"recommendation\": \"consider\"}, \"confidence\": 0.76, \"match_score\": 53}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 4 лет, навыки Docker, PyTorch, Python, TensorFlow, зарплата 145764-177673, локация Санкт-Петербург, формат Да, занятость Частичная. Вакансия: ML Engineer, опыт 2-4 лет, навыки Docker, MLOps, OpenCV, PyTorch, Python, зарплата 207737-284145, локация Санкт-Петербург, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): Docker, PyTorch, Python. Недостающие навыки: MLOps, OpenCV.\", \"experience_match\": \"Опыт кандидата 4 лет находится в диапазоне вакансии (2-4 лет).\", \"salary_match\": \"Ожидания кандидата (145764-177673 руб.) не пересекаются с вилкой вакансии (207737-284145 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: Docker, PyTorch, Python\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: MLOps, OpenCV\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 71}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 2 лет, навыки A/B testing, Pandas, Power BI, Python, SQL, Statistics, Tableau, зарплата 149946-230316, локация Казань, формат Да, занятость Частичная. Вакансия: DevOps Engineer, опыт 6-9 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Terraform, зарплата 199174-253795, локация Санкт-Петербург, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/5): нет. Недостающие навыки: AWS, ArgoCD, Docker, GitLab CI, Terraform.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (6-9 лет).\", \"salary_match\": \"Ожидания кандидата (149946-230316 руб.) пересекаются с вилкой вакансии (199174-253795 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: AWS, ArgoCD, Docker, GitLab CI, Terraform\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.69, \"match_score\": 36}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 5 лет, навыки CSS, HTML, Next.js, Redux, TypeScript, Vue, зарплата 203100-250609, локация Казань, формат Да, занятость Частичная. Вакансия: Frontend Developer, опыт 4-6 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 100672-177744, локация Регионы, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/6): CSS, Next.js, Redux, TypeScript. Недостающие навыки: REST API, React.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (4-6 лет).\", \"salary_match\": \"Ожидания кандидата (203100-250609 руб.) не пересекаются с вилкой вакансии (100672-177744 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: CSS, Next.js, Redux, TypeScript\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: REST API, React\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 76}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 8 лет, навыки AsyncIO, Django, Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 154088-189738, локация Новосибирск, формат Нет, занятость Полная. Вакансия: ML Engineer, опыт 3-4 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, Python, зарплата 233897-289099, локация Москва, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/6): Docker, Python. Недостающие навыки: Albumentations, MLOps, OpenCV, PyTorch.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (154088-189738 руб.) не пересекаются с вилкой вакансии (233897-289099 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker, Python\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV, PyTorch\"], \"recommendation\": \"consider\"}, \"confidence\": 0.79, \"match_score\": 60}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 3 лет, навыки Airflow, Docker, MLflow, Pandas, Python, TensorFlow, зарплата 268793-331048, локация Регионы, формат Да, занятость Полная. Вакансия: ML Engineer, опыт 2-3 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, Python, зарплата 166451-260974, локация Регионы, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/6): Docker, Python. Недостающие навыки: Albumentations, MLOps, OpenCV, PyTorch.\", \"experience_match\": \"Опыт кандидата 3 лет находится в диапазоне вакансии (2-3 лет).\", \"salary_match\": \"Ожидания кандидата (268793-331048 руб.) не пересекаются с вилкой вакансии (166451-260974 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: Docker, Python\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV, PyTorch\"], \"recommendation\": \"consider\"}, \"confidence\": 0.79, \"match_score\": 61}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 4 лет, навыки AWS, Docker, Kubernetes, Linux, Prometheus, зарплата 215276-270971, локация Новосибирск, формат Да, занятость Частичная. Вакансия: DevOps Engineer, опыт 6-8 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Terraform, зарплата 198064-281360, локация Новосибирск, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): AWS, Docker. Недостающие навыки: ArgoCD, GitLab CI, Terraform.\", \"experience_match\": \"Опыт кандидата 4 лет ниже диапазона вакансии (6-8 лет).\", \"salary_match\": \"Ожидания кандидата (215276-270971 руб.) пересекаются с вилкой вакансии (198064-281360 руб.).\", \"location_match\": \"Локация совпадает: Новосибирск.\", \"strengths\": [\"Совпавшие навыки: AWS, Docker\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, GitLab CI, Terraform\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.8, \"match_score\": 62}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 7 лет, навыки Airflow, MLflow, Pandas, Python, TensorFlow, зарплата 138279-220230, локация Санкт-Петербург, формат Да, занятость Полная. Вакансия: ML Engineer, опыт 2-3 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, зарплата 145544-202075, локация Санкт-Петербург, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/5): нет. Недостающие навыки: Albumentations, Docker, MLOps, OpenCV, PyTorch.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (3 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (138279-220230 руб.) пересекаются с вилкой вакансии (145544-202075 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, Docker, MLOps, OpenCV, PyTorch\"], \"recommendation\": \"consider\"}, \"confidence\": 0.76, \"match_score\": 53}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 3 лет, навыки Docker, FastAPI, PostgreSQL, Redis, зарплата 161631-201690, локация Санкт-Петербург, формат Гибрид, занятость Полная. Вакансия: ML Engineer, опыт 2-5 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, Python, зарплата 235515-322450, локация Москва, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/6): Docker. Недостающие навыки: Albumentations, MLOps, OpenCV, PyTorch, Python.\", \"experience_match\": \"Опыт кандидата 3 лет находится в диапазоне вакансии (2-5 лет).\", \"salary_match\": \"Ожидания кандидата (161631-201690 руб.) не пересекаются с вилкой вакансии (235515-322450 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV, PyTorch, Python\"], \"recommendation\": \"consider\"}, \"confidence\": 0.76, \"match_score\": 52}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 7 лет, навыки Docker, MLflow, NumPy, Pandas, PyTorch, Python, TensorFlow, зарплата 163862-218304, локация Регионы, формат Нет, занятость Полная. Вакансия: ML Engineer, опыт 4-7 лет, навыки Albumentations, Docker, MLOps, PyTorch, Python, зарплата 177883-228022, локация Регионы, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): Docker, PyTorch, Python. Недостающие навыки: Albumentations, MLOps.\", \"experience_match\": \"Опыт кандидата 7 лет находится в диапазоне вакансии (4-7 лет).\", \"salary_match\": \"Ожидания кандидата (163862-218304 руб.) пересекаются с вилкой вакансии (177883-228022 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: Docker, PyTorch, Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps\"], \"recommendation\": \"hire\"}, \"confidence\": 0.87, \"match_score\": 80}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 1 лет, навыки CSS, HTML, Next.js, Vue, зарплата 125210-210789, локация Санкт-Петербург, формат Нет, занятость Полная. Вакансия: Frontend Developer, опыт 3-4 лет, навыки CSS, Next.js, REST API, Redux, TypeScript, зарплата 159001-205890, локация Регионы, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): CSS, Next.js. Недостающие навыки: REST API, Redux, TypeScript.\", \"experience_match\": \"Опыт кандидата 1 лет ниже диапазона вакансии (3-4 лет).\", \"salary_match\": \"Ожидания кандидата (125210-210789 руб.) пересекаются с вилкой вакансии (159001-205890 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: CSS, Next.js\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: REST API, Redux, TypeScript\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.77, \"match_score\": 56}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 8 лет, навыки A/B testing, Pandas, Power BI, Python, SQL, Statistics, Tableau, зарплата 217651-267463, локация Казань, формат Да, занятость Полная. Вакансия: Data Analyst, опыт 3-4 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 155878-210588, локация Санкт-Петербург, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): A/B testing, Python, SQL, Tableau. Недостающие навыки: Product metrics.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (217651-267463 руб.) не пересекаются с вилкой вакансии (155878-210588 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: A/B testing, Python, SQL, Tableau\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Product metrics\"], \"recommendation\": \"consider\"}, \"confidence\": 0.87, \"match_score\": 79}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 3 лет, навыки HTML, Next.js, TypeScript, Vue, зарплата 181207-223968, локация Москва, формат Нет, занятость Полная. Вакансия: Python Backend Developer, опыт 1-3 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 170757-224965, локация Регионы, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/6): нет. Недостающие навыки: Docker, FastAPI, Kafka, PostgreSQL, Python, Redis.\", \"experience_match\": \"Опыт кандидата 3 лет находится в диапазоне вакансии (1-3 лет).\", \"salary_match\": \"Ожидания кандидата (181207-223968 руб.) пересекаются с вилкой вакансии (170757-224965 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Docker, FastAPI, Kafka, PostgreSQL, Python\"], \"recommendation\": \"consider\"}, \"confidence\": 0.76, \"match_score\": 53}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 4 лет, навыки Django, Docker, Python, Redis, зарплата 247686-328393, локация Москва, формат Нет, занятость Полная. Вакансия: Python Backend Developer, опыт 3-4 лет, навыки Docker, Kafka, PostgreSQL, Python, зарплата 167013-230768, локация Москва, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): Docker, Python. Недостающие навыки: Kafka, PostgreSQL.\", \"experience_match\": \"Опыт кандидата 4 лет находится в диапазоне вакансии (3-4 лет).\", \"salary_match\": \"Ожидания кандидата (247686-328393 руб.) не пересекаются с вилкой вакансии (167013-230768 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: Docker, Python\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Kafka, PostgreSQL\"], \"recommendation\": \"consider\"}, \"confidence\": 0.82, \"match_score\": 67}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 5 лет, навыки Airflow, NumPy, PyTorch, Python, TensorFlow, зарплата 169660-215646, локация Регионы, формат Гибрид, занятость Частичная. Вакансия: ML Engineer, опыт 1-2 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, Python, зарплата 293531-409719, локация Регионы, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/6): PyTorch, Python. Недостающие навыки: Albumentations, Docker, MLOps, OpenCV.\", \"experience_match\": \"Опыт кандидата 5 лет выше верхней границы (2 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (169660-215646 руб.) не пересекаются с вилкой вакансии (293531-409719 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: PyTorch, Python\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, Docker, MLOps, OpenCV\"], \"recommendation\": \"consider\"}, \"confidence\": 0.78, \"match_score\": 58}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 7 лет, навыки HTML, JavaScript, Next.js, React, Vue, зарплата 144005-224698, локация Москва, формат Да, занятость Полная. Вакансия: Frontend Developer, опыт 1-3 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 181447-264346, локация Москва, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/6): Next.js, React. Недостающие навыки: CSS, REST API, Redux, TypeScript.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (3 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (144005-224698 руб.) пересекаются с вилкой вакансии (181447-264346 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: Next.js, React\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: CSS, REST API, Redux, TypeScript\"], \"recommendation\": \"consider\"}, \"confidence\": 0.81, \"match_score\": 66}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 2 лет, навыки JavaScript, Next.js, React, Redux, TypeScript, Vue, зарплата 216976-286732, локация Регионы, формат Да, занятость Полная. Вакансия: Frontend Developer, опыт 6-8 лет, навыки CSS, Next.js, REST API, Redux, TypeScript, зарплата 198519-318030, локация Регионы, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): Next.js, Redux, TypeScript. Недостающие навыки: CSS, REST API.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (6-8 лет).\", \"salary_match\": \"Ожидания кандидата (216976-286732 руб.) пересекаются с вилкой вакансии (198519-318030 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: Next.js, Redux, TypeScript\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: CSS, REST API\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.81, \"match_score\": 65}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 4 лет, навыки A/B testing, Python, SQL, Statistics, зарплата 82528-159917, локация Москва, формат Гибрид, занятость Частичная. Вакансия: Data Analyst, опыт 1-2 лет, навыки A/B testing, Product metrics, Python, SQL, зарплата 113525-198238, локация Москва, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): A/B testing, Python, SQL. Недостающие навыки: Product metrics.\", \"experience_match\": \"Опыт кандидата 4 лет выше верхней границы (2 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (82528-159917 руб.) пересекаются с вилкой вакансии (113525-198238 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: A/B testing, Python, SQL\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Product metrics\"], \"recommendation\": \"hire\"}, \"confidence\": 0.89, \"match_score\": 85}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 8 лет, навыки CSS, HTML, JavaScript, Next.js, Redux, Vue, зарплата 118810-149733, локация Санкт-Петербург, формат Гибрид, занятость Частичная. Вакансия: Frontend Developer, опыт 6-8 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 188371-267040, локация Санкт-Петербург, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): CSS, Next.js, Redux. Недостающие навыки: REST API, React, TypeScript.\", \"experience_match\": \"Опыт кандидата 8 лет находится в диапазоне вакансии (6-8 лет).\", \"salary_match\": \"Ожидания кандидата (118810-149733 руб.) не пересекаются с вилкой вакансии (188371-267040 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: CSS, Next.js, Redux\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: REST API, React, TypeScript\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 69}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 8 лет, навыки AsyncIO, PostgreSQL, Python, Redis, зарплата 115248-179614, локация Санкт-Петербург, формат Нет, занятость Частичная. Вакансия: Python Backend Developer, опыт 3-4 лет, навыки Kafka, PostgreSQL, Python, Redis, зарплата 159032-230616, локация Санкт-Петербург, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): PostgreSQL, Python, Redis. Недостающие навыки: Kafka.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (115248-179614 руб.) пересекаются с вилкой вакансии (159032-230616 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: PostgreSQL, Python, Redis\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Kafka\"], \"recommendation\": \"hire\"}, \"confidence\": 0.9, \"match_score\": 87}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 3 лет, навыки AsyncIO, Django, Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 279865-364301, локация Новосибирск, формат Да, занятость Частичная. Вакансия: Data Analyst, опыт 2-3 лет, навыки A/B testing, Product metrics, Python, SQL, зарплата 285491-337812, локация Казань, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/4): Python. Недостающие навыки: A/B testing, Product metrics, SQL.\", \"experience_match\": \"Опыт кандидата 3 лет находится в диапазоне вакансии (2-3 лет).\", \"salary_match\": \"Ожидания кандидата (279865-364301 руб.) пересекаются с вилкой вакансии (285491-337812 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, SQL\"], \"recommendation\": \"consider\"}, \"confidence\": 0.81, \"match_score\": 64}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 2 лет, навыки AsyncIO, Docker, FastAPI, Python, Redis, зарплата 179959-235370, локация Казань, формат Гибрид, занятость Полная. Вакансия: Python Backend Developer, опыт 6-8 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, зарплата 218281-285602, локация Казань, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): Docker, FastAPI, Python. Недостающие навыки: Kafka, PostgreSQL.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (6-8 лет).\", \"salary_match\": \"Ожидания кандидата (179959-235370 руб.) пересекаются с вилкой вакансии (218281-285602 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: Docker, FastAPI, Python\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Kafka, PostgreSQL\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.8, \"match_score\": 63}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 3 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 129597-194414, локация Санкт-Петербург, формат Да, занятость Полная. Вакансия: Data Analyst, опыт 4-7 лет, навыки A/B testing, Product metrics, Python, SQL, зарплата 253865-326453, локация Москва, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: A/B testing, Product metrics, Python, SQL.\", \"experience_match\": \"Опыт кандидата 3 лет ниже диапазона вакансии (4-7 лет).\", \"salary_match\": \"Ожидания кандидата (129597-194414 руб.) не пересекаются с вилкой вакансии (253865-326453 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, Python, SQL\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.7, \"match_score\": 38}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 7 лет, навыки A/B testing, Pandas, Power BI, Python, SQL, зарплата 189839-254164, локация Санкт-Петербург, формат Гибрид, занятость Частичная. Вакансия: Data Analyst, опыт 1-3 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 242086-284953, локация Санкт-Петербург, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): A/B testing, Python, SQL. Недостающие навыки: Product metrics, Tableau.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (3 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (189839-254164 руб.) пересекаются с вилкой вакансии (242086-284953 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: A/B testing, Python, SQL\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Product metrics, Tableau\"], \"recommendation\": \"consider\"}, \"confidence\": 0.86, \"match_score\": 78}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 2 лет, навыки Airflow, Docker, MLflow, NumPy, PyTorch, TensorFlow, зарплата 155325-217324, локация Казань, формат Гибрид, занятость Полная. Вакансия: ML Engineer, опыт 3-5 лет, навыки Albumentations, MLOps, OpenCV, PyTorch, зарплата 291055-343554, локация Казань, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/4): PyTorch. Недостающие навыки: Albumentations, MLOps, OpenCV.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (3-5 лет).\", \"salary_match\": \"Ожидания кандидата (155325-217324 руб.) не пересекаются с вилкой вакансии (291055-343554 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: PyTorch\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.75, \"match_score\": 49}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 6 лет, навыки Power BI, Python, Statistics, Tableau, зарплата 121157-152361, локация Казань, формат Да, занятость Частичная. Вакансия: Data Analyst, опыт 4-6 лет, навыки A/B testing, Product metrics, Python, SQL, зарплата 200906-287031, локация Казань, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/4): Python. Недостающие навыки: A/B testing, Product metrics, SQL.\", \"experience_match\": \"Опыт кандидата 6 лет находится в диапазоне вакансии (4-6 лет).\", \"salary_match\": \"Ожидания кандидата (121157-152361 руб.) не пересекаются с вилкой вакансии (200906-287031 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: Python\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, SQL\"], \"recommendation\": \"consider\"}, \"confidence\": 0.77, \"match_score\": 56}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 5 лет, навыки Airflow, Docker, NumPy, Pandas, Python, зарплата 249775-322929, локация Казань, формат Да, занятость Частичная. Вакансия: ML Engineer, опыт 3-6 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, Python, зарплата 240345-357163, локация Москва, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/6): Docker, Python. Недостающие навыки: Albumentations, MLOps, OpenCV, PyTorch.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (3-6 лет).\", \"salary_match\": \"Ожидания кандидата (249775-322929 руб.) пересекаются с вилкой вакансии (240345-357163 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker, Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV, PyTorch\"], \"recommendation\": \"consider\"}, \"confidence\": 0.82, \"match_score\": 68}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 8 лет, навыки Docker, GitLab CI, Kubernetes, Prometheus, Terraform, зарплата 146231-199475, локация Москва, формат Гибрид, занятость Полная. Вакансия: DevOps Engineer, опыт 5-7 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Terraform, зарплата 244474-325103, локация Москва, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): Docker, GitLab CI, Terraform. Недостающие навыки: AWS, ArgoCD.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (7 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (146231-199475 руб.) не пересекаются с вилкой вакансии (244474-325103 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: Docker, GitLab CI, Terraform\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: AWS, ArgoCD\"], \"recommendation\": \"consider\"}, \"confidence\": 0.84, \"match_score\": 72}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 8 лет, навыки HTML, JavaScript, Next.js, React, Redux, TypeScript, зарплата 128218-186525, локация Санкт-Петербург, формат Нет, занятость Полная. Вакансия: ML Engineer, опыт 5-6 лет, навыки Docker, MLOps, OpenCV, PyTorch, Python, зарплата 175964-267648, локация Москва, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/5): нет. Недостающие навыки: Docker, MLOps, OpenCV, PyTorch, Python.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (128218-186525 руб.) пересекаются с вилкой вакансии (175964-267648 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Docker, MLOps, OpenCV, PyTorch, Python\"], \"recommendation\": \"consider\"}, \"confidence\": 0.75, \"match_score\": 51}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 4 лет, навыки AsyncIO, Django, Docker, FastAPI, зарплата 279381-323343, локация Новосибирск, формат Нет, занятость Полная. Вакансия: Python Backend Developer, опыт 5-7 лет, навыки Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 284647-344487, локация Новосибирск, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): Docker, FastAPI. Недостающие навыки: PostgreSQL, Python, Redis.\", \"experience_match\": \"Опыт кандидата 4 лет ниже диапазона вакансии (5-7 лет).\", \"salary_match\": \"Ожидания кандидата (279381-323343 руб.) пересекаются с вилкой вакансии (284647-344487 руб.).\", \"location_match\": \"Локация совпадает: Новосибирск.\", \"strengths\": [\"Совпавшие навыки: Docker, FastAPI\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: PostgreSQL, Python, Redis\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.82, \"match_score\": 68}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 6 лет, навыки AsyncIO, Django, Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 248709-306902, локация Екатеринбург, формат Нет, занятость Полная. Вакансия: Python Backend Developer, опыт 5-8 лет, навыки FastAPI, Kafka, PostgreSQL, Redis, зарплата 145999-200995, локация Санкт-Петербург, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): FastAPI, PostgreSQL, Redis. Недостающие навыки: Kafka.\", \"experience_match\": \"Опыт кандидата 6 лет находится в диапазоне вакансии (5-8 лет).\", \"salary_match\": \"Ожидания кандидата (248709-306902 руб.) не пересекаются с вилкой вакансии (145999-200995 руб.).\", \"location_match\": \"Локация не совпадает (Екатеринбург vs Санкт-Петербург).\", \"strengths\": [\"Совпавшие навыки: FastAPI, PostgreSQL, Redis\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Kafka\", \"Несовпадение по локации/формату\"], \"recommendation\": \"consider\"}, \"confidence\": 0.84, \"match_score\": 72}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 6 лет, навыки A/B testing, Pandas, Power BI, Python, SQL, Tableau, зарплата 117818-160863, локация Москва, формат Да, занятость Частичная. Вакансия: Data Analyst, опыт 5-7 лет, навыки A/B testing, Python, SQL, Tableau, зарплата 243463-336073, локация Москва, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/4): A/B testing, Python, SQL, Tableau. Недостающие навыки: нет.\", \"experience_match\": \"Опыт кандидата 6 лет находится в диапазоне вакансии (5-7 лет).\", \"salary_match\": \"Ожидания кандидата (117818-160863 руб.) не пересекаются с вилкой вакансии (243463-336073 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: A/B testing, Python, SQL, Tableau\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [], \"recommendation\": \"hire\"}, \"confidence\": 0.91, \"match_score\": 91}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 7 лет, навыки A/B testing, Pandas, Power BI, Python, SQL, Statistics, Tableau, зарплата 86985-156104, локация Санкт-Петербург, формат Да, занятость Частичная. Вакансия: Frontend Developer, опыт 5-7 лет, навыки CSS, Next.js, REST API, React, Redux, зарплата 234410-309213, локация Санкт-Петербург, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/5): нет. Недостающие навыки: CSS, Next.js, REST API, React, Redux.\", \"experience_match\": \"Опыт кандидата 7 лет находится в диапазоне вакансии (5-7 лет).\", \"salary_match\": \"Ожидания кандидата (86985-156104 руб.) не пересекаются с вилкой вакансии (234410-309213 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: CSS, Next.js, REST API, React, Redux\"], \"recommendation\": \"reject\"}, \"confidence\": 0.73, \"match_score\": 44}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 7 лет, навыки AsyncIO, Django, Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 247158-298683, локация Москва, формат Да, занятость Полная. Вакансия: Python Backend Developer, опыт 3-5 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 124161-181688, локация Москва, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (5/6): Docker, FastAPI, PostgreSQL, Python, Redis. Недостающие навыки: Kafka.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (247158-298683 руб.) не пересекаются с вилкой вакансии (124161-181688 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: Docker, FastAPI, PostgreSQL, Python, Redis\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Kafka\"], \"recommendation\": \"hire\"}, \"confidence\": 0.87, \"match_score\": 81}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 8 лет, навыки AsyncIO, Django, Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 95755-152870, локация Москва, формат Нет, занятость Частичная. Вакансия: Python Backend Developer, опыт 2-4 лет, навыки Docker, PostgreSQL, Python, Redis, зарплата 139287-243055, локация Москва, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/4): Docker, PostgreSQL, Python, Redis. Недостающие навыки: нет.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (95755-152870 руб.) пересекаются с вилкой вакансии (139287-243055 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: Docker, PostgreSQL, Python, Redis\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [], \"recommendation\": \"hire\"}, \"confidence\": 0.93, \"match_score\": 96}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 1 лет, навыки A/B testing, Power BI, Statistics, Tableau, зарплата 159710-237352, локация Регионы, формат Гибрид, занятость Частичная. Вакансия: Data Analyst, опыт 6-9 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 178273-226323, локация Регионы, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): A/B testing, Tableau. Недостающие навыки: Product metrics, Python, SQL.\", \"experience_match\": \"Опыт кандидата 1 лет ниже диапазона вакансии (6-9 лет).\", \"salary_match\": \"Ожидания кандидата (159710-237352 руб.) пересекаются с вилкой вакансии (178273-226323 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: A/B testing, Tableau\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Product metrics, Python, SQL\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.75, \"match_score\": 50}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 1 лет, навыки A/B testing, Pandas, Power BI, Python, SQL, Statistics, Tableau, зарплата 174631-239103, локация Казань, формат Нет, занятость Полная. Вакансия: Data Analyst, опыт 2-4 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 161088-201823, локация Казань, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): A/B testing, Python, SQL, Tableau. Недостающие навыки: Product metrics.\", \"experience_match\": \"Опыт кандидата 1 лет ниже диапазона вакансии (2-4 лет).\", \"salary_match\": \"Ожидания кандидата (174631-239103 руб.) пересекаются с вилкой вакансии (161088-201823 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: A/B testing, Python, SQL, Tableau\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Product metrics\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 76}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 9 лет, навыки AsyncIO, Django, Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 230556-312412, локация Казань, формат Нет, занятость Частичная. Вакансия: Python Backend Developer, опыт 2-5 лет, навыки Docker, FastAPI, Kafka, Python, зарплата 269213-331686, локация Екатеринбург, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): Docker, FastAPI, Python. Недостающие навыки: Kafka.\", \"experience_match\": \"Опыт кандидата 9 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (230556-312412 руб.) пересекаются с вилкой вакансии (269213-331686 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker, FastAPI, Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Kafka\"], \"recommendation\": \"hire\"}, \"confidence\": 0.89, \"match_score\": 85}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 3 лет, навыки HTML, JavaScript, React, Redux, TypeScript, зарплата 192924-251904, локация Москва, формат Нет, занятость Частичная. Вакансия: Data Analyst, опыт 4-5 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 179382-255384, локация Санкт-Петербург, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/5): нет. Недостающие навыки: A/B testing, Product metrics, Python, SQL, Tableau.\", \"experience_match\": \"Опыт кандидата 3 лет ниже диапазона вакансии (4-5 лет).\", \"salary_match\": \"Ожидания кандидата (192924-251904 руб.) пересекаются с вилкой вакансии (179382-255384 руб.).\", \"location_match\": \"Локация не совпадает (Москва vs Санкт-Петербург).\", \"strengths\": [\"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, Python, SQL, Tableau\", \"Недостаточный опыт\", \"Несовпадение по локации/формату\"], \"recommendation\": \"reject\"}, \"confidence\": 0.71, \"match_score\": 40}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 2 лет, навыки Docker, NumPy, PyTorch, Python, зарплата 166148-196357, локация Регионы, формат Гибрид, занятость Полная. Вакансия: Data Analyst, опыт 5-8 лет, навыки A/B testing, Product metrics, SQL, Tableau, зарплата 253854-302670, локация Москва, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: A/B testing, Product metrics, SQL, Tableau.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (5-8 лет).\", \"salary_match\": \"Ожидания кандидата (166148-196357 руб.) не пересекаются с вилкой вакансии (253854-302670 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, SQL, Tableau\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.67, \"match_score\": 29}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 5 лет, навыки AsyncIO, Django, PostgreSQL, Python, Redis, зарплата 103253-156870, локация Казань, формат Нет, занятость Полная. Вакансия: Python Backend Developer, опыт 2-4 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 172195-248105, локация Екатеринбург, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): PostgreSQL, Python, Redis. Недостающие навыки: Docker, FastAPI, Kafka.\", \"experience_match\": \"Опыт кандидата 5 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (103253-156870 руб.) не пересекаются с вилкой вакансии (172195-248105 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: PostgreSQL, Python, Redis\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Docker, FastAPI, Kafka\"], \"recommendation\": \"consider\"}, \"confidence\": 0.82, \"match_score\": 68}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 7 лет, навыки CSS, JavaScript, Redux, TypeScript, зарплата 184028-257441, локация Казань, формат Нет, занятость Частичная. Вакансия: Frontend Developer, опыт 1-2 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 134264-205298, локация Казань, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): CSS, Redux, TypeScript. Недостающие навыки: Next.js, REST API, React.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (2 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (184028-257441 руб.) пересекаются с вилкой вакансии (134264-205298 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: CSS, Redux, TypeScript\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Next.js, REST API, React\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 76}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 2 лет, навыки A/B testing, Pandas, Power BI, SQL, Statistics, Tableau, зарплата 105887-137972, локация Казань, формат Да, занятость Полная. Вакансия: Data Analyst, опыт 4-5 лет, навыки A/B testing, Python, SQL, Tableau, зарплата 211868-316081, локация Казань, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): A/B testing, SQL, Tableau. Недостающие навыки: Python.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (4-5 лет).\", \"salary_match\": \"Ожидания кандидата (105887-137972 руб.) не пересекаются с вилкой вакансии (211868-316081 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: A/B testing, SQL, Tableau\"], \"weaknesses\": [\"Недостающие навыки: Python\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.82, \"match_score\": 68}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 6 лет, навыки Airflow, MLflow, Pandas, TensorFlow, зарплата 178475-220177, локация Москва, формат Нет, занятость Частичная. Вакансия: ML Engineer, опыт 1-2 лет, навыки Albumentations, Docker, MLOps, OpenCV, зарплата 292198-366993, локация Москва, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: Albumentations, Docker, MLOps, OpenCV.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (2 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (178475-220177 руб.) не пересекаются с вилкой вакансии (292198-366993 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, Docker, MLOps, OpenCV\"], \"recommendation\": \"reject\"}, \"confidence\": 0.73, \"match_score\": 45}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 2 лет, навыки AsyncIO, Django, Docker, FastAPI, PostgreSQL, зарплата 269697-321436, локация Санкт-Петербург, формат Да, занятость Частичная. Вакансия: Python Backend Developer, опыт 3-6 лет, навыки Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 212692-261370, локация Санкт-Петербург, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): Docker, FastAPI, PostgreSQL. Недостающие навыки: Python, Redis.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (3-6 лет).\", \"salary_match\": \"Ожидания кандидата (269697-321436 руб.) не пересекаются с вилкой вакансии (212692-261370 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: Docker, FastAPI, PostgreSQL\"], \"weaknesses\": [\"Недостающие навыки: Python, Redis\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.8, \"match_score\": 63}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 2 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 168166-239463, локация Москва, формат Гибрид, занятость Частичная. Вакансия: DevOps Engineer, опыт 7-10 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Terraform, зарплата 266235-324072, локация Москва, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (5/5): AWS, Docker, GitLab CI, Kubernetes, Terraform. Недостающие навыки: нет.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (7-10 лет).\", \"salary_match\": \"Ожидания кандидата (168166-239463 руб.) не пересекаются с вилкой вакансии (266235-324072 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: AWS, Docker, GitLab CI, Kubernetes, Terraform\"], \"weaknesses\": [\"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 71}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 7 лет, навыки Docker, Kubernetes, Prometheus, Terraform, зарплата 196675-283521, локация Санкт-Петербург, формат Гибрид, занятость Частичная. Вакансия: Data Analyst, опыт 2-3 лет, навыки A/B testing, Product metrics, Python, SQL, зарплата 320239-410901, локация Санкт-Петербург, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: A/B testing, Product metrics, Python, SQL.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (3 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (196675-283521 руб.) не пересекаются с вилкой вакансии (320239-410901 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, Python, SQL\"], \"recommendation\": \"reject\"}, \"confidence\": 0.73, \"match_score\": 45}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 4 лет, навыки CSS, HTML, JavaScript, Next.js, React, Redux, Vue, зарплата 140650-208414, локация Регионы, формат Нет, занятость Полная. Вакансия: Frontend Developer, опыт 6-7 лет, навыки CSS, Next.js, REST API, React, Redux, зарплата 110546-173508, локация Регионы, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): CSS, Next.js, React, Redux. Недостающие навыки: REST API.\", \"experience_match\": \"Опыт кандидата 4 лет ниже диапазона вакансии (6-7 лет).\", \"salary_match\": \"Ожидания кандидата (140650-208414 руб.) пересекаются с вилкой вакансии (110546-173508 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: CSS, Next.js, React, Redux\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: REST API\", \"Недостаточный опыт\"], \"recommendation\": \"hire\"}, \"confidence\": 0.87, \"match_score\": 80}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 1 лет, навыки CSS, HTML, Next.js, React, TypeScript, Vue, зарплата 120574-196754, локация Москва, формат Нет, занятость Частичная. Вакансия: Frontend Developer, опыт 5-6 лет, навыки CSS, REST API, React, Redux, TypeScript, зарплата 216855-269783, локация Казань, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): CSS, React, TypeScript. Недостающие навыки: REST API, Redux.\", \"experience_match\": \"Опыт кандидата 1 лет ниже диапазона вакансии (5-6 лет).\", \"salary_match\": \"Ожидания кандидата (120574-196754 руб.) не пересекаются с вилкой вакансии (216855-269783 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: CSS, React, TypeScript\"], \"weaknesses\": [\"Недостающие навыки: REST API, Redux\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.75, \"match_score\": 51}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 9 лет, навыки A/B testing, Pandas, Power BI, Python, SQL, Statistics, Tableau, зарплата 102997-170609, локация Санкт-Петербург, формат Нет, занятость Частичная. Вакансия: Data Analyst, опыт 2-5 лет, навыки A/B testing, Product metrics, SQL, Tableau, зарплата 232800-352758, локация Регионы, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): A/B testing, SQL, Tableau. Недостающие навыки: Product metrics.\", \"experience_match\": \"Опыт кандидата 9 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (102997-170609 руб.) не пересекаются с вилкой вакансии (232800-352758 руб.).\", \"location_match\": \"Локация не совпадает (Санкт-Петербург vs Регионы).\", \"strengths\": [\"Совпавшие навыки: A/B testing, SQL, Tableau\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Product metrics\", \"Несовпадение по локации/формату\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 71}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 1 лет, навыки A/B testing, Pandas, Power BI, SQL, Statistics, Tableau, зарплата 185957-271507, локация Москва, формат Гибрид, занятость Полная. Вакансия: Data Analyst, опыт 5-8 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 143437-221996, локация Москва, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): A/B testing, SQL, Tableau. Недостающие навыки: Product metrics, Python.\", \"experience_match\": \"Опыт кандидата 1 лет ниже диапазона вакансии (5-8 лет).\", \"salary_match\": \"Ожидания кандидата (185957-271507 руб.) пересекаются с вилкой вакансии (143437-221996 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: A/B testing, SQL, Tableau\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Product metrics, Python\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.79, \"match_score\": 60}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 8 лет, навыки AWS, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 286376-321584, локация Москва, формат Да, занятость Частичная. Вакансия: DevOps Engineer, опыт 4-5 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 170104-241566, локация Москва, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/6): AWS, GitLab CI, Kubernetes, Terraform. Недостающие навыки: ArgoCD, Docker.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (286376-321584 руб.) не пересекаются с вилкой вакансии (170104-241566 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: AWS, GitLab CI, Kubernetes, Terraform\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, Docker\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 75}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 4 лет, навыки AsyncIO, Django, Docker, FastAPI, Python, Redis, зарплата 232006-302395, локация Москва, формат Нет, занятость Полная. Вакансия: DevOps Engineer, опыт 5-6 лет, навыки Docker, GitLab CI, Kubernetes, Terraform, зарплата 124571-185594, локация Новосибирск, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/4): Docker. Недостающие навыки: GitLab CI, Kubernetes, Terraform.\", \"experience_match\": \"Опыт кандидата 4 лет ниже диапазона вакансии (5-6 лет).\", \"salary_match\": \"Ожидания кандидата (232006-302395 руб.) не пересекаются с вилкой вакансии (124571-185594 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker\"], \"weaknesses\": [\"Недостающие навыки: GitLab CI, Kubernetes, Terraform\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.76, \"match_score\": 53}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 2 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 160032-212402, локация Новосибирск, формат Нет, занятость Полная. Вакансия: ML Engineer, опыт 6-7 лет, навыки Docker, MLOps, OpenCV, PyTorch, Python, зарплата 215746-278752, локация Москва, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/5): Docker. Недостающие навыки: MLOps, OpenCV, PyTorch, Python.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (6-7 лет).\", \"salary_match\": \"Ожидания кандидата (160032-212402 руб.) не пересекаются с вилкой вакансии (215746-278752 руб.).\", \"location_match\": \"Локация не совпадает (Новосибирск vs Москва).\", \"strengths\": [\"Совпавшие навыки: Docker\"], \"weaknesses\": [\"Недостающие навыки: MLOps, OpenCV, PyTorch, Python\", \"Недостаточный опыт\", \"Несовпадение по локации/формату\"], \"recommendation\": \"reject\"}, \"confidence\": 0.67, \"match_score\": 31}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 8 лет, навыки Docker, FastAPI, PostgreSQL, Redis, зарплата 209116-272501, локация Новосибирск, формат Да, занятость Частичная. Вакансия: Python Backend Developer, опыт 6-7 лет, навыки Docker, FastAPI, Kafka, Python, зарплата 231181-343539, локация Новосибирск, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): Docker, FastAPI. Недостающие навыки: Kafka, Python.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (7 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (209116-272501 руб.) пересекаются с вилкой вакансии (231181-343539 руб.).\", \"location_match\": \"Локация совпадает: Новосибирск.\", \"strengths\": [\"Совпавшие навыки: Docker, FastAPI\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Kafka, Python\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 74}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 8 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 244144-328807, локация Новосибирск, формат Нет, занятость Частичная. Вакансия: DevOps Engineer, опыт 7-9 лет, навыки ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 213855-286960, локация Новосибирск, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): Docker, GitLab CI, Kubernetes, Terraform. Недостающие навыки: ArgoCD.\", \"experience_match\": \"Опыт кандидата 8 лет находится в диапазоне вакансии (7-9 лет).\", \"salary_match\": \"Ожидания кандидата (244144-328807 руб.) пересекаются с вилкой вакансии (213855-286960 руб.).\", \"location_match\": \"Локация совпадает: Новосибирск.\", \"strengths\": [\"Совпавшие навыки: Docker, GitLab CI, Kubernetes, Terraform\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD\"], \"recommendation\": \"hire\"}, \"confidence\": 0.91, \"match_score\": 91}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 3 лет, навыки AsyncIO, Django, Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 227845-286363, локация Москва, формат Да, занятость Частичная. Вакансия: Python Backend Developer, опыт 2-5 лет, навыки FastAPI, Kafka, PostgreSQL, Python, зарплата 238936-343705, локация Москва, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): FastAPI, PostgreSQL, Python. Недостающие навыки: Kafka.\", \"experience_match\": \"Опыт кандидата 3 лет находится в диапазоне вакансии (2-5 лет).\", \"salary_match\": \"Ожидания кандидата (227845-286363 руб.) пересекаются с вилкой вакансии (238936-343705 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: FastAPI, PostgreSQL, Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Kafka\"], \"recommendation\": \"hire\"}, \"confidence\": 0.9, \"match_score\": 88}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 8 лет, навыки CSS, HTML, JavaScript, TypeScript, зарплата 204162-260010, локация Казань, формат Гибрид, занятость Частичная. Вакансия: Frontend Developer, опыт 6-7 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 150973-255259, локация Казань, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/6): CSS, TypeScript. Недостающие навыки: Next.js, REST API, React, Redux.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (7 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (204162-260010 руб.) пересекаются с вилкой вакансии (150973-255259 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: CSS, TypeScript\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Next.js, REST API, React, Redux\"], \"recommendation\": \"consider\"}, \"confidence\": 0.82, \"match_score\": 68}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 5 лет, навыки Pandas, Power BI, Python, SQL, Statistics, Tableau, зарплата 224348-292593, локация Регионы, формат Да, занятость Частичная. Вакансия: Data Analyst, опыт 4-6 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 182637-289630, локация Регионы, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): Python, SQL, Tableau. Недостающие навыки: A/B testing, Product metrics.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (4-6 лет).\", \"salary_match\": \"Ожидания кандидата (224348-292593 руб.) пересекаются с вилкой вакансии (182637-289630 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: Python, SQL, Tableau\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics\"], \"recommendation\": \"hire\"}, \"confidence\": 0.87, \"match_score\": 80}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 6 лет, навыки Airflow, Docker, NumPy, TensorFlow, зарплата 204689-278275, локация Москва, формат Да, занятость Частичная. Вакансия: ML Engineer, опыт 3-4 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, зарплата 244456-334058, локация Казань, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/5): Docker. Недостающие навыки: Albumentations, MLOps, OpenCV, PyTorch.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (204689-278275 руб.) пересекаются с вилкой вакансии (244456-334058 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV, PyTorch\"], \"recommendation\": \"consider\"}, \"confidence\": 0.8, \"match_score\": 62}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 9 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 284073-339109, локация Регионы, формат Нет, занятость Частичная. Вакансия: DevOps Engineer, опыт 3-5 лет, навыки AWS, ArgoCD, Docker, Kubernetes, зарплата 295559-356134, локация Регионы, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): AWS, Docker, Kubernetes. Недостающие навыки: ArgoCD.\", \"experience_match\": \"Опыт кандидата 9 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (284073-339109 руб.) пересекаются с вилкой вакансии (295559-356134 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: AWS, Docker, Kubernetes\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD\"], \"recommendation\": \"hire\"}, \"confidence\": 0.9, \"match_score\": 87}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 6 лет, навыки A/B testing, Pandas, Power BI, SQL, Statistics, зарплата 221034-264919, локация Казань, формат Гибрид, занятость Полная. Вакансия: Data Analyst, опыт 2-5 лет, навыки A/B testing, Product metrics, Python, SQL, зарплата 163425-275045, локация Казань, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): A/B testing, SQL. Недостающие навыки: Product metrics, Python.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (221034-264919 руб.) пересекаются с вилкой вакансии (163425-275045 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: A/B testing, SQL\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Product metrics, Python\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 76}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 5 лет, навыки CSS, HTML, Next.js, React, TypeScript, Vue, зарплата 186796-256338, локация Санкт-Петербург, формат Да, занятость Полная. Вакансия: Frontend Developer, опыт 4-5 лет, навыки Next.js, REST API, React, Redux, зарплата 234166-327340, локация Санкт-Петербург, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): Next.js, React. Недостающие навыки: REST API, Redux.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (4-5 лет).\", \"salary_match\": \"Ожидания кандидата (186796-256338 руб.) пересекаются с вилкой вакансии (234166-327340 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: Next.js, React\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: REST API, Redux\"], \"recommendation\": \"consider\"}, \"confidence\": 0.86, \"match_score\": 77}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 4 лет, навыки Airflow, Docker, MLflow, NumPy, Pandas, Python, TensorFlow, зарплата 235777-288636, локация Казань, формат Гибрид, занятость Частичная. Вакансия: ML Engineer, опыт 4-6 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, зарплата 286928-362770, локация Казань, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/5): Docker. Недостающие навыки: Albumentations, MLOps, OpenCV, PyTorch.\", \"experience_match\": \"Опыт кандидата 4 лет находится в диапазоне вакансии (4-6 лет).\", \"salary_match\": \"Ожидания кандидата (235777-288636 руб.) пересекаются с вилкой вакансии (286928-362770 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: Docker\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV, PyTorch\"], \"recommendation\": \"consider\"}, \"confidence\": 0.81, \"match_score\": 64}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 9 лет, навыки Pandas, Python, SQL, Statistics, зарплата 230133-267617, локация Казань, формат Да, занятость Частичная. Вакансия: Data Analyst, опыт 5-6 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 180225-291411, локация Казань, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): Python, SQL. Недостающие навыки: A/B testing, Product metrics, Tableau.\", \"experience_match\": \"Опыт кандидата 9 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (230133-267617 руб.) пересекаются с вилкой вакансии (180225-291411 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: Python, SQL\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, Tableau\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 69}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 5 лет, навыки Airflow, Docker, MLflow, Pandas, PyTorch, зарплата 153935-214344, локация Казань, формат Гибрид, занятость Частичная. Вакансия: ML Engineer, опыт 1-2 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, Python, зарплата 244528-346948, локация Казань, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/6): Docker, PyTorch. Недостающие навыки: Albumentations, MLOps, OpenCV, Python.\", \"experience_match\": \"Опыт кандидата 5 лет выше верхней границы (2 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (153935-214344 руб.) не пересекаются с вилкой вакансии (244528-346948 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: Docker, PyTorch\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV, Python\"], \"recommendation\": \"consider\"}, \"confidence\": 0.79, \"match_score\": 60}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 7 лет, навыки A/B testing, Pandas, Power BI, Python, SQL, Statistics, Tableau, зарплата 180484-210789, локация Санкт-Петербург, формат Гибрид, занятость Полная. Вакансия: ML Engineer, опыт 4-6 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, Python, зарплата 119889-229852, локация Казань, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/6): Python. Недостающие навыки: Albumentations, Docker, MLOps, OpenCV, PyTorch.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (180484-210789 руб.) пересекаются с вилкой вакансии (119889-229852 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, Docker, MLOps, OpenCV, PyTorch\"], \"recommendation\": \"consider\"}, \"confidence\": 0.79, \"match_score\": 61}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 6 лет, навыки Airflow, MLflow, NumPy, Python, TensorFlow, зарплата 282429-369010, локация Регионы, формат Гибрид, занятость Полная. Вакансия: ML Engineer, опыт 1-3 лет, навыки Albumentations, Docker, PyTorch, Python, зарплата 250748-365203, локация Регионы, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/4): Python. Недостающие навыки: Albumentations, Docker, PyTorch.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (3 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (282429-369010 руб.) пересекаются с вилкой вакансии (250748-365203 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, Docker, PyTorch\"], \"recommendation\": \"consider\"}, \"confidence\": 0.8, \"match_score\": 63}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 2 лет, навыки Airflow, MLflow, NumPy, Pandas, PyTorch, Python, TensorFlow, зарплата 229605-313391, локация Санкт-Петербург, формат Нет, занятость Полная. Вакансия: ML Engineer, опыт 5-7 лет, навыки Albumentations, Docker, PyTorch, Python, зарплата 263856-375236, локация Санкт-Петербург, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): PyTorch, Python. Недостающие навыки: Albumentations, Docker.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (5-7 лет).\", \"salary_match\": \"Ожидания кандидата (229605-313391 руб.) пересекаются с вилкой вакансии (263856-375236 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: PyTorch, Python\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, Docker\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.79, \"match_score\": 60}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 7 лет, навыки AsyncIO, Django, Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 250701-334665, локация Санкт-Петербург, формат Гибрид, занятость Частичная. Вакансия: Python Backend Developer, опыт 4-6 лет, навыки FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 165908-260309, локация Санкт-Петербург, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): FastAPI, PostgreSQL, Python, Redis. Недостающие навыки: Kafka.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (250701-334665 руб.) пересекаются с вилкой вакансии (165908-260309 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: FastAPI, PostgreSQL, Python, Redis\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Kafka\"], \"recommendation\": \"hire\"}, \"confidence\": 0.91, \"match_score\": 89}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 6 лет, навыки Airflow, Docker, MLflow, NumPy, PyTorch, Python, TensorFlow, зарплата 222746-257985, локация Москва, формат Нет, занятость Полная. Вакансия: ML Engineer, опыт 1-4 лет, навыки Docker, MLOps, OpenCV, PyTorch, Python, зарплата 256898-298470, локация Москва, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): Docker, PyTorch, Python. Недостающие навыки: MLOps, OpenCV.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (222746-257985 руб.) пересекаются с вилкой вакансии (256898-298470 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: Docker, PyTorch, Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: MLOps, OpenCV\"], \"recommendation\": \"consider\"}, \"confidence\": 0.86, \"match_score\": 78}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 3 лет, навыки JavaScript, Redux, TypeScript, Vue, зарплата 135201-175535, локация Казань, формат Да, занятость Полная. Вакансия: Data Analyst, опыт 3-4 лет, навыки A/B testing, Python, SQL, Tableau, зарплата 231050-283030, локация Москва, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: A/B testing, Python, SQL, Tableau.\", \"experience_match\": \"Опыт кандидата 3 лет находится в диапазоне вакансии (3-4 лет).\", \"salary_match\": \"Ожидания кандидата (135201-175535 руб.) не пересекаются с вилкой вакансии (231050-283030 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Python, SQL, Tableau\"], \"recommendation\": \"reject\"}, \"confidence\": 0.73, \"match_score\": 46}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 2 лет, навыки Docker, Kubernetes, Linux, Terraform, зарплата 243972-288075, локация Санкт-Петербург, формат Гибрид, занятость Частичная. Вакансия: DevOps Engineer, опыт 4-7 лет, навыки ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 264542-340558, локация Санкт-Петербург, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): Docker, Kubernetes, Terraform. Недостающие навыки: ArgoCD, GitLab CI.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (4-7 лет).\", \"salary_match\": \"Ожидания кандидата (243972-288075 руб.) пересекаются с вилкой вакансии (264542-340558 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: Docker, Kubernetes, Terraform\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, GitLab CI\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.82, \"match_score\": 67}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 5 лет, навыки CSS, HTML, JavaScript, Next.js, React, зарплата 105985-143143, локация Москва, формат Гибрид, занятость Полная. Вакансия: Frontend Developer, опыт 5-7 лет, навыки CSS, Next.js, REST API, React, Redux, зарплата 162726-234505, локация Москва, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): CSS, Next.js, React. Недостающие навыки: REST API, Redux.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (5-7 лет).\", \"salary_match\": \"Ожидания кандидата (105985-143143 руб.) не пересекаются с вилкой вакансии (162726-234505 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: CSS, Next.js, React\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: REST API, Redux\"], \"recommendation\": \"consider\"}, \"confidence\": 0.84, \"match_score\": 73}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 7 лет, навыки Airflow, Pandas, PyTorch, Python, зарплата 233628-320828, локация Санкт-Петербург, формат Гибрид, занятость Частичная. Вакансия: ML Engineer, опыт 5-8 лет, навыки Albumentations, Docker, OpenCV, PyTorch, Python, зарплата 156326-214424, локация Санкт-Петербург, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): PyTorch, Python. Недостающие навыки: Albumentations, Docker, OpenCV.\", \"experience_match\": \"Опыт кандидата 7 лет находится в диапазоне вакансии (5-8 лет).\", \"salary_match\": \"Ожидания кандидата (233628-320828 руб.) не пересекаются с вилкой вакансии (156326-214424 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: PyTorch, Python\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, Docker, OpenCV\"], \"recommendation\": \"consider\"}, \"confidence\": 0.81, \"match_score\": 64}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 8 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Prometheus, Terraform, зарплата 316369-362215, локация Санкт-Петербург, формат Нет, занятость Полная. Вакансия: Data Analyst, опыт 5-8 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 326131-443296, локация Новосибирск, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/5): нет. Недостающие навыки: A/B testing, Product metrics, Python, SQL, Tableau.\", \"experience_match\": \"Опыт кандидата 8 лет находится в диапазоне вакансии (5-8 лет).\", \"salary_match\": \"Ожидания кандидата (316369-362215 руб.) пересекаются с вилкой вакансии (326131-443296 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, Python, SQL, Tableau\"], \"recommendation\": \"consider\"}, \"confidence\": 0.77, \"match_score\": 55}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 3 лет, навыки CSS, JavaScript, Next.js, Redux, TypeScript, Vue, зарплата 134894-214797, локация Санкт-Петербург, формат Нет, занятость Частичная. Вакансия: Frontend Developer, опыт 6-8 лет, навыки CSS, React, Redux, TypeScript, зарплата 202944-307098, локация Санкт-Петербург, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): CSS, Redux, TypeScript. Недостающие навыки: React.\", \"experience_match\": \"Опыт кандидата 3 лет ниже диапазона вакансии (6-8 лет).\", \"salary_match\": \"Ожидания кандидата (134894-214797 руб.) пересекаются с вилкой вакансии (202944-307098 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: CSS, Redux, TypeScript\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: React\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 76}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 8 лет, навыки GitLab CI, Linux, Prometheus, Terraform, зарплата 312239-358538, локация Москва, формат Гибрид, занятость Частичная. Вакансия: Data Analyst, опыт 3-6 лет, навыки A/B testing, Product metrics, Python, Tableau, зарплата 153575-198303, локация Москва, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: A/B testing, Product metrics, Python, Tableau.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (312239-358538 руб.) не пересекаются с вилкой вакансии (153575-198303 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, Python, Tableau\"], \"recommendation\": \"reject\"}, \"confidence\": 0.72, \"match_score\": 43}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 3 лет, навыки A/B testing, Power BI, Python, Statistics, зарплата 235218-310878, локация Регионы, формат Гибрид, занятость Частичная. Вакансия: DevOps Engineer, опыт 5-6 лет, навыки ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 112322-219535, локация Казань, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/5): нет. Недостающие навыки: ArgoCD, Docker, GitLab CI, Kubernetes, Terraform.\", \"experience_match\": \"Опыт кандидата 3 лет ниже диапазона вакансии (5-6 лет).\", \"salary_match\": \"Ожидания кандидата (235218-310878 руб.) не пересекаются с вилкой вакансии (112322-219535 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [], \"weaknesses\": [\"Недостающие навыки: ArgoCD, Docker, GitLab CI, Kubernetes, Terraform\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.69, \"match_score\": 34}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 8 лет, навыки A/B testing, Pandas, Power BI, Python, SQL, Statistics, Tableau, зарплата 210479-272755, локация Санкт-Петербург, формат Да, занятость Полная. Вакансия: Data Analyst, опыт 5-8 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 157161-277037, локация Санкт-Петербург, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): A/B testing, Python, SQL, Tableau. Недостающие навыки: Product metrics.\", \"experience_match\": \"Опыт кандидата 8 лет находится в диапазоне вакансии (5-8 лет).\", \"salary_match\": \"Ожидания кандидата (210479-272755 руб.) пересекаются с вилкой вакансии (157161-277037 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: A/B testing, Python, SQL, Tableau\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Product metrics\"], \"recommendation\": \"hire\"}, \"confidence\": 0.91, \"match_score\": 89}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 5 лет, навыки AsyncIO, Docker, FastAPI, Python, зарплата 217248-261288, локация Екатеринбург, формат Да, занятость Частичная. Вакансия: Python Backend Developer, опыт 4-7 лет, навыки Kafka, PostgreSQL, Python, Redis, зарплата 261356-318698, локация Екатеринбург, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/4): Python. Недостающие навыки: Kafka, PostgreSQL, Redis.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (4-7 лет).\", \"salary_match\": \"Ожидания кандидата (217248-261288 руб.) не пересекаются с вилкой вакансии (261356-318698 руб.).\", \"location_match\": \"Локация совпадает: Екатеринбург.\", \"strengths\": [\"Совпавшие навыки: Python\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Kafka, PostgreSQL, Redis\"], \"recommendation\": \"consider\"}, \"confidence\": 0.77, \"match_score\": 56}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 9 лет, навыки Django, Docker, FastAPI, Redis, зарплата 152696-202977, локация Казань, формат Нет, занятость Частичная. Вакансия: ML Engineer, опыт 3-6 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, зарплата 242949-308379, локация Екатеринбург, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/5): Docker. Недостающие навыки: Albumentations, MLOps, OpenCV, PyTorch.\", \"experience_match\": \"Опыт кандидата 9 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (152696-202977 руб.) не пересекаются с вилкой вакансии (242949-308379 руб.).\", \"location_match\": \"Локация не совпадает (Казань vs Екатеринбург).\", \"strengths\": [\"Совпавшие навыки: Docker\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV, PyTorch\", \"Несовпадение по локации/формату\"], \"recommendation\": \"reject\"}, \"confidence\": 0.74, \"match_score\": 48}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 2 лет, навыки CSS, HTML, Next.js, Redux, TypeScript, Vue, зарплата 189673-219717, локация Санкт-Петербург, формат Да, занятость Полная. Вакансия: Frontend Developer, опыт 2-4 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 212145-278663, локация Казань, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/6): CSS, Next.js, Redux, TypeScript. Недостающие навыки: REST API, React.\", \"experience_match\": \"Опыт кандидата 2 лет находится в диапазоне вакансии (2-4 лет).\", \"salary_match\": \"Ожидания кандидата (189673-219717 руб.) пересекаются с вилкой вакансии (212145-278663 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: CSS, Next.js, Redux, TypeScript\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: REST API, React\"], \"recommendation\": \"hire\"}, \"confidence\": 0.88, \"match_score\": 83}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 1 лет, навыки CSS, JavaScript, Next.js, React, Redux, зарплата 210568-248748, локация Казань, формат Гибрид, занятость Полная. Вакансия: Frontend Developer, опыт 6-8 лет, навыки CSS, Next.js, React, Redux, TypeScript, зарплата 199230-249558, локация Казань, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): CSS, Next.js, React, Redux. Недостающие навыки: TypeScript.\", \"experience_match\": \"Опыт кандидата 1 лет ниже диапазона вакансии (6-8 лет).\", \"salary_match\": \"Ожидания кандидата (210568-248748 руб.) пересекаются с вилкой вакансии (199230-249558 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: CSS, Next.js, React, Redux\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: TypeScript\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 70}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 3 лет, навыки Airflow, Docker, Pandas, Python, TensorFlow, зарплата 244572-284228, локация Санкт-Петербург, формат Да, занятость Полная. Вакансия: ML Engineer, опыт 3-5 лет, навыки Docker, MLOps, OpenCV, PyTorch, Python, зарплата 254680-306701, локация Казань, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): Docker, Python. Недостающие навыки: MLOps, OpenCV, PyTorch.\", \"experience_match\": \"Опыт кандидата 3 лет находится в диапазоне вакансии (3-5 лет).\", \"salary_match\": \"Ожидания кандидата (244572-284228 руб.) пересекаются с вилкой вакансии (254680-306701 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker, Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: MLOps, OpenCV, PyTorch\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 71}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 6 лет, навыки Django, FastAPI, PostgreSQL, Redis, зарплата 138517-201497, локация Казань, формат Да, занятость Полная. Вакансия: Data Analyst, опыт 2-3 лет, навыки A/B testing, Product metrics, Python, Tableau, зарплата 283084-385573, локация Казань, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: A/B testing, Product metrics, Python, Tableau.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (3 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (138517-201497 руб.) не пересекаются с вилкой вакансии (283084-385573 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, Python, Tableau\"], \"recommendation\": \"reject\"}, \"confidence\": 0.73, \"match_score\": 45}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 5 лет, навыки A/B testing, Pandas, Power BI, Python, SQL, Statistics, Tableau, зарплата 156912-230110, локация Регионы, формат Нет, занятость Полная. Вакансия: Python Backend Developer, опыт 1-4 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 226633-284964, локация Казань, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/6): Python. Недостающие навыки: Docker, FastAPI, Kafka, PostgreSQL, Redis.\", \"experience_match\": \"Опыт кандидата 5 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (156912-230110 руб.) пересекаются с вилкой вакансии (226633-284964 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Docker, FastAPI, Kafka, PostgreSQL, Redis\"], \"recommendation\": \"consider\"}, \"confidence\": 0.79, \"match_score\": 61}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 10 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 153402-234773, локация Санкт-Петербург, формат Гибрид, занятость Полная. Вакансия: DevOps Engineer, опыт 6-8 лет, навыки ArgoCD, Docker, GitLab CI, Terraform, зарплата 175137-270615, локация Санкт-Петербург, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): Docker, GitLab CI, Terraform. Недостающие навыки: ArgoCD.\", \"experience_match\": \"Опыт кандидата 10 лет выше верхней границы (8 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (153402-234773 руб.) пересекаются с вилкой вакансии (175137-270615 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: Docker, GitLab CI, Terraform\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD\"], \"recommendation\": \"hire\"}, \"confidence\": 0.9, \"match_score\": 87}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 4 лет, навыки CSS, HTML, JavaScript, Next.js, React, Redux, TypeScript, зарплата 181795-232267, локация Санкт-Петербург, формат Да, занятость Полная. Вакансия: Frontend Developer, опыт 5-6 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 186320-285193, локация Санкт-Петербург, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (5/6): CSS, Next.js, React, Redux, TypeScript. Недостающие навыки: REST API.\", \"experience_match\": \"Опыт кандидата 4 лет ниже диапазона вакансии (5-6 лет).\", \"salary_match\": \"Ожидания кандидата (181795-232267 руб.) пересекаются с вилкой вакансии (186320-285193 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: CSS, Next.js, React, Redux, TypeScript\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: REST API\", \"Недостаточный опыт\"], \"recommendation\": \"hire\"}, \"confidence\": 0.9, \"match_score\": 87}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 4 лет, навыки Airflow, Docker, MLflow, TensorFlow, зарплата 135806-215637, локация Регионы, формат Да, занятость Частичная. Вакансия: Data Analyst, опыт 1-3 лет, навыки Product metrics, Python, SQL, Tableau, зарплата 306133-359306, локация Санкт-Петербург, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: Product metrics, Python, SQL, Tableau.\", \"experience_match\": \"Опыт кандидата 4 лет выше верхней границы (3 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (135806-215637 руб.) не пересекаются с вилкой вакансии (306133-359306 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Product metrics, Python, SQL, Tableau\"], \"recommendation\": \"reject\"}, \"confidence\": 0.72, \"match_score\": 43}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 5 лет, навыки A/B testing, Pandas, Power BI, SQL, Tableau, зарплата 178384-234379, локация Санкт-Петербург, формат Нет, занятость Полная. Вакансия: Data Analyst, опыт 5-8 лет, навыки A/B testing, Product metrics, Python, SQL, зарплата 161990-211951, локация Санкт-Петербург, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): A/B testing, SQL. Недостающие навыки: Product metrics, Python.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (5-8 лет).\", \"salary_match\": \"Ожидания кандидата (178384-234379 руб.) пересекаются с вилкой вакансии (161990-211951 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: A/B testing, SQL\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Product metrics, Python\"], \"recommendation\": \"consider\"}, \"confidence\": 0.86, \"match_score\": 77}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 7 лет, навыки Pandas, Power BI, Python, SQL, Statistics, Tableau, зарплата 96974-149411, локация Санкт-Петербург, формат Гибрид, занятость Полная. Вакансия: Frontend Developer, опыт 1-4 лет, навыки CSS, Next.js, REST API, React, TypeScript, зарплата 168936-249477, локация Казань, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/5): нет. Недостающие навыки: CSS, Next.js, REST API, React, TypeScript.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (96974-149411 руб.) не пересекаются с вилкой вакансии (168936-249477 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: CSS, Next.js, REST API, React, TypeScript\"], \"recommendation\": \"reject\"}, \"confidence\": 0.73, \"match_score\": 45}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 8 лет, навыки A/B testing, Python, SQL, Statistics, Tableau, зарплата 200258-231146, локация Казань, формат Да, занятость Полная. Вакансия: Data Analyst, опыт 3-5 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 202438-281908, локация Регионы, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): A/B testing, Python, SQL, Tableau. Недостающие навыки: Product metrics.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (200258-231146 руб.) пересекаются с вилкой вакансии (202438-281908 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: A/B testing, Python, SQL, Tableau\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Product metrics\"], \"recommendation\": \"hire\"}, \"confidence\": 0.91, \"match_score\": 89}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 5 лет, навыки AsyncIO, Django, Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 221646-279882, локация Москва, формат Нет, занятость Полная. Вакансия: Python Backend Developer, опыт 3-5 лет, навыки FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 253149-336892, локация Москва, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): FastAPI, PostgreSQL, Python, Redis. Недостающие навыки: Kafka.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (3-5 лет).\", \"salary_match\": \"Ожидания кандидата (221646-279882 руб.) пересекаются с вилкой вакансии (253149-336892 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: FastAPI, PostgreSQL, Python, Redis\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Kafka\"], \"recommendation\": \"hire\"}, \"confidence\": 0.91, \"match_score\": 89}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 8 лет, навыки Docker, GitLab CI, Kubernetes, Linux, зарплата 257671-346501, локация Регионы, формат Нет, занятость Частичная. Вакансия: DevOps Engineer, опыт 2-5 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 331878-420150, локация Регионы, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): Docker, GitLab CI, Kubernetes. Недостающие навыки: AWS, ArgoCD, Terraform.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (257671-346501 руб.) пересекаются с вилкой вакансии (331878-420150 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: Docker, GitLab CI, Kubernetes\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: AWS, ArgoCD, Terraform\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 76}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 7 лет, навыки AWS, GitLab CI, Kubernetes, Prometheus, Terraform, зарплата 169242-227343, локация Санкт-Петербург, формат Гибрид, занятость Частичная. Вакансия: DevOps Engineer, опыт 2-4 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 332961-427708, локация Санкт-Петербург, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/6): AWS, GitLab CI, Kubernetes, Terraform. Недостающие навыки: ArgoCD, Docker.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (169242-227343 руб.) не пересекаются с вилкой вакансии (332961-427708 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: AWS, GitLab CI, Kubernetes, Terraform\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, Docker\"], \"recommendation\": \"consider\"}, \"confidence\": 0.84, \"match_score\": 73}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 6 лет, навыки JavaScript, Next.js, React, Redux, TypeScript, Vue, зарплата 128936-160120, локация Москва, формат Гибрид, занятость Частичная. Вакансия: Frontend Developer, опыт 1-2 лет, навыки CSS, Next.js, REST API, Redux, зарплата 185094-254068, локация Москва, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): Next.js, Redux. Недостающие навыки: CSS, REST API.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (2 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (128936-160120 руб.) не пересекаются с вилкой вакансии (185094-254068 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: Next.js, Redux\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: CSS, REST API\"], \"recommendation\": \"consider\"}, \"confidence\": 0.82, \"match_score\": 68}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 3 лет, навыки A/B testing, Pandas, Power BI, Python, SQL, Statistics, Tableau, зарплата 218711-256448, локация Регионы, формат Да, занятость Частичная. Вакансия: Data Analyst, опыт 6-9 лет, навыки A/B testing, Product metrics, Python, Tableau, зарплата 165742-265436, локация Регионы, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): A/B testing, Python, Tableau. Недостающие навыки: Product metrics.\", \"experience_match\": \"Опыт кандидата 3 лет ниже диапазона вакансии (6-9 лет).\", \"salary_match\": \"Ожидания кандидата (218711-256448 руб.) пересекаются с вилкой вакансии (165742-265436 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: A/B testing, Python, Tableau\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Product metrics\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 74}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 7 лет, навыки CSS, JavaScript, React, TypeScript, Vue, зарплата 143723-218940, локация Санкт-Петербург, формат Да, занятость Частичная. Вакансия: Frontend Developer, опыт 2-4 лет, навыки Next.js, React, Redux, TypeScript, зарплата 165376-220234, локация Москва, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): React, TypeScript. Недостающие навыки: Next.js, Redux.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (143723-218940 руб.) пересекаются с вилкой вакансии (165376-220234 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: React, TypeScript\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Next.js, Redux\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 76}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 4 лет, навыки Docker, NumPy, PyTorch, TensorFlow, зарплата 194458-225547, локация Москва, формат Гибрид, занятость Полная. Вакансия: ML Engineer, опыт 1-4 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, Python, зарплата 306140-393352, локация Казань, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/6): Docker, PyTorch. Недостающие навыки: Albumentations, MLOps, OpenCV, Python.\", \"experience_match\": \"Опыт кандидата 4 лет находится в диапазоне вакансии (1-4 лет).\", \"salary_match\": \"Ожидания кандидата (194458-225547 руб.) не пересекаются с вилкой вакансии (306140-393352 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker, PyTorch\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV, Python\"], \"recommendation\": \"consider\"}, \"confidence\": 0.79, \"match_score\": 59}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 4 лет, навыки CSS, HTML, JavaScript, Next.js, React, TypeScript, зарплата 199504-231226, локация Казань, формат Гибрид, занятость Полная. Вакансия: Frontend Developer, опыт 2-3 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 194606-307222, локация Казань, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/6): CSS, Next.js, React, TypeScript. Недостающие навыки: REST API, Redux.\", \"experience_match\": \"Опыт кандидата 4 лет выше верхней границы (3 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (199504-231226 руб.) пересекаются с вилкой вакансии (194606-307222 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: CSS, Next.js, React, TypeScript\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: REST API, Redux\"], \"recommendation\": \"hire\"}, \"confidence\": 0.88, \"match_score\": 83}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 7 лет, навыки AWS, GitLab CI, Linux, Terraform, зарплата 131696-178951, локация Новосибирск, формат Нет, занятость Полная. Вакансия: DevOps Engineer, опыт 3-5 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 265275-366231, локация Новосибирск, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): AWS, GitLab CI, Terraform. Недостающие навыки: ArgoCD, Docker, Kubernetes.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (131696-178951 руб.) не пересекаются с вилкой вакансии (265275-366231 руб.).\", \"location_match\": \"Локация совпадает: Новосибирск.\", \"strengths\": [\"Совпавшие навыки: AWS, GitLab CI, Terraform\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, Docker, Kubernetes\"], \"recommendation\": \"consider\"}, \"confidence\": 0.82, \"match_score\": 68}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 4 лет, навыки Docker, MLflow, NumPy, Pandas, PyTorch, Python, TensorFlow, зарплата 279607-349759, локация Регионы, формат Нет, занятость Полная. Вакансия: ML Engineer, опыт 3-4 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, Python, зарплата 151612-265434, локация Казань, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): Docker, PyTorch, Python. Недостающие навыки: Albumentations, MLOps, OpenCV.\", \"experience_match\": \"Опыт кандидата 4 лет находится в диапазоне вакансии (3-4 лет).\", \"salary_match\": \"Ожидания кандидата (279607-349759 руб.) не пересекаются с вилкой вакансии (151612-265434 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker, PyTorch, Python\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 69}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 6 лет, навыки AWS, Docker, Kubernetes, Prometheus, Terraform, зарплата 246000-278282, локация Санкт-Петербург, формат Гибрид, занятость Полная. Вакансия: DevOps Engineer, опыт 5-8 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Kubernetes, зарплата 218424-313701, локация Санкт-Петербург, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): AWS, Docker, Kubernetes. Недостающие навыки: ArgoCD, GitLab CI.\", \"experience_match\": \"Опыт кандидата 6 лет находится в диапазоне вакансии (5-8 лет).\", \"salary_match\": \"Ожидания кандидата (246000-278282 руб.) пересекаются с вилкой вакансии (218424-313701 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: AWS, Docker, Kubernetes\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, GitLab CI\"], \"recommendation\": \"hire\"}, \"confidence\": 0.87, \"match_score\": 80}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 8 лет, навыки AsyncIO, Django, Docker, FastAPI, PostgreSQL, Python, зарплата 99902-130605, локация Санкт-Петербург, формат Гибрид, занятость Частичная. Вакансия: Python Backend Developer, опыт 3-5 лет, навыки Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 185694-240725, локация Санкт-Петербург, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): Docker, FastAPI, PostgreSQL, Python. Недостающие навыки: Redis.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (99902-130605 руб.) не пересекаются с вилкой вакансии (185694-240725 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: Docker, FastAPI, PostgreSQL, Python\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Redis\"], \"recommendation\": \"hire\"}, \"confidence\": 0.87, \"match_score\": 81}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 5 лет, навыки AsyncIO, Django, Docker, FastAPI, Python, зарплата 221294-274369, локация Санкт-Петербург, формат Нет, занятость Частичная. Вакансия: Python Backend Developer, опыт 6-8 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 213450-326679, локация Новосибирск, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): Docker, FastAPI, Python. Недостающие навыки: Kafka, PostgreSQL, Redis.\", \"experience_match\": \"Опыт кандидата 5 лет ниже диапазона вакансии (6-8 лет).\", \"salary_match\": \"Ожидания кандидата (221294-274369 руб.) пересекаются с вилкой вакансии (213450-326679 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker, FastAPI, Python\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Kafka, PostgreSQL, Redis\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 71}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 2 лет, навыки Django, Docker, FastAPI, Redis, зарплата 277700-350290, локация Санкт-Петербург, формат Да, занятость Частичная. Вакансия: Data Analyst, опыт 6-7 лет, навыки A/B testing, Product metrics, SQL, Tableau, зарплата 262853-357840, локация Новосибирск, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: A/B testing, Product metrics, SQL, Tableau.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (6-7 лет).\", \"salary_match\": \"Ожидания кандидата (277700-350290 руб.) пересекаются с вилкой вакансии (262853-357840 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, SQL, Tableau\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.7, \"match_score\": 38}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 10 лет, навыки Docker, Kubernetes, Linux, Prometheus, Terraform, зарплата 254707-297408, локация Москва, формат Да, занятость Частичная. Вакансия: DevOps Engineer, опыт 7-8 лет, навыки ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 314634-378652, локация Москва, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): Docker, Kubernetes, Terraform. Недостающие навыки: ArgoCD, GitLab CI.\", \"experience_match\": \"Опыт кандидата 10 лет выше верхней границы (8 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (254707-297408 руб.) не пересекаются с вилкой вакансии (314634-378652 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: Docker, Kubernetes, Terraform\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, GitLab CI\"], \"recommendation\": \"consider\"}, \"confidence\": 0.84, \"match_score\": 72}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 5 лет, навыки AsyncIO, Django, FastAPI, PostgreSQL, Python, Redis, зарплата 155531-198433, локация Казань, формат Гибрид, занятость Полная. Вакансия: Python Backend Developer, опыт 5-6 лет, навыки Kafka, PostgreSQL, Python, Redis, зарплата 163117-246269, локация Казань, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): PostgreSQL, Python, Redis. Недостающие навыки: Kafka.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (5-6 лет).\", \"salary_match\": \"Ожидания кандидата (155531-198433 руб.) пересекаются с вилкой вакансии (163117-246269 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: PostgreSQL, Python, Redis\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Kafka\"], \"recommendation\": \"hire\"}, \"confidence\": 0.89, \"match_score\": 86}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 5 лет, навыки Django, Docker, PostgreSQL, Redis, зарплата 110697-166570, локация Екатеринбург, формат Гибрид, занятость Полная. Вакансия: Python Backend Developer, опыт 5-7 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 129103-234914, локация Екатеринбург, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): Docker, PostgreSQL, Redis. Недостающие навыки: FastAPI, Kafka, Python.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (5-7 лет).\", \"salary_match\": \"Ожидания кандидата (110697-166570 руб.) пересекаются с вилкой вакансии (129103-234914 руб.).\", \"location_match\": \"Локация совпадает: Екатеринбург.\", \"strengths\": [\"Совпавшие навыки: Docker, PostgreSQL, Redis\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: FastAPI, Kafka, Python\"], \"recommendation\": \"consider\"}, \"confidence\": 0.86, \"match_score\": 77}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 1 лет, навыки Docker, MLflow, NumPy, Pandas, Python, зарплата 289220-346936, локация Санкт-Петербург, формат Да, занятость Частичная. Вакансия: ML Engineer, опыт 2-3 лет, навыки Albumentations, MLOps, OpenCV, Python, зарплата 210739-283778, локация Санкт-Петербург, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/4): Python. Недостающие навыки: Albumentations, MLOps, OpenCV.\", \"experience_match\": \"Опыт кандидата 1 лет ниже диапазона вакансии (2-3 лет).\", \"salary_match\": \"Ожидания кандидата (289220-346936 руб.) не пересекаются с вилкой вакансии (210739-283778 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: Python\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.72, \"match_score\": 43}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 4 лет, навыки A/B testing, Pandas, Power BI, Python, SQL, зарплата 206614-270327, локация Регионы, формат Да, занятость Полная. Вакансия: Frontend Developer, опыт 3-5 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 131347-185115, локация Санкт-Петербург, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/6): нет. Недостающие навыки: CSS, Next.js, REST API, React, Redux, TypeScript.\", \"experience_match\": \"Опыт кандидата 4 лет находится в диапазоне вакансии (3-5 лет).\", \"salary_match\": \"Ожидания кандидата (206614-270327 руб.) не пересекаются с вилкой вакансии (131347-185115 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: CSS, Next.js, REST API, React, Redux\"], \"recommendation\": \"reject\"}, \"confidence\": 0.73, \"match_score\": 46}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 2 лет, навыки AsyncIO, Django, FastAPI, PostgreSQL, Python, Redis, зарплата 164475-205887, локация Санкт-Петербург, формат Гибрид, занятость Полная. Вакансия: Python Backend Developer, опыт 3-4 лет, навыки Kafka, PostgreSQL, Python, Redis, зарплата 177395-251735, локация Санкт-Петербург, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): PostgreSQL, Python, Redis. Недостающие навыки: Kafka.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (3-4 лет).\", \"salary_match\": \"Ожидания кандидата (164475-205887 руб.) пересекаются с вилкой вакансии (177395-251735 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: PostgreSQL, Python, Redis\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Kafka\", \"Недостаточный опыт\"], \"recommendation\": \"hire\"}, \"confidence\": 0.87, \"match_score\": 80}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 7 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Prometheus, зарплата 162180-240151, локация Москва, формат Гибрид, занятость Полная. Вакансия: DevOps Engineer, опыт 3-5 лет, навыки AWS, ArgoCD, Docker, Terraform, зарплата 169661-281363, локация Москва, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): AWS, Docker. Недостающие навыки: ArgoCD, Terraform.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (162180-240151 руб.) пересекаются с вилкой вакансии (169661-281363 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: AWS, Docker\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, Terraform\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 74}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 1 лет, навыки A/B testing, Pandas, Python, SQL, Statistics, зарплата 117838-207710, локация Регионы, формат Да, занятость Полная. Вакансия: DevOps Engineer, опыт 4-5 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 216864-262958, локация Санкт-Петербург, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/6): нет. Недостающие навыки: AWS, ArgoCD, Docker, GitLab CI, Kubernetes, Terraform.\", \"experience_match\": \"Опыт кандидата 1 лет ниже диапазона вакансии (4-5 лет).\", \"salary_match\": \"Ожидания кандидата (117838-207710 руб.) не пересекаются с вилкой вакансии (216864-262958 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [], \"weaknesses\": [\"Недостающие навыки: AWS, ArgoCD, Docker, GitLab CI, Kubernetes\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.65, \"match_score\": 26}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 7 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 300418-376331, локация Екатеринбург, формат Да, занятость Частичная. Вакансия: DevOps Engineer, опыт 6-9 лет, навыки ArgoCD, GitLab CI, Kubernetes, Terraform, зарплата 312067-364550, локация Екатеринбург, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): GitLab CI, Kubernetes, Terraform. Недостающие навыки: ArgoCD.\", \"experience_match\": \"Опыт кандидата 7 лет находится в диапазоне вакансии (6-9 лет).\", \"salary_match\": \"Ожидания кандидата (300418-376331 руб.) пересекаются с вилкой вакансии (312067-364550 руб.).\", \"location_match\": \"Локация совпадает: Екатеринбург.\", \"strengths\": [\"Совпавшие навыки: GitLab CI, Kubernetes, Terraform\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD\"], \"recommendation\": \"hire\"}, \"confidence\": 0.9, \"match_score\": 88}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 2 лет, навыки HTML, Next.js, React, Redux, TypeScript, зарплата 104095-165306, локация Москва, формат Нет, занятость Частичная. Вакансия: DevOps Engineer, опыт 1-3 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Kubernetes, зарплата 156304-254783, локация Регионы, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/5): нет. Недостающие навыки: AWS, ArgoCD, Docker, GitLab CI, Kubernetes.\", \"experience_match\": \"Опыт кандидата 2 лет находится в диапазоне вакансии (1-3 лет).\", \"salary_match\": \"Ожидания кандидата (104095-165306 руб.) пересекаются с вилкой вакансии (156304-254783 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: AWS, ArgoCD, Docker, GitLab CI, Kubernetes\"], \"recommendation\": \"consider\"}, \"confidence\": 0.77, \"match_score\": 55}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 5 лет, навыки AsyncIO, Django, Docker, PostgreSQL, Python, зарплата 160234-223492, локация Москва, формат Нет, занятость Частичная. Вакансия: Python Backend Developer, опыт 2-4 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Redis, зарплата 176388-269997, локация Москва, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): Docker, PostgreSQL. Недостающие навыки: FastAPI, Kafka, Redis.\", \"experience_match\": \"Опыт кандидата 5 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (160234-223492 руб.) пересекаются с вилкой вакансии (176388-269997 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: Docker, PostgreSQL\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: FastAPI, Kafka, Redis\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 71}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 2 лет, навыки HTML, JavaScript, Next.js, React, Redux, TypeScript, Vue, зарплата 136129-198325, локация Казань, формат Гибрид, занятость Полная. Вакансия: Frontend Developer, опыт 4-5 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 195310-279125, локация Казань, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/6): Next.js, React, Redux, TypeScript. Недостающие навыки: CSS, REST API.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (4-5 лет).\", \"salary_match\": \"Ожидания кандидата (136129-198325 руб.) пересекаются с вилкой вакансии (195310-279125 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: Next.js, React, Redux, TypeScript\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: CSS, REST API\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 70}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 2 лет, навыки CSS, HTML, Next.js, React, Redux, TypeScript, Vue, зарплата 171939-239403, локация Регионы, формат Нет, занятость Частичная. Вакансия: Frontend Developer, опыт 4-5 лет, навыки CSS, REST API, Redux, TypeScript, зарплата 190487-277610, локация Регионы, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): CSS, Redux, TypeScript. Недостающие навыки: REST API.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (4-5 лет).\", \"salary_match\": \"Ожидания кандидата (171939-239403 руб.) пересекаются с вилкой вакансии (190487-277610 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: CSS, Redux, TypeScript\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: REST API\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 76}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 5 лет, навыки MLflow, NumPy, Pandas, PyTorch, Python, TensorFlow, зарплата 292211-330635, локация Казань, формат Гибрид, занятость Полная. Вакансия: ML Engineer, опыт 2-5 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, зарплата 178410-298029, локация Казань, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/5): PyTorch. Недостающие навыки: Albumentations, Docker, MLOps, OpenCV.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (2-5 лет).\", \"salary_match\": \"Ожидания кандидата (292211-330635 руб.) пересекаются с вилкой вакансии (178410-298029 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: PyTorch\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, Docker, MLOps, OpenCV\"], \"recommendation\": \"consider\"}, \"confidence\": 0.81, \"match_score\": 64}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 7 лет, навыки Airflow, MLflow, NumPy, PyTorch, Python, зарплата 143618-177166, локация Регионы, формат Нет, занятость Полная. Вакансия: DevOps Engineer, опыт 2-5 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 277818-373284, локация Регионы, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/6): нет. Недостающие навыки: AWS, ArgoCD, Docker, GitLab CI, Kubernetes, Terraform.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (143618-177166 руб.) не пересекаются с вилкой вакансии (277818-373284 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: AWS, ArgoCD, Docker, GitLab CI, Kubernetes\"], \"recommendation\": \"reject\"}, \"confidence\": 0.72, \"match_score\": 43}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 2 лет, навыки Airflow, Docker, PyTorch, TensorFlow, зарплата 159033-244647, локация Санкт-Петербург, формат Гибрид, занятость Частичная. Вакансия: ML Engineer, опыт 1-2 лет, навыки Albumentations, Docker, MLOps, PyTorch, зарплата 224254-294101, локация Санкт-Петербург, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): Docker, PyTorch. Недостающие навыки: Albumentations, MLOps.\", \"experience_match\": \"Опыт кандидата 2 лет находится в диапазоне вакансии (1-2 лет).\", \"salary_match\": \"Ожидания кандидата (159033-244647 руб.) пересекаются с вилкой вакансии (224254-294101 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: Docker, PyTorch\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps\"], \"recommendation\": \"consider\"}, \"confidence\": 0.86, \"match_score\": 77}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 5 лет, навыки AsyncIO, Django, Python, Redis, зарплата 221137-259043, локация Екатеринбург, формат Да, занятость Полная. Вакансия: Python Backend Developer, опыт 2-5 лет, навыки Docker, Kafka, PostgreSQL, Python, Redis, зарплата 256828-372904, локация Екатеринбург, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): Python, Redis. Недостающие навыки: Docker, Kafka, PostgreSQL.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (2-5 лет).\", \"salary_match\": \"Ожидания кандидата (221137-259043 руб.) пересекаются с вилкой вакансии (256828-372904 руб.).\", \"location_match\": \"Локация совпадает: Екатеринбург.\", \"strengths\": [\"Совпавшие навыки: Python, Redis\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Docker, Kafka, PostgreSQL\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 71}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 2 лет, навыки CSS, Next.js, React, Redux, Vue, зарплата 92506-130112, локация Казань, формат Да, занятость Частичная. Вакансия: Frontend Developer, опыт 3-5 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 137139-222795, локация Казань, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/6): CSS, Next.js, React, Redux. Недостающие навыки: REST API, TypeScript.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (3-5 лет).\", \"salary_match\": \"Ожидания кандидата (92506-130112 руб.) не пересекаются с вилкой вакансии (137139-222795 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: CSS, Next.js, React, Redux\"], \"weaknesses\": [\"Недостающие навыки: REST API, TypeScript\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.82, \"match_score\": 68}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 5 лет, навыки A/B testing, Pandas, Power BI, Python, SQL, Statistics, Tableau, зарплата 168478-201129, локация Москва, формат Да, занятость Полная. Вакансия: Data Analyst, опыт 4-6 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 181522-281564, локация Москва, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): A/B testing, Python, SQL, Tableau. Недостающие навыки: Product metrics.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (4-6 лет).\", \"salary_match\": \"Ожидания кандидата (168478-201129 руб.) пересекаются с вилкой вакансии (181522-281564 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: A/B testing, Python, SQL, Tableau\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Product metrics\"], \"recommendation\": \"hire\"}, \"confidence\": 0.91, \"match_score\": 91}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 1 лет, навыки CSS, HTML, React, Redux, Vue, зарплата 121224-164355, локация Казань, формат Нет, занятость Частичная. Вакансия: Frontend Developer, опыт 4-6 лет, навыки CSS, REST API, React, Redux, TypeScript, зарплата 152759-201724, локация Казань, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): CSS, React, Redux. Недостающие навыки: REST API, TypeScript.\", \"experience_match\": \"Опыт кандидата 1 лет ниже диапазона вакансии (4-6 лет).\", \"salary_match\": \"Ожидания кандидата (121224-164355 руб.) пересекаются с вилкой вакансии (152759-201724 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: CSS, React, Redux\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: REST API, TypeScript\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.8, \"match_score\": 63}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 7 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Prometheus, зарплата 299128-379437, локация Санкт-Петербург, формат Да, занятость Частичная. Вакансия: DevOps Engineer, опыт 6-7 лет, навыки ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 202299-275069, локация Санкт-Петербург, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): Docker, GitLab CI, Kubernetes. Недостающие навыки: ArgoCD, Terraform.\", \"experience_match\": \"Опыт кандидата 7 лет находится в диапазоне вакансии (6-7 лет).\", \"salary_match\": \"Ожидания кандидата (299128-379437 руб.) не пересекаются с вилкой вакансии (202299-275069 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: Docker, GitLab CI, Kubernetes\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, Terraform\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 71}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 8 лет, навыки Docker, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 289080-362344, локация Санкт-Петербург, формат Да, занятость Полная. Вакансия: Python Backend Developer, опыт 4-6 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, зарплата 166117-217983, локация Москва, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/5): Docker. Недостающие навыки: FastAPI, Kafka, PostgreSQL, Python.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (289080-362344 руб.) не пересекаются с вилкой вакансии (166117-217983 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: FastAPI, Kafka, PostgreSQL, Python\"], \"recommendation\": \"consider\"}, \"confidence\": 0.76, \"match_score\": 52}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 6 лет, навыки HTML, JavaScript, Next.js, React, Redux, TypeScript, Vue, зарплата 138573-204773, локация Казань, формат Нет, занятость Полная. Вакансия: Frontend Developer, опыт 2-4 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 150568-254639, локация Казань, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/6): Next.js, React, Redux, TypeScript. Недостающие навыки: CSS, REST API.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (138573-204773 руб.) пересекаются с вилкой вакансии (150568-254639 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: Next.js, React, Redux, TypeScript\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: CSS, REST API\"], \"recommendation\": \"hire\"}, \"confidence\": 0.87, \"match_score\": 81}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 5 лет, навыки CSS, JavaScript, Next.js, React, TypeScript, Vue, зарплата 179899-222970, локация Москва, формат Нет, занятость Частичная. Вакансия: Frontend Developer, опыт 4-5 лет, навыки CSS, Next.js, REST API, React, Redux, зарплата 237839-310495, локация Казань, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): CSS, Next.js, React. Недостающие навыки: REST API, Redux.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (4-5 лет).\", \"salary_match\": \"Ожидания кандидата (179899-222970 руб.) не пересекаются с вилкой вакансии (237839-310495 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: CSS, Next.js, React\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: REST API, Redux\"], \"recommendation\": \"consider\"}, \"confidence\": 0.84, \"match_score\": 73}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 7 лет, навыки AsyncIO, Django, Docker, FastAPI, Redis, зарплата 268601-343207, локация Казань, формат Нет, занятость Полная. Вакансия: Data Analyst, опыт 2-4 лет, навыки A/B testing, Product metrics, Python, SQL, зарплата 151979-243643, локация Москва, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: A/B testing, Product metrics, Python, SQL.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (268601-343207 руб.) не пересекаются с вилкой вакансии (151979-243643 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, Python, SQL\"], \"recommendation\": \"reject\"}, \"confidence\": 0.73, \"match_score\": 45}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 5 лет, навыки A/B testing, Power BI, Python, SQL, Statistics, Tableau, зарплата 209525-276178, локация Москва, формат Нет, занятость Частичная. Вакансия: ML Engineer, опыт 5-7 лет, навыки Albumentations, Docker, OpenCV, Python, зарплата 231491-300919, локация Москва, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/4): Python. Недостающие навыки: Albumentations, Docker, OpenCV.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (5-7 лет).\", \"salary_match\": \"Ожидания кандидата (209525-276178 руб.) пересекаются с вилкой вакансии (231491-300919 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, Docker, OpenCV\"], \"recommendation\": \"consider\"}, \"confidence\": 0.81, \"match_score\": 64}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 1 лет, навыки Next.js, React, Redux, TypeScript, Vue, зарплата 124958-183692, локация Регионы, формат Гибрид, занятость Полная. Вакансия: DevOps Engineer, опыт 6-8 лет, навыки ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 176370-225434, локация Москва, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/5): нет. Недостающие навыки: ArgoCD, Docker, GitLab CI, Kubernetes, Terraform.\", \"experience_match\": \"Опыт кандидата 1 лет ниже диапазона вакансии (6-8 лет).\", \"salary_match\": \"Ожидания кандидата (124958-183692 руб.) пересекаются с вилкой вакансии (176370-225434 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, Docker, GitLab CI, Kubernetes, Terraform\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.68, \"match_score\": 32}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 7 лет, навыки AsyncIO, Django, FastAPI, PostgreSQL, зарплата 122061-152692, локация Казань, формат Гибрид, занятость Полная. Вакансия: Python Backend Developer, опыт 3-5 лет, навыки Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 203516-244944, локация Казань, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): FastAPI, PostgreSQL. Недостающие навыки: Docker, Python, Redis.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (122061-152692 руб.) не пересекаются с вилкой вакансии (203516-244944 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: FastAPI, PostgreSQL\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Docker, Python, Redis\"], \"recommendation\": \"consider\"}, \"confidence\": 0.8, \"match_score\": 63}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 4 лет, навыки CSS, JavaScript, React, Redux, TypeScript, Vue, зарплата 170052-234775, локация Москва, формат Да, занятость Частичная. Вакансия: Frontend Developer, опыт 5-6 лет, навыки CSS, REST API, React, TypeScript, зарплата 178212-272358, локация Санкт-Петербург, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): CSS, React, TypeScript. Недостающие навыки: REST API.\", \"experience_match\": \"Опыт кандидата 4 лет ниже диапазона вакансии (5-6 лет).\", \"salary_match\": \"Ожидания кандидата (170052-234775 руб.) пересекаются с вилкой вакансии (178212-272358 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: CSS, React, TypeScript\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: REST API\", \"Недостаточный опыт\"], \"recommendation\": \"hire\"}, \"confidence\": 0.88, \"match_score\": 83}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 2 лет, навыки A/B testing, Pandas, SQL, Tableau, зарплата 212288-294987, локация Казань, формат Нет, занятость Частичная. Вакансия: Data Analyst, опыт 6-8 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 188619-285361, локация Казань, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): A/B testing, SQL, Tableau. Недостающие навыки: Product metrics, Python.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (6-8 лет).\", \"salary_match\": \"Ожидания кандидата (212288-294987 руб.) пересекаются с вилкой вакансии (188619-285361 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: A/B testing, SQL, Tableau\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Product metrics, Python\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.8, \"match_score\": 63}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 9 лет, навыки AsyncIO, Django, Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 150323-205529, локация Новосибирск, формат Гибрид, занятость Полная. Вакансия: Data Analyst, опыт 6-9 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 141919-258067, локация Санкт-Петербург, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/5): Python. Недостающие навыки: A/B testing, Product metrics, SQL, Tableau.\", \"experience_match\": \"Опыт кандидата 9 лет находится в диапазоне вакансии (6-9 лет).\", \"salary_match\": \"Ожидания кандидата (150323-205529 руб.) пересекаются с вилкой вакансии (141919-258067 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, SQL, Tableau\"], \"recommendation\": \"consider\"}, \"confidence\": 0.8, \"match_score\": 62}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 8 лет, навыки Airflow, Docker, Pandas, PyTorch, Python, TensorFlow, зарплата 227694-309225, локация Санкт-Петербург, формат Нет, занятость Полная. Вакансия: ML Engineer, опыт 5-6 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, Python, зарплата 249799-319914, локация Санкт-Петербург, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): Docker, PyTorch, Python. Недостающие навыки: Albumentations, MLOps, OpenCV.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (227694-309225 руб.) пересекаются с вилкой вакансии (249799-319914 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: Docker, PyTorch, Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 74}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 4 лет, навыки Pandas, Power BI, Python, SQL, Statistics, Tableau, зарплата 144974-218022, локация Москва, формат Гибрид, занятость Частичная. Вакансия: Data Analyst, опыт 2-3 лет, навыки A/B testing, Python, SQL, Tableau, зарплата 173216-248150, локация Санкт-Петербург, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): Python, SQL, Tableau. Недостающие навыки: A/B testing.\", \"experience_match\": \"Опыт кандидата 4 лет выше верхней границы (3 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (144974-218022 руб.) пересекаются с вилкой вакансии (173216-248150 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Python, SQL, Tableau\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: A/B testing\"], \"recommendation\": \"hire\"}, \"confidence\": 0.89, \"match_score\": 85}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 8 лет, навыки A/B testing, Pandas, Power BI, Python, Statistics, Tableau, зарплата 102283-140096, локация Регионы, формат Гибрид, занятость Полная. Вакансия: ML Engineer, опыт 5-8 лет, навыки Docker, OpenCV, PyTorch, Python, зарплата 147582-239330, локация Москва, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/4): Python. Недостающие навыки: Docker, OpenCV, PyTorch.\", \"experience_match\": \"Опыт кандидата 8 лет находится в диапазоне вакансии (5-8 лет).\", \"salary_match\": \"Ожидания кандидата (102283-140096 руб.) не пересекаются с вилкой вакансии (147582-239330 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Python\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Docker, OpenCV, PyTorch\"], \"recommendation\": \"consider\"}, \"confidence\": 0.77, \"match_score\": 56}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 7 лет, навыки Airflow, Docker, MLflow, NumPy, PyTorch, Python, TensorFlow, зарплата 299412-355888, локация Москва, формат Нет, занятость Полная. Вакансия: ML Engineer, опыт 2-5 лет, навыки Albumentations, Docker, OpenCV, Python, зарплата 204225-248680, локация Москва, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): Docker, Python. Недостающие навыки: Albumentations, OpenCV.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (299412-355888 руб.) не пересекаются с вилкой вакансии (204225-248680 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: Docker, Python\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, OpenCV\"], \"recommendation\": \"consider\"}, \"confidence\": 0.81, \"match_score\": 66}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 6 лет, навыки CSS, JavaScript, Redux, Vue, зарплата 182932-245178, локация Санкт-Петербург, формат Да, занятость Частичная. Вакансия: DevOps Engineer, опыт 1-2 лет, навыки AWS, Docker, Kubernetes, Terraform, зарплата 151785-202892, локация Казань, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: AWS, Docker, Kubernetes, Terraform.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (2 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (182932-245178 руб.) пересекаются с вилкой вакансии (151785-202892 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: AWS, Docker, Kubernetes, Terraform\"], \"recommendation\": \"consider\"}, \"confidence\": 0.75, \"match_score\": 51}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 1 лет, навыки CSS, HTML, Next.js, React, Redux, TypeScript, Vue, зарплата 177408-237878, локация Регионы, формат Нет, занятость Полная. Вакансия: Frontend Developer, опыт 6-8 лет, навыки CSS, REST API, React, Redux, TypeScript, зарплата 221016-303531, локация Регионы, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): CSS, React, Redux, TypeScript. Недостающие навыки: REST API.\", \"experience_match\": \"Опыт кандидата 1 лет ниже диапазона вакансии (6-8 лет).\", \"salary_match\": \"Ожидания кандидата (177408-237878 руб.) пересекаются с вилкой вакансии (221016-303531 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: CSS, React, Redux, TypeScript\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: REST API\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 70}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 7 лет, навыки AWS, Docker, GitLab CI, Kubernetes, зарплата 165290-245276, локация Москва, формат Да, занятость Полная. Вакансия: DevOps Engineer, опыт 6-9 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 273473-323040, локация Москва, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/6): AWS, Docker, GitLab CI, Kubernetes. Недостающие навыки: ArgoCD, Terraform.\", \"experience_match\": \"Опыт кандидата 7 лет находится в диапазоне вакансии (6-9 лет).\", \"salary_match\": \"Ожидания кандидата (165290-245276 руб.) не пересекаются с вилкой вакансии (273473-323040 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: AWS, Docker, GitLab CI, Kubernetes\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, Terraform\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 76}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 1 лет, навыки Docker, MLflow, Pandas, PyTorch, зарплата 131914-179455, локация Москва, формат Да, занятость Частичная. Вакансия: Python Backend Developer, опыт 2-3 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 177391-295986, локация Москва, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/6): Docker. Недостающие навыки: FastAPI, Kafka, PostgreSQL, Python, Redis.\", \"experience_match\": \"Опыт кандидата 1 лет ниже диапазона вакансии (2-3 лет).\", \"salary_match\": \"Ожидания кандидата (131914-179455 руб.) пересекаются с вилкой вакансии (177391-295986 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: Docker\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: FastAPI, Kafka, PostgreSQL, Python, Redis\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.74, \"match_score\": 48}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 8 лет, навыки Airflow, Docker, MLflow, NumPy, Pandas, PyTorch, Python, зарплата 251887-330574, локация Санкт-Петербург, формат Нет, занятость Полная. Вакансия: Python Backend Developer, опыт 3-4 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 299917-399471, локация Регионы, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/6): Docker, Python. Недостающие навыки: FastAPI, Kafka, PostgreSQL, Redis.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (251887-330574 руб.) пересекаются с вилкой вакансии (299917-399471 руб.).\", \"location_match\": \"Локация не совпадает (Санкт-Петербург vs Регионы).\", \"strengths\": [\"Совпавшие навыки: Docker, Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: FastAPI, Kafka, PostgreSQL, Redis\", \"Несовпадение по локации/формату\"], \"recommendation\": \"consider\"}, \"confidence\": 0.79, \"match_score\": 60}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 9 лет, навыки A/B testing, Pandas, Power BI, SQL, Statistics, зарплата 233866-265389, локация Санкт-Петербург, формат Гибрид, занятость Частичная. Вакансия: Data Analyst, опыт 3-5 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 118443-176679, локация Казань, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): A/B testing, SQL. Недостающие навыки: Product metrics, Python, Tableau.\", \"experience_match\": \"Опыт кандидата 9 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (233866-265389 руб.) не пересекаются с вилкой вакансии (118443-176679 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: A/B testing, SQL\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Product metrics, Python, Tableau\"], \"recommendation\": \"consider\"}, \"confidence\": 0.8, \"match_score\": 63}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 7 лет, навыки AsyncIO, Django, Docker, FastAPI, Python, Redis, зарплата 221430-261316, локация Москва, формат Гибрид, занятость Полная. Вакансия: Python Backend Developer, опыт 6-8 лет, навыки Docker, FastAPI, Kafka, Python, Redis, зарплата 136219-235269, локация Казань, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): Docker, FastAPI, Python, Redis. Недостающие навыки: Kafka.\", \"experience_match\": \"Опыт кандидата 7 лет находится в диапазоне вакансии (6-8 лет).\", \"salary_match\": \"Ожидания кандидата (221430-261316 руб.) пересекаются с вилкой вакансии (136219-235269 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker, FastAPI, Python, Redis\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Kafka\"], \"recommendation\": \"hire\"}, \"confidence\": 0.91, \"match_score\": 91}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 7 лет, навыки AsyncIO, Docker, Python, Redis, зарплата 166578-242988, локация Новосибирск, формат Да, занятость Частичная. Вакансия: Python Backend Developer, опыт 3-6 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 235741-342049, локация Москва, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): Docker, Python, Redis. Недостающие навыки: FastAPI, Kafka, PostgreSQL.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (166578-242988 руб.) пересекаются с вилкой вакансии (235741-342049 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker, Python, Redis\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: FastAPI, Kafka, PostgreSQL\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 76}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 4 лет, навыки A/B testing, Power BI, Python, SQL, зарплата 164549-195316, локация Казань, формат Гибрид, занятость Частичная. Вакансия: Data Analyst, опыт 2-4 лет, навыки A/B testing, Python, SQL, Tableau, зарплата 236032-339450, локация Казань, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): A/B testing, Python, SQL. Недостающие навыки: Tableau.\", \"experience_match\": \"Опыт кандидата 4 лет находится в диапазоне вакансии (2-4 лет).\", \"salary_match\": \"Ожидания кандидата (164549-195316 руб.) не пересекаются с вилкой вакансии (236032-339450 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: A/B testing, Python, SQL\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Tableau\"], \"recommendation\": \"consider\"}, \"confidence\": 0.86, \"match_score\": 78}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 6 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 123234-161380, локация Новосибирск, формат Нет, занятость Частичная. Вакансия: Data Analyst, опыт 7-10 лет, навыки A/B testing, Product metrics, Python, Tableau, зарплата 230450-344956, локация Екатеринбург, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: A/B testing, Product metrics, Python, Tableau.\", \"experience_match\": \"Опыт кандидата 6 лет ниже диапазона вакансии (7-10 лет).\", \"salary_match\": \"Ожидания кандидата (123234-161380 руб.) не пересекаются с вилкой вакансии (230450-344956 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, Python, Tableau\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.71, \"match_score\": 41}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 4 лет, навыки Airflow, MLflow, PyTorch, Python, TensorFlow, зарплата 254724-305801, локация Регионы, формат Нет, занятость Частичная. Вакансия: ML Engineer, опыт 4-5 лет, навыки MLOps, OpenCV, PyTorch, Python, зарплата 214202-304096, локация Регионы, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): PyTorch, Python. Недостающие навыки: MLOps, OpenCV.\", \"experience_match\": \"Опыт кандидата 4 лет находится в диапазоне вакансии (4-5 лет).\", \"salary_match\": \"Ожидания кандидата (254724-305801 руб.) пересекаются с вилкой вакансии (214202-304096 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: PyTorch, Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: MLOps, OpenCV\"], \"recommendation\": \"consider\"}, \"confidence\": 0.86, \"match_score\": 77}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 6 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Terraform, зарплата 284242-359627, локация Екатеринбург, формат Гибрид, занятость Полная. Вакансия: DevOps Engineer, опыт 6-7 лет, навыки AWS, ArgoCD, Docker, Kubernetes, Terraform, зарплата 253669-299147, локация Екатеринбург, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): AWS, Docker, Kubernetes, Terraform. Недостающие навыки: ArgoCD.\", \"experience_match\": \"Опыт кандидата 6 лет находится в диапазоне вакансии (6-7 лет).\", \"salary_match\": \"Ожидания кандидата (284242-359627 руб.) пересекаются с вилкой вакансии (253669-299147 руб.).\", \"location_match\": \"Локация совпадает: Екатеринбург.\", \"strengths\": [\"Совпавшие навыки: AWS, Docker, Kubernetes, Terraform\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD\"], \"recommendation\": \"hire\"}, \"confidence\": 0.91, \"match_score\": 91}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 8 лет, навыки A/B testing, Pandas, Power BI, Python, SQL, Tableau, зарплата 141551-229698, локация Регионы, формат Гибрид, занятость Частичная. Вакансия: Data Analyst, опыт 1-2 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 145563-236503, локация Регионы, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): A/B testing, Python, SQL, Tableau. Недостающие навыки: Product metrics.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (2 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (141551-229698 руб.) пересекаются с вилкой вакансии (145563-236503 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: A/B testing, Python, SQL, Tableau\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Product metrics\"], \"recommendation\": \"hire\"}, \"confidence\": 0.9, \"match_score\": 87}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 2 лет, навыки MLflow, Pandas, Python, TensorFlow, зарплата 211171-269890, локация Регионы, формат Нет, занятость Частичная. Вакансия: ML Engineer, опыт 1-2 лет, навыки Albumentations, Docker, PyTorch, Python, зарплата 250009-311811, локация Регионы, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/4): Python. Недостающие навыки: Albumentations, Docker, PyTorch.\", \"experience_match\": \"Опыт кандидата 2 лет находится в диапазоне вакансии (1-2 лет).\", \"salary_match\": \"Ожидания кандидата (211171-269890 руб.) пересекаются с вилкой вакансии (250009-311811 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, Docker, PyTorch\"], \"recommendation\": \"consider\"}, \"confidence\": 0.81, \"match_score\": 64}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 8 лет, навыки Airflow, Docker, MLflow, Pandas, PyTorch, Python, TensorFlow, зарплата 276443-346709, локация Казань, формат Нет, занятость Полная. Вакансия: ML Engineer, опыт 2-5 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, Python, зарплата 259604-299668, локация Казань, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): Docker, PyTorch, Python. Недостающие навыки: Albumentations, MLOps, OpenCV.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (276443-346709 руб.) пересекаются с вилкой вакансии (259604-299668 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: Docker, PyTorch, Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 74}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 6 лет, навыки Airflow, MLflow, PyTorch, Python, зарплата 141710-175109, локация Санкт-Петербург, формат Гибрид, занятость Полная. Вакансия: ML Engineer, опыт 3-5 лет, навыки Albumentations, MLOps, OpenCV, PyTorch, Python, зарплата 193099-264506, локация Санкт-Петербург, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): PyTorch, Python. Недостающие навыки: Albumentations, MLOps, OpenCV.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (141710-175109 руб.) не пересекаются с вилкой вакансии (193099-264506 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: PyTorch, Python\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV\"], \"recommendation\": \"consider\"}, \"confidence\": 0.79, \"match_score\": 61}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 1 лет, навыки CSS, HTML, JavaScript, Next.js, Redux, Vue, зарплата 172444-214993, локация Москва, формат Нет, занятость Полная. Вакансия: Frontend Developer, опыт 3-4 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 190462-239120, локация Москва, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): CSS, Next.js, Redux. Недостающие навыки: REST API, React, TypeScript.\", \"experience_match\": \"Опыт кандидата 1 лет ниже диапазона вакансии (3-4 лет).\", \"salary_match\": \"Ожидания кандидата (172444-214993 руб.) пересекаются с вилкой вакансии (190462-239120 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: CSS, Next.js, Redux\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: REST API, React, TypeScript\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.78, \"match_score\": 58}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 8 лет, навыки CSS, HTML, JavaScript, Next.js, TypeScript, Vue, зарплата 132027-168936, локация Регионы, формат Гибрид, занятость Полная. Вакансия: Frontend Developer, опыт 3-4 лет, навыки Next.js, REST API, React, Redux, зарплата 145612-235954, локация Москва, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/4): Next.js. Недостающие навыки: REST API, React, Redux.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (132027-168936 руб.) пересекаются с вилкой вакансии (145612-235954 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Next.js\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: REST API, React, Redux\"], \"recommendation\": \"consider\"}, \"confidence\": 0.81, \"match_score\": 65}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 6 лет, навыки AWS, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 120571-207044, локация Новосибирск, формат Нет, занятость Частичная. Вакансия: DevOps Engineer, опыт 7-8 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 188965-248178, локация Новосибирск, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/6): AWS, GitLab CI, Kubernetes, Terraform. Недостающие навыки: ArgoCD, Docker.\", \"experience_match\": \"Опыт кандидата 6 лет ниже диапазона вакансии (7-8 лет).\", \"salary_match\": \"Ожидания кандидата (120571-207044 руб.) пересекаются с вилкой вакансии (188965-248178 руб.).\", \"location_match\": \"Локация совпадает: Новосибирск.\", \"strengths\": [\"Совпавшие навыки: AWS, GitLab CI, Kubernetes, Terraform\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, Docker\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.87, \"match_score\": 79}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 4 лет, навыки AsyncIO, Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 121431-159707, локация Екатеринбург, формат Гибрид, занятость Полная. Вакансия: Python Backend Developer, опыт 6-8 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, зарплата 203711-263109, локация Новосибирск, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): Docker, FastAPI, PostgreSQL, Python. Недостающие навыки: Kafka.\", \"experience_match\": \"Опыт кандидата 4 лет ниже диапазона вакансии (6-8 лет).\", \"salary_match\": \"Ожидания кандидата (121431-159707 руб.) не пересекаются с вилкой вакансии (203711-263109 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker, FastAPI, PostgreSQL, Python\"], \"weaknesses\": [\"Недостающие навыки: Kafka\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 74}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 5 лет, навыки A/B testing, Pandas, Power BI, SQL, Statistics, зарплата 122990-200069, локация Москва, формат Да, занятость Частичная. Вакансия: Data Analyst, опыт 3-6 лет, навыки Product metrics, Python, SQL, Tableau, зарплата 142386-194338, локация Казань, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/4): SQL. Недостающие навыки: Product metrics, Python, Tableau.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (3-6 лет).\", \"salary_match\": \"Ожидания кандидата (122990-200069 руб.) пересекаются с вилкой вакансии (142386-194338 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: SQL\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Product metrics, Python, Tableau\"], \"recommendation\": \"consider\"}, \"confidence\": 0.81, \"match_score\": 64}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 4 лет, навыки CSS, Next.js, React, Vue, зарплата 180687-251668, локация Казань, формат Гибрид, занятость Частичная. Вакансия: Data Analyst, опыт 3-4 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 162799-270512, локация Регионы, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/5): нет. Недостающие навыки: A/B testing, Product metrics, Python, SQL, Tableau.\", \"experience_match\": \"Опыт кандидата 4 лет находится в диапазоне вакансии (3-4 лет).\", \"salary_match\": \"Ожидания кандидата (180687-251668 руб.) пересекаются с вилкой вакансии (162799-270512 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, Python, SQL, Tableau\"], \"recommendation\": \"consider\"}, \"confidence\": 0.76, \"match_score\": 53}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 7 лет, навыки AsyncIO, Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 278696-321420, локация Казань, формат Да, занятость Полная. Вакансия: Data Analyst, опыт 3-5 лет, навыки Product metrics, Python, SQL, Tableau, зарплата 285680-372681, локация Санкт-Петербург, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/4): Python. Недостающие навыки: Product metrics, SQL, Tableau.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (278696-321420 руб.) пересекаются с вилкой вакансии (285680-372681 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Product metrics, SQL, Tableau\"], \"recommendation\": \"consider\"}, \"confidence\": 0.8, \"match_score\": 63}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 8 лет, навыки AsyncIO, Django, Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 196251-254438, локация Санкт-Петербург, формат Да, занятость Полная. Вакансия: Python Backend Developer, опыт 6-7 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, зарплата 181633-259180, локация Санкт-Петербург, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): Docker, FastAPI, PostgreSQL, Python. Недостающие навыки: Kafka.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (7 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (196251-254438 руб.) пересекаются с вилкой вакансии (181633-259180 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: Docker, FastAPI, PostgreSQL, Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Kafka\"], \"recommendation\": \"hire\"}, \"confidence\": 0.9, \"match_score\": 87}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 2 лет, навыки CSS, HTML, JavaScript, Next.js, React, TypeScript, Vue, зарплата 106282-158231, локация Москва, формат Гибрид, занятость Полная. Вакансия: DevOps Engineer, опыт 5-6 лет, навыки ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 162556-273286, локация Регионы, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/5): нет. Недостающие навыки: ArgoCD, Docker, GitLab CI, Kubernetes, Terraform.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (5-6 лет).\", \"salary_match\": \"Ожидания кандидата (106282-158231 руб.) не пересекаются с вилкой вакансии (162556-273286 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [], \"weaknesses\": [\"Недостающие навыки: ArgoCD, Docker, GitLab CI, Kubernetes, Terraform\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.67, \"match_score\": 31}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 9 лет, навыки AsyncIO, Django, Docker, FastAPI, PostgreSQL, Python, зарплата 214111-293258, локация Москва, формат Да, занятость Полная. Вакансия: Python Backend Developer, опыт 5-6 лет, навыки FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 289750-401261, локация Москва, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): FastAPI, PostgreSQL, Python. Недостающие навыки: Kafka, Redis.\", \"experience_match\": \"Опыт кандидата 9 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (214111-293258 руб.) пересекаются с вилкой вакансии (289750-401261 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: FastAPI, PostgreSQL, Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Kafka, Redis\"], \"recommendation\": \"hire\"}, \"confidence\": 0.87, \"match_score\": 80}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 4 лет, навыки Airflow, Docker, MLflow, Pandas, PyTorch, Python, зарплата 289187-361656, локация Санкт-Петербург, формат Да, занятость Частичная. Вакансия: ML Engineer, опыт 3-6 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, зарплата 158717-278557, локация Санкт-Петербург, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): Docker, PyTorch. Недостающие навыки: Albumentations, MLOps, OpenCV.\", \"experience_match\": \"Опыт кандидата 4 лет находится в диапазоне вакансии (3-6 лет).\", \"salary_match\": \"Ожидания кандидата (289187-361656 руб.) не пересекаются с вилкой вакансии (158717-278557 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: Docker, PyTorch\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV\"], \"recommendation\": \"consider\"}, \"confidence\": 0.81, \"match_score\": 64}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 5 лет, навыки A/B testing, Power BI, Python, SQL, Statistics, Tableau, зарплата 106039-171545, локация Санкт-Петербург, формат Да, занятость Частичная. Вакансия: Data Analyst, опыт 1-2 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 154081-231482, локация Санкт-Петербург, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): A/B testing, Python, SQL, Tableau. Недостающие навыки: Product metrics.\", \"experience_match\": \"Опыт кандидата 5 лет выше верхней границы (2 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (106039-171545 руб.) пересекаются с вилкой вакансии (154081-231482 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: A/B testing, Python, SQL, Tableau\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Product metrics\"], \"recommendation\": \"hire\"}, \"confidence\": 0.91, \"match_score\": 89}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 6 лет, навыки AsyncIO, Django, Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 267956-316705, локация Санкт-Петербург, формат Гибрид, занятость Полная. Вакансия: Python Backend Developer, опыт 5-7 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 179112-245482, локация Санкт-Петербург, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (5/6): Docker, FastAPI, PostgreSQL, Python, Redis. Недостающие навыки: Kafka.\", \"experience_match\": \"Опыт кандидата 6 лет находится в диапазоне вакансии (5-7 лет).\", \"salary_match\": \"Ожидания кандидата (267956-316705 руб.) не пересекаются с вилкой вакансии (179112-245482 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: Docker, FastAPI, PostgreSQL, Python, Redis\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Kafka\"], \"recommendation\": \"hire\"}, \"confidence\": 0.89, \"match_score\": 84}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 3 лет, навыки CSS, HTML, JavaScript, React, Redux, TypeScript, Vue, зарплата 103094-166012, локация Регионы, формат Нет, занятость Полная. Вакансия: Frontend Developer, опыт 4-5 лет, навыки CSS, Next.js, Redux, TypeScript, зарплата 187030-240184, локация Регионы, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): CSS, Redux, TypeScript. Недостающие навыки: Next.js.\", \"experience_match\": \"Опыт кандидата 3 лет ниже диапазона вакансии (4-5 лет).\", \"salary_match\": \"Ожидания кандидата (103094-166012 руб.) не пересекаются с вилкой вакансии (187030-240184 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: CSS, Redux, TypeScript\"], \"weaknesses\": [\"Недостающие навыки: Next.js\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.84, \"match_score\": 72}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 5 лет, навыки CSS, JavaScript, React, Redux, зарплата 102177-156581, локация Санкт-Петербург, формат Гибрид, занятость Полная. Вакансия: Frontend Developer, опыт 6-7 лет, навыки CSS, REST API, React, Redux, TypeScript, зарплата 137214-181915, локация Санкт-Петербург, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): CSS, React, Redux. Недостающие навыки: REST API, TypeScript.\", \"experience_match\": \"Опыт кандидата 5 лет ниже диапазона вакансии (6-7 лет).\", \"salary_match\": \"Ожидания кандидата (102177-156581 руб.) пересекаются с вилкой вакансии (137214-181915 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: CSS, React, Redux\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: REST API, TypeScript\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.86, \"match_score\": 77}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 8 лет, навыки Django, Docker, FastAPI, PostgreSQL, зарплата 182225-258287, локация Казань, формат Да, занятость Полная. Вакансия: Python Backend Developer, опыт 6-8 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Redis, зарплата 238946-314455, локация Казань, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): Docker, FastAPI, PostgreSQL. Недостающие навыки: Kafka, Redis.\", \"experience_match\": \"Опыт кандидата 8 лет находится в диапазоне вакансии (6-8 лет).\", \"salary_match\": \"Ожидания кандидата (182225-258287 руб.) пересекаются с вилкой вакансии (238946-314455 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: Docker, FastAPI, PostgreSQL\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Kafka, Redis\"], \"recommendation\": \"hire\"}, \"confidence\": 0.87, \"match_score\": 80}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 9 лет, навыки A/B testing, Pandas, Power BI, SQL, Statistics, Tableau, зарплата 183911-268481, локация Казань, формат Гибрид, занятость Частичная. Вакансия: Data Analyst, опыт 2-3 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 221049-320594, локация Казань, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): A/B testing, SQL, Tableau. Недостающие навыки: Product metrics, Python.\", \"experience_match\": \"Опыт кандидата 9 лет выше верхней границы (3 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (183911-268481 руб.) пересекаются с вилкой вакансии (221049-320594 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: A/B testing, SQL, Tableau\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Product metrics, Python\"], \"recommendation\": \"consider\"}, \"confidence\": 0.86, \"match_score\": 78}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 8 лет, навыки AsyncIO, Django, Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 126681-203951, локация Казань, формат Нет, занятость Частичная. Вакансия: Python Backend Developer, опыт 4-7 лет, навыки FastAPI, Kafka, Python, Redis, зарплата 126672-231863, локация Казань, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): FastAPI, Python, Redis. Недостающие навыки: Kafka.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (7 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (126681-203951 руб.) пересекаются с вилкой вакансии (126672-231863 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: FastAPI, Python, Redis\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Kafka\"], \"recommendation\": \"hire\"}, \"confidence\": 0.89, \"match_score\": 85}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 7 лет, навыки CSS, HTML, JavaScript, Next.js, Redux, TypeScript, Vue, зарплата 78934-151211, локация Казань, формат Гибрид, занятость Частичная. Вакансия: Frontend Developer, опыт 1-3 лет, навыки Next.js, REST API, React, Redux, зарплата 216517-262104, локация Регионы, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): Next.js, Redux. Недостающие навыки: REST API, React.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (3 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (78934-151211 руб.) не пересекаются с вилкой вакансии (216517-262104 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Next.js, Redux\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: REST API, React\"], \"recommendation\": \"consider\"}, \"confidence\": 0.81, \"match_score\": 66}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 4 лет, навыки Power BI, Python, SQL, Tableau, зарплата 179118-259606, локация Казань, формат Да, занятость Полная. Вакансия: Data Analyst, опыт 5-6 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 178283-225065, локация Казань, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): Python, SQL, Tableau. Недостающие навыки: A/B testing, Product metrics.\", \"experience_match\": \"Опыт кандидата 4 лет ниже диапазона вакансии (5-6 лет).\", \"salary_match\": \"Ожидания кандидата (179118-259606 руб.) пересекаются с вилкой вакансии (178283-225065 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: Python, SQL, Tableau\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.86, \"match_score\": 77}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 9 лет, навыки AWS, GitLab CI, Prometheus, Terraform, зарплата 220837-279936, локация Санкт-Петербург, формат Гибрид, занятость Полная. Вакансия: DevOps Engineer, опыт 2-4 лет, навыки AWS, ArgoCD, Docker, Kubernetes, Terraform, зарплата 161643-231061, локация Санкт-Петербург, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): AWS, Terraform. Недостающие навыки: ArgoCD, Docker, Kubernetes.\", \"experience_match\": \"Опыт кандидата 9 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (220837-279936 руб.) пересекаются с вилкой вакансии (161643-231061 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: AWS, Terraform\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, Docker, Kubernetes\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 69}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 8 лет, навыки A/B testing, Pandas, Power BI, Python, Statistics, зарплата 216929-254520, локация Москва, формат Гибрид, занятость Частичная. Вакансия: Data Analyst, опыт 4-7 лет, навыки A/B testing, Python, SQL, Tableau, зарплата 111077-191837, локация Москва, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): A/B testing, Python. Недостающие навыки: SQL, Tableau.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (7 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (216929-254520 руб.) не пересекаются с вилкой вакансии (111077-191837 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: A/B testing, Python\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: SQL, Tableau\"], \"recommendation\": \"consider\"}, \"confidence\": 0.82, \"match_score\": 68}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 3 лет, навыки AWS, Kubernetes, Linux, Prometheus, Terraform, зарплата 289435-335749, локация Москва, формат Да, занятость Частичная. Вакансия: DevOps Engineer, опыт 7-8 лет, навыки AWS, ArgoCD, GitLab CI, Kubernetes, Terraform, зарплата 179119-247899, локация Москва, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): AWS, Kubernetes, Terraform. Недостающие навыки: ArgoCD, GitLab CI.\", \"experience_match\": \"Опыт кандидата 3 лет ниже диапазона вакансии (7-8 лет).\", \"salary_match\": \"Ожидания кандидата (289435-335749 руб.) не пересекаются с вилкой вакансии (179119-247899 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: AWS, Kubernetes, Terraform\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, GitLab CI\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.78, \"match_score\": 57}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 7 лет, навыки Airflow, MLflow, NumPy, PyTorch, Python, TensorFlow, зарплата 292327-323607, локация Регионы, формат Нет, занятость Частичная. Вакансия: Data Analyst, опыт 4-7 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 290913-380770, локация Санкт-Петербург, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/5): Python. Недостающие навыки: A/B testing, Product metrics, SQL, Tableau.\", \"experience_match\": \"Опыт кандидата 7 лет находится в диапазоне вакансии (4-7 лет).\", \"salary_match\": \"Ожидания кандидата (292327-323607 руб.) пересекаются с вилкой вакансии (290913-380770 руб.).\", \"location_match\": \"Локация не совпадает (Регионы vs Санкт-Петербург).\", \"strengths\": [\"Совпавшие навыки: Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, SQL, Tableau\", \"Несовпадение по локации/формату\"], \"recommendation\": \"consider\"}, \"confidence\": 0.77, \"match_score\": 56}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 3 лет, навыки CSS, HTML, JavaScript, Redux, TypeScript, Vue, зарплата 168519-208887, локация Санкт-Петербург, формат Нет, занятость Полная. Вакансия: Frontend Developer, опыт 1-3 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 204668-254233, локация Санкт-Петербург, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): CSS, Redux, TypeScript. Недостающие навыки: Next.js, REST API, React.\", \"experience_match\": \"Опыт кандидата 3 лет находится в диапазоне вакансии (1-3 лет).\", \"salary_match\": \"Ожидания кандидата (168519-208887 руб.) пересекаются с вилкой вакансии (204668-254233 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: CSS, Redux, TypeScript\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Next.js, REST API, React\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 75}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 3 лет, навыки A/B testing, Pandas, Power BI, Python, SQL, Statistics, Tableau, зарплата 94299-169515, локация Регионы, формат Да, занятость Полная. Вакансия: Data Analyst, опыт 4-6 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 245520-304163, локация Регионы, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): A/B testing, Python, SQL, Tableau. Недостающие навыки: Product metrics.\", \"experience_match\": \"Опыт кандидата 3 лет ниже диапазона вакансии (4-6 лет).\", \"salary_match\": \"Ожидания кандидата (94299-169515 руб.) не пересекаются с вилкой вакансии (245520-304163 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: A/B testing, Python, SQL, Tableau\"], \"weaknesses\": [\"Недостающие навыки: Product metrics\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 76}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 1 лет, навыки CSS, HTML, JavaScript, React, Redux, TypeScript, Vue, зарплата 108579-168066, локация Регионы, формат Да, занятость Частичная. Вакансия: Data Analyst, опыт 2-4 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 136659-216239, локация Москва, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/5): нет. Недостающие навыки: A/B testing, Product metrics, Python, SQL, Tableau.\", \"experience_match\": \"Опыт кандидата 1 лет ниже диапазона вакансии (2-4 лет).\", \"salary_match\": \"Ожидания кандидата (108579-168066 руб.) пересекаются с вилкой вакансии (136659-216239 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, Python, SQL, Tableau\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.72, \"match_score\": 42}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 3 лет, навыки AsyncIO, Docker, PostgreSQL, Redis, зарплата 136507-212858, локация Новосибирск, формат Нет, занятость Полная. Вакансия: Python Backend Developer, опыт 4-6 лет, навыки Docker, FastAPI, Kafka, Python, Redis, зарплата 232816-305847, локация Новосибирск, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): Docker, Redis. Недостающие навыки: FastAPI, Kafka, Python.\", \"experience_match\": \"Опыт кандидата 3 лет ниже диапазона вакансии (4-6 лет).\", \"salary_match\": \"Ожидания кандидата (136507-212858 руб.) не пересекаются с вилкой вакансии (232816-305847 руб.).\", \"location_match\": \"Локация совпадает: Новосибирск.\", \"strengths\": [\"Совпавшие навыки: Docker, Redis\"], \"weaknesses\": [\"Недостающие навыки: FastAPI, Kafka, Python\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.78, \"match_score\": 58}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 1 лет, навыки A/B testing, Pandas, Power BI, Python, SQL, Tableau, зарплата 234057-321410, локация Москва, формат Да, занятость Полная. Вакансия: Data Analyst, опыт 1-2 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 140859-200215, локация Санкт-Петербург, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): A/B testing, Python, SQL, Tableau. Недостающие навыки: Product metrics.\", \"experience_match\": \"Опыт кандидата 1 лет находится в диапазоне вакансии (1-2 лет).\", \"salary_match\": \"Ожидания кандидата (234057-321410 руб.) не пересекаются с вилкой вакансии (140859-200215 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: A/B testing, Python, SQL, Tableau\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Product metrics\"], \"recommendation\": \"hire\"}, \"confidence\": 0.87, \"match_score\": 80}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 7 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 163636-230946, локация Москва, формат Гибрид, занятость Частичная. Вакансия: DevOps Engineer, опыт 6-7 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 313969-415823, локация Москва, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (5/6): AWS, Docker, GitLab CI, Kubernetes, Terraform. Недостающие навыки: ArgoCD.\", \"experience_match\": \"Опыт кандидата 7 лет находится в диапазоне вакансии (6-7 лет).\", \"salary_match\": \"Ожидания кандидата (163636-230946 руб.) не пересекаются с вилкой вакансии (313969-415823 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: AWS, Docker, GitLab CI, Kubernetes, Terraform\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD\"], \"recommendation\": \"hire\"}, \"confidence\": 0.88, \"match_score\": 82}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 10 лет, навыки AWS, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 269589-343311, локация Регионы, формат Да, занятость Полная. Вакансия: Frontend Developer, опыт 5-8 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 336154-430873, локация Екатеринбург, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/6): нет. Недостающие навыки: CSS, Next.js, REST API, React, Redux, TypeScript.\", \"experience_match\": \"Опыт кандидата 10 лет выше верхней границы (8 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (269589-343311 руб.) пересекаются с вилкой вакансии (336154-430873 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: CSS, Next.js, REST API, React, Redux\"], \"recommendation\": \"consider\"}, \"confidence\": 0.76, \"match_score\": 53}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 1 лет, навыки Airflow, Docker, Pandas, PyTorch, Python, зарплата 295867-362623, локация Регионы, формат Гибрид, занятость Полная. Вакансия: ML Engineer, опыт 4-6 лет, навыки Docker, MLOps, OpenCV, PyTorch, Python, зарплата 267192-367191, локация Регионы, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): Docker, PyTorch, Python. Недостающие навыки: MLOps, OpenCV.\", \"experience_match\": \"Опыт кандидата 1 лет ниже диапазона вакансии (4-6 лет).\", \"salary_match\": \"Ожидания кандидата (295867-362623 руб.) пересекаются с вилкой вакансии (267192-367191 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: Docker, PyTorch, Python\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: MLOps, OpenCV\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.79, \"match_score\": 61}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 7 лет, навыки AsyncIO, Django, Docker, FastAPI, Python, зарплата 253135-310792, локация Казань, формат Гибрид, занятость Частичная. Вакансия: Python Backend Developer, опыт 4-5 лет, навыки FastAPI, Kafka, PostgreSQL, Redis, зарплата 139492-251561, локация Казань, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/4): FastAPI. Недостающие навыки: Kafka, PostgreSQL, Redis.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (253135-310792 руб.) не пересекаются с вилкой вакансии (139492-251561 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: FastAPI\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Kafka, PostgreSQL, Redis\"], \"recommendation\": \"consider\"}, \"confidence\": 0.77, \"match_score\": 56}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 4 лет, навыки AWS, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 287199-370445, локация Регионы, формат Нет, занятость Частичная. Вакансия: DevOps Engineer, опыт 6-9 лет, навыки ArgoCD, Docker, GitLab CI, Terraform, зарплата 176508-266566, локация Москва, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): GitLab CI, Terraform. Недостающие навыки: ArgoCD, Docker.\", \"experience_match\": \"Опыт кандидата 4 лет ниже диапазона вакансии (6-9 лет).\", \"salary_match\": \"Ожидания кандидата (287199-370445 руб.) не пересекаются с вилкой вакансии (176508-266566 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: GitLab CI, Terraform\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, Docker\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.79, \"match_score\": 60}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 3 лет, навыки Docker, GitLab CI, Kubernetes, Linux, Prometheus, зарплата 163043-238018, локация Москва, формат Да, занятость Полная. Вакансия: DevOps Engineer, опыт 4-6 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Kubernetes, зарплата 230516-346176, локация Москва, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): Docker, GitLab CI, Kubernetes. Недостающие навыки: AWS, ArgoCD.\", \"experience_match\": \"Опыт кандидата 3 лет ниже диапазона вакансии (4-6 лет).\", \"salary_match\": \"Ожидания кандидата (163043-238018 руб.) пересекаются с вилкой вакансии (230516-346176 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: Docker, GitLab CI, Kubernetes\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: AWS, ArgoCD\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 75}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 6 лет, навыки A/B testing, Pandas, Power BI, Python, SQL, Statistics, Tableau, зарплата 102942-135591, локация Казань, формат Да, занятость Полная. Вакансия: Python Backend Developer, опыт 6-9 лет, навыки Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 153311-233912, локация Регионы, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/5): Python. Недостающие навыки: Docker, FastAPI, PostgreSQL, Redis.\", \"experience_match\": \"Опыт кандидата 6 лет находится в диапазоне вакансии (6-9 лет).\", \"salary_match\": \"Ожидания кандидата (102942-135591 руб.) не пересекаются с вилкой вакансии (153311-233912 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Python\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Docker, FastAPI, PostgreSQL, Redis\"], \"recommendation\": \"consider\"}, \"confidence\": 0.77, \"match_score\": 55}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 3 лет, навыки AWS, Docker, Kubernetes, Linux, Terraform, зарплата 287440-324513, локация Екатеринбург, формат Да, занятость Частичная. Вакансия: DevOps Engineer, опыт 3-5 лет, навыки AWS, Docker, GitLab CI, Terraform, зарплата 221807-333230, локация Регионы, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): AWS, Docker, Terraform. Недостающие навыки: GitLab CI.\", \"experience_match\": \"Опыт кандидата 3 лет находится в диапазоне вакансии (3-5 лет).\", \"salary_match\": \"Ожидания кандидата (287440-324513 руб.) пересекаются с вилкой вакансии (221807-333230 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: AWS, Docker, Terraform\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: GitLab CI\"], \"recommendation\": \"hire\"}, \"confidence\": 0.9, \"match_score\": 88}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 2 лет, навыки AsyncIO, Django, Docker, FastAPI, PostgreSQL, Python, зарплата 158233-199365, локация Казань, формат Да, занятость Частичная. Вакансия: Python Backend Developer, опыт 6-7 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 252054-356651, локация Санкт-Петербург, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/6): Docker, FastAPI, PostgreSQL, Python. Недостающие навыки: Kafka, Redis.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (6-7 лет).\", \"salary_match\": \"Ожидания кандидата (158233-199365 руб.) не пересекаются с вилкой вакансии (252054-356651 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker, FastAPI, PostgreSQL, Python\"], \"weaknesses\": [\"Недостающие навыки: Kafka, Redis\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.78, \"match_score\": 58}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 7 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 247479-291414, локация Москва, формат Да, занятость Полная. Вакансия: DevOps Engineer, опыт 4-7 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Kubernetes, зарплата 334101-436520, локация Москва, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): AWS, Docker, GitLab CI, Kubernetes. Недостающие навыки: ArgoCD.\", \"experience_match\": \"Опыт кандидата 7 лет находится в диапазоне вакансии (4-7 лет).\", \"salary_match\": \"Ожидания кандидата (247479-291414 руб.) не пересекаются с вилкой вакансии (334101-436520 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: AWS, Docker, GitLab CI, Kubernetes\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD\"], \"recommendation\": \"hire\"}, \"confidence\": 0.87, \"match_score\": 80}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 6 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 314982-349616, локация Екатеринбург, формат Нет, занятость Полная. Вакансия: DevOps Engineer, опыт 4-5 лет, навыки ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 293433-412376, локация Екатеринбург, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): Docker, GitLab CI, Kubernetes, Terraform. Недостающие навыки: ArgoCD.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (314982-349616 руб.) пересекаются с вилкой вакансии (293433-412376 руб.).\", \"location_match\": \"Локация совпадает: Екатеринбург.\", \"strengths\": [\"Совпавшие навыки: Docker, GitLab CI, Kubernetes, Terraform\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD\"], \"recommendation\": \"hire\"}, \"confidence\": 0.9, \"match_score\": 87}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 7 лет, навыки Airflow, MLflow, NumPy, Pandas, Python, зарплата 135422-191357, локация Москва, формат Да, занятость Частичная. Вакансия: ML Engineer, опыт 5-6 лет, навыки Docker, MLOps, OpenCV, PyTorch, Python, зарплата 197227-264923, локация Москва, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/5): Python. Недостающие навыки: Docker, MLOps, OpenCV, PyTorch.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (135422-191357 руб.) не пересекаются с вилкой вакансии (197227-264923 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: Python\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Docker, MLOps, OpenCV, PyTorch\"], \"recommendation\": \"consider\"}, \"confidence\": 0.77, \"match_score\": 54}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 6 лет, навыки AsyncIO, FastAPI, PostgreSQL, Python, Redis, зарплата 227385-270391, локация Санкт-Петербург, формат Да, занятость Частичная. Вакансия: Python Backend Developer, опыт 6-9 лет, навыки Docker, Kafka, PostgreSQL, Python, Redis, зарплата 268894-327184, локация Санкт-Петербург, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): PostgreSQL, Python, Redis. Недостающие навыки: Docker, Kafka.\", \"experience_match\": \"Опыт кандидата 6 лет находится в диапазоне вакансии (6-9 лет).\", \"salary_match\": \"Ожидания кандидата (227385-270391 руб.) пересекаются с вилкой вакансии (268894-327184 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: PostgreSQL, Python, Redis\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Docker, Kafka\"], \"recommendation\": \"hire\"}, \"confidence\": 0.87, \"match_score\": 80}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 5 лет, навыки MLflow, Pandas, PyTorch, Python, TensorFlow, зарплата 246390-315473, локация Регионы, формат Нет, занятость Частичная. Вакансия: ML Engineer, опыт 4-7 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, Python, зарплата 277394-361674, локация Санкт-Петербург, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/6): PyTorch, Python. Недостающие навыки: Albumentations, Docker, MLOps, OpenCV.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (4-7 лет).\", \"salary_match\": \"Ожидания кандидата (246390-315473 руб.) пересекаются с вилкой вакансии (277394-361674 руб.).\", \"location_match\": \"Локация не совпадает (Регионы vs Санкт-Петербург).\", \"strengths\": [\"Совпавшие навыки: PyTorch, Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, Docker, MLOps, OpenCV\", \"Несовпадение по локации/формату\"], \"recommendation\": \"consider\"}, \"confidence\": 0.8, \"match_score\": 62}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 5 лет, навыки AsyncIO, Django, Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 278763-335398, локация Санкт-Петербург, формат Гибрид, занятость Частичная. Вакансия: Python Backend Developer, опыт 2-3 лет, навыки Docker, Kafka, PostgreSQL, Python, Redis, зарплата 248077-360845, локация Санкт-Петербург, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): Docker, PostgreSQL, Python, Redis. Недостающие навыки: Kafka.\", \"experience_match\": \"Опыт кандидата 5 лет выше верхней границы (3 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (278763-335398 руб.) пересекаются с вилкой вакансии (248077-360845 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: Docker, PostgreSQL, Python, Redis\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Kafka\"], \"recommendation\": \"hire\"}, \"confidence\": 0.91, \"match_score\": 89}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 7 лет, навыки Docker, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 260489-313969, локация Новосибирск, формат Да, занятость Полная. Вакансия: DevOps Engineer, опыт 5-6 лет, навыки AWS, Docker, GitLab CI, Kubernetes, зарплата 204430-303663, локация Новосибирск, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): Docker, GitLab CI, Kubernetes. Недостающие навыки: AWS.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (260489-313969 руб.) пересекаются с вилкой вакансии (204430-303663 руб.).\", \"location_match\": \"Локация совпадает: Новосибирск.\", \"strengths\": [\"Совпавшие навыки: Docker, GitLab CI, Kubernetes\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: AWS\"], \"recommendation\": \"hire\"}, \"confidence\": 0.89, \"match_score\": 85}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 2 лет, навыки Airflow, Docker, MLflow, PyTorch, Python, TensorFlow, зарплата 286377-365464, локация Регионы, формат Гибрид, занятость Частичная. Вакансия: DevOps Engineer, опыт 4-6 лет, навыки AWS, ArgoCD, Docker, Terraform, зарплата 208724-325343, локация Москва, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/4): Docker. Недостающие навыки: AWS, ArgoCD, Terraform.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (4-6 лет).\", \"salary_match\": \"Ожидания кандидата (286377-365464 руб.) пересекаются с вилкой вакансии (208724-325343 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: AWS, ArgoCD, Terraform\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.75, \"match_score\": 51}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 1 лет, навыки Airflow, Docker, NumPy, Pandas, PyTorch, Python, зарплата 234211-302428, локация Санкт-Петербург, формат Да, занятость Частичная. Вакансия: ML Engineer, опыт 5-7 лет, навыки Albumentations, OpenCV, PyTorch, Python, зарплата 140263-244266, локация Санкт-Петербург, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): PyTorch, Python. Недостающие навыки: Albumentations, OpenCV.\", \"experience_match\": \"Опыт кандидата 1 лет ниже диапазона вакансии (5-7 лет).\", \"salary_match\": \"Ожидания кандидата (234211-302428 руб.) пересекаются с вилкой вакансии (140263-244266 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: PyTorch, Python\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, OpenCV\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.77, \"match_score\": 55}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 1 лет, навыки A/B testing, Pandas, Power BI, Python, SQL, Statistics, Tableau, зарплата 234566-292206, локация Казань, формат Гибрид, занятость Полная. Вакансия: Data Analyst, опыт 2-3 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 122753-202751, локация Казань, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): A/B testing, Python, SQL, Tableau. Недостающие навыки: Product metrics.\", \"experience_match\": \"Опыт кандидата 1 лет ниже диапазона вакансии (2-3 лет).\", \"salary_match\": \"Ожидания кандидата (234566-292206 руб.) не пересекаются с вилкой вакансии (122753-202751 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: A/B testing, Python, SQL, Tableau\"], \"weaknesses\": [\"Недостающие навыки: Product metrics\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 70}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 4 лет, навыки HTML, Next.js, TypeScript, Vue, зарплата 72434-102962, локация Регионы, формат Да, занятость Частичная. Вакансия: Python Backend Developer, опыт 5-7 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 190065-245871, локация Санкт-Петербург, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/6): нет. Недостающие навыки: Docker, FastAPI, Kafka, PostgreSQL, Python, Redis.\", \"experience_match\": \"Опыт кандидата 4 лет ниже диапазона вакансии (5-7 лет).\", \"salary_match\": \"Ожидания кандидата (72434-102962 руб.) не пересекаются с вилкой вакансии (190065-245871 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [], \"weaknesses\": [\"Недостающие навыки: Docker, FastAPI, Kafka, PostgreSQL, Python\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.71, \"match_score\": 39}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 6 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Prometheus, зарплата 277212-362890, локация Санкт-Петербург, формат Да, занятость Частичная. Вакансия: DevOps Engineer, опыт 5-8 лет, навыки AWS, Docker, GitLab CI, Terraform, зарплата 251371-349850, локация Санкт-Петербург, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): AWS, Docker, GitLab CI. Недостающие навыки: Terraform.\", \"experience_match\": \"Опыт кандидата 6 лет находится в диапазоне вакансии (5-8 лет).\", \"salary_match\": \"Ожидания кандидата (277212-362890 руб.) пересекаются с вилкой вакансии (251371-349850 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: AWS, Docker, GitLab CI\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Terraform\"], \"recommendation\": \"hire\"}, \"confidence\": 0.9, \"match_score\": 88}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 8 лет, навыки HTML, Next.js, Redux, TypeScript, Vue, зарплата 96799-135482, локация Регионы, формат Гибрид, занятость Частичная. Вакансия: Frontend Developer, опыт 3-5 лет, навыки CSS, Next.js, React, Redux, TypeScript, зарплата 122884-187445, локация Регионы, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): Next.js, Redux, TypeScript. Недостающие навыки: CSS, React.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (96799-135482 руб.) пересекаются с вилкой вакансии (122884-187445 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: Next.js, Redux, TypeScript\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: CSS, React\"], \"recommendation\": \"consider\"}, \"confidence\": 0.86, \"match_score\": 78}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 3 лет, навыки CSS, HTML, JavaScript, React, Redux, TypeScript, Vue, зарплата 184864-217144, локация Санкт-Петербург, формат Нет, занятость Полная. Вакансия: Frontend Developer, опыт 1-2 лет, навыки CSS, Next.js, REST API, Redux, зарплата 133090-181902, локация Казань, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): CSS, Redux. Недостающие навыки: Next.js, REST API.\", \"experience_match\": \"Опыт кандидата 3 лет выше верхней границы (2 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (184864-217144 руб.) не пересекаются с вилкой вакансии (133090-181902 руб.).\", \"location_match\": \"Локация не совпадает (Санкт-Петербург vs Казань).\", \"strengths\": [\"Совпавшие навыки: CSS, Redux\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Next.js, REST API\", \"Несовпадение по локации/формату\"], \"recommendation\": \"consider\"}, \"confidence\": 0.8, \"match_score\": 62}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 6 лет, навыки CSS, HTML, JavaScript, Redux, зарплата 77249-159680, локация Санкт-Петербург, формат Нет, занятость Полная. Вакансия: DevOps Engineer, опыт 4-6 лет, навыки AWS, Docker, GitLab CI, Kubernetes, зарплата 198261-247355, локация Регионы, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: AWS, Docker, GitLab CI, Kubernetes.\", \"experience_match\": \"Опыт кандидата 6 лет находится в диапазоне вакансии (4-6 лет).\", \"salary_match\": \"Ожидания кандидата (77249-159680 руб.) не пересекаются с вилкой вакансии (198261-247355 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: AWS, Docker, GitLab CI, Kubernetes\"], \"recommendation\": \"reject\"}, \"confidence\": 0.73, \"match_score\": 44}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 8 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Terraform, зарплата 246476-288188, локация Регионы, формат Нет, занятость Частичная. Вакансия: DevOps Engineer, опыт 6-9 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 247049-360330, локация Регионы, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (5/6): AWS, Docker, GitLab CI, Kubernetes, Terraform. Недостающие навыки: ArgoCD.\", \"experience_match\": \"Опыт кандидата 8 лет находится в диапазоне вакансии (6-9 лет).\", \"salary_match\": \"Ожидания кандидата (246476-288188 руб.) пересекаются с вилкой вакансии (247049-360330 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: AWS, Docker, GitLab CI, Kubernetes, Terraform\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD\"], \"recommendation\": \"hire\"}, \"confidence\": 0.92, \"match_score\": 92}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 8 лет, навыки AWS, GitLab CI, Kubernetes, Prometheus, Terraform, зарплата 264940-305363, локация Москва, формат Нет, занятость Полная. Вакансия: DevOps Engineer, опыт 7-9 лет, навыки ArgoCD, Docker, Kubernetes, Terraform, зарплата 234516-301223, локация Москва, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): Kubernetes, Terraform. Недостающие навыки: ArgoCD, Docker.\", \"experience_match\": \"Опыт кандидата 8 лет находится в диапазоне вакансии (7-9 лет).\", \"salary_match\": \"Ожидания кандидата (264940-305363 руб.) пересекаются с вилкой вакансии (234516-301223 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: Kubernetes, Terraform\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, Docker\"], \"recommendation\": \"consider\"}, \"confidence\": 0.86, \"match_score\": 77}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 1 лет, навыки Airflow, MLflow, NumPy, Pandas, PyTorch, TensorFlow, зарплата 152705-238661, локация Санкт-Петербург, формат Гибрид, занятость Частичная. Вакансия: ML Engineer, опыт 4-5 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, Python, зарплата 290014-358105, локация Санкт-Петербург, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/6): PyTorch. Недостающие навыки: Albumentations, Docker, MLOps, OpenCV, Python.\", \"experience_match\": \"Опыт кандидата 1 лет ниже диапазона вакансии (4-5 лет).\", \"salary_match\": \"Ожидания кандидата (152705-238661 руб.) не пересекаются с вилкой вакансии (290014-358105 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: PyTorch\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, Docker, MLOps, OpenCV, Python\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.69, \"match_score\": 35}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 8 лет, навыки AsyncIO, Django, Docker, FastAPI, Redis, зарплата 131263-185514, локация Екатеринбург, формат Да, занятость Полная. Вакансия: Frontend Developer, опыт 5-7 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 201693-284210, локация Москва, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/6): нет. Недостающие навыки: CSS, Next.js, REST API, React, Redux, TypeScript.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (7 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (131263-185514 руб.) не пересекаются с вилкой вакансии (201693-284210 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: CSS, Next.js, REST API, React, Redux\"], \"recommendation\": \"reject\"}, \"confidence\": 0.73, \"match_score\": 45}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 5 лет, навыки A/B testing, Pandas, Power BI, Python, SQL, Tableau, зарплата 116383-196114, локация Москва, формат Нет, занятость Полная. Вакансия: Data Analyst, опыт 4-7 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 232974-331773, локация Москва, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): A/B testing, Python, SQL, Tableau. Недостающие навыки: Product metrics.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (4-7 лет).\", \"salary_match\": \"Ожидания кандидата (116383-196114 руб.) не пересекаются с вилкой вакансии (232974-331773 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: A/B testing, Python, SQL, Tableau\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Product metrics\"], \"recommendation\": \"hire\"}, \"confidence\": 0.87, \"match_score\": 80}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 2 лет, навыки Django, Docker, FastAPI, PostgreSQL, Python, зарплата 230239-261689, локация Казань, формат Гибрид, занятость Полная. Вакансия: Python Backend Developer, опыт 3-4 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, зарплата 168742-222147, локация Казань, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): Docker, FastAPI, PostgreSQL, Python. Недостающие навыки: Kafka.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (3-4 лет).\", \"salary_match\": \"Ожидания кандидата (230239-261689 руб.) не пересекаются с вилкой вакансии (168742-222147 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: Docker, FastAPI, PostgreSQL, Python\"], \"weaknesses\": [\"Недостающие навыки: Kafka\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 74}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 3 лет, навыки AsyncIO, Django, Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 179001-223521, локация Москва, формат Да, занятость Полная. Вакансия: Python Backend Developer, опыт 5-7 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, зарплата 207311-269348, локация Москва, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): Docker, FastAPI, PostgreSQL, Python. Недостающие навыки: Kafka.\", \"experience_match\": \"Опыт кандидата 3 лет ниже диапазона вакансии (5-7 лет).\", \"salary_match\": \"Ожидания кандидата (179001-223521 руб.) пересекаются с вилкой вакансии (207311-269348 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: Docker, FastAPI, PostgreSQL, Python\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Kafka\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.87, \"match_score\": 79}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 3 лет, навыки A/B testing, Pandas, Power BI, Python, SQL, Statistics, Tableau, зарплата 112706-173029, локация Регионы, формат Гибрид, занятость Частичная. Вакансия: Data Analyst, опыт 6-7 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 207119-272414, локация Регионы, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): A/B testing, Python, SQL, Tableau. Недостающие навыки: Product metrics.\", \"experience_match\": \"Опыт кандидата 3 лет ниже диапазона вакансии (6-7 лет).\", \"salary_match\": \"Ожидания кандидата (112706-173029 руб.) не пересекаются с вилкой вакансии (207119-272414 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: A/B testing, Python, SQL, Tableau\"], \"weaknesses\": [\"Недостающие навыки: Product metrics\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 70}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 8 лет, навыки Django, Docker, FastAPI, PostgreSQL, Python, зарплата 203389-288723, локация Москва, формат Нет, занятость Полная. Вакансия: ML Engineer, опыт 2-4 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, Python, зарплата 226660-302578, локация Екатеринбург, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/6): Docker, Python. Недостающие навыки: Albumentations, MLOps, OpenCV, PyTorch.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (203389-288723 руб.) пересекаются с вилкой вакансии (226660-302578 руб.).\", \"location_match\": \"Локация не совпадает (Москва vs Екатеринбург).\", \"strengths\": [\"Совпавшие навыки: Docker, Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV, PyTorch\", \"Несовпадение по локации/формату\"], \"recommendation\": \"consider\"}, \"confidence\": 0.8, \"match_score\": 62}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 8 лет, навыки AsyncIO, Docker, FastAPI, PostgreSQL, Redis, зарплата 106889-178434, локация Казань, формат Нет, занятость Полная. Вакансия: DevOps Engineer, опыт 5-7 лет, навыки AWS, ArgoCD, GitLab CI, Kubernetes, Terraform, зарплата 173361-267420, локация Казань, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/5): нет. Недостающие навыки: AWS, ArgoCD, GitLab CI, Kubernetes, Terraform.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (7 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (106889-178434 руб.) пересекаются с вилкой вакансии (173361-267420 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: AWS, ArgoCD, GitLab CI, Kubernetes, Terraform\"], \"recommendation\": \"consider\"}, \"confidence\": 0.75, \"match_score\": 51}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 1 лет, навыки CSS, JavaScript, Next.js, Redux, зарплата 144498-228286, локация Казань, формат Нет, занятость Полная. Вакансия: Frontend Developer, опыт 2-4 лет, навыки Next.js, React, Redux, TypeScript, зарплата 211716-328323, локация Казань, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): Next.js, Redux. Недостающие навыки: React, TypeScript.\", \"experience_match\": \"Опыт кандидата 1 лет ниже диапазона вакансии (2-4 лет).\", \"salary_match\": \"Ожидания кандидата (144498-228286 руб.) пересекаются с вилкой вакансии (211716-328323 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: Next.js, Redux\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: React, TypeScript\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.81, \"match_score\": 65}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 5 лет, навыки AWS, Docker, Kubernetes, Linux, Prometheus, Terraform, зарплата 265821-319909, локация Регионы, формат Нет, занятость Полная. Вакансия: DevOps Engineer, опыт 2-4 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Kubernetes, зарплата 180116-256628, локация Регионы, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): AWS, Docker, Kubernetes. Недостающие навыки: ArgoCD, GitLab CI.\", \"experience_match\": \"Опыт кандидата 5 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (265821-319909 руб.) не пересекаются с вилкой вакансии (180116-256628 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: AWS, Docker, Kubernetes\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, GitLab CI\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 70}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 4 лет, навыки A/B testing, Power BI, Python, Tableau, зарплата 91056-135408, локация Санкт-Петербург, формат Нет, занятость Частичная. Вакансия: Frontend Developer, опыт 4-5 лет, навыки CSS, Next.js, React, TypeScript, зарплата 170991-230996, локация Казань, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: CSS, Next.js, React, TypeScript.\", \"experience_match\": \"Опыт кандидата 4 лет находится в диапазоне вакансии (4-5 лет).\", \"salary_match\": \"Ожидания кандидата (91056-135408 руб.) не пересекаются с вилкой вакансии (170991-230996 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: CSS, Next.js, React, TypeScript\"], \"recommendation\": \"reject\"}, \"confidence\": 0.73, \"match_score\": 44}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 3 лет, навыки HTML, JavaScript, Next.js, TypeScript, зарплата 94080-173537, локация Казань, формат Нет, занятость Частичная. Вакансия: Frontend Developer, опыт 6-7 лет, навыки CSS, REST API, React, Redux, TypeScript, зарплата 194454-281257, локация Казань, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/5): TypeScript. Недостающие навыки: CSS, REST API, React, Redux.\", \"experience_match\": \"Опыт кандидата 3 лет ниже диапазона вакансии (6-7 лет).\", \"salary_match\": \"Ожидания кандидата (94080-173537 руб.) не пересекаются с вилкой вакансии (194454-281257 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: TypeScript\"], \"weaknesses\": [\"Недостающие навыки: CSS, REST API, React, Redux\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.72, \"match_score\": 43}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 2 лет, навыки AsyncIO, Django, Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 227692-274413, локация Казань, формат Нет, занятость Полная. Вакансия: Data Analyst, опыт 4-5 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 238777-298310, локация Екатеринбург, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/5): Python. Недостающие навыки: A/B testing, Product metrics, SQL, Tableau.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (4-5 лет).\", \"salary_match\": \"Ожидания кандидата (227692-274413 руб.) пересекаются с вилкой вакансии (238777-298310 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Python\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, SQL, Tableau\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.75, \"match_score\": 49}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 3 лет, навыки Docker, MLflow, NumPy, TensorFlow, зарплата 203723-292346, локация Москва, формат Нет, занятость Полная. Вакансия: ML Engineer, опыт 2-4 лет, навыки Albumentations, Docker, MLOps, OpenCV, Python, зарплата 254744-329911, локация Москва, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/5): Docker. Недостающие навыки: Albumentations, MLOps, OpenCV, Python.\", \"experience_match\": \"Опыт кандидата 3 лет находится в диапазоне вакансии (2-4 лет).\", \"salary_match\": \"Ожидания кандидата (203723-292346 руб.) пересекаются с вилкой вакансии (254744-329911 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: Docker\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV, Python\"], \"recommendation\": \"consider\"}, \"confidence\": 0.8, \"match_score\": 62}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 3 лет, навыки CSS, HTML, JavaScript, Redux, Vue, зарплата 80965-131167, локация Казань, формат Да, занятость Полная. Вакансия: Frontend Developer, опыт 4-5 лет, навыки CSS, REST API, React, TypeScript, зарплата 195591-240820, локация Казань, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/4): CSS. Недостающие навыки: REST API, React, TypeScript.\", \"experience_match\": \"Опыт кандидата 3 лет ниже диапазона вакансии (4-5 лет).\", \"salary_match\": \"Ожидания кандидата (80965-131167 руб.) не пересекаются с вилкой вакансии (195591-240820 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: CSS\"], \"weaknesses\": [\"Недостающие навыки: REST API, React, TypeScript\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.75, \"match_score\": 49}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 8 лет, навыки HTML, Next.js, React, Redux, TypeScript, зарплата 89204-154438, локация Казань, формат Нет, занятость Полная. Вакансия: Frontend Developer, опыт 5-6 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 155877-262184, локация Казань, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/6): Next.js, React, Redux, TypeScript. Недостающие навыки: CSS, REST API.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (89204-154438 руб.) не пересекаются с вилкой вакансии (155877-262184 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: Next.js, React, Redux, TypeScript\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: CSS, REST API\"], \"recommendation\": \"consider\"}, \"confidence\": 0.84, \"match_score\": 73}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 3 лет, навыки CSS, HTML, JavaScript, TypeScript, Vue, зарплата 121219-190713, локация Москва, формат Да, занятость Полная. Вакансия: Frontend Developer, опыт 2-3 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 120197-169028, локация Москва, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/6): CSS, TypeScript. Недостающие навыки: Next.js, REST API, React, Redux.\", \"experience_match\": \"Опыт кандидата 3 лет находится в диапазоне вакансии (2-3 лет).\", \"salary_match\": \"Ожидания кандидата (121219-190713 руб.) пересекаются с вилкой вакансии (120197-169028 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: CSS, TypeScript\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Next.js, REST API, React, Redux\"], \"recommendation\": \"consider\"}, \"confidence\": 0.82, \"match_score\": 68}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 10 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Prometheus, Terraform, зарплата 176072-216958, локация Санкт-Петербург, формат Нет, занятость Частичная. Вакансия: DevOps Engineer, опыт 2-5 лет, навыки AWS, ArgoCD, GitLab CI, Kubernetes, Terraform, зарплата 316138-392541, локация Регионы, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): AWS, GitLab CI, Kubernetes, Terraform. Недостающие навыки: ArgoCD.\", \"experience_match\": \"Опыт кандидата 10 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (176072-216958 руб.) не пересекаются с вилкой вакансии (316138-392541 руб.).\", \"location_match\": \"Локация не совпадает (Санкт-Петербург vs Регионы).\", \"strengths\": [\"Совпавшие навыки: AWS, GitLab CI, Kubernetes, Terraform\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD\", \"Несовпадение по локации/формату\"], \"recommendation\": \"consider\"}, \"confidence\": 0.84, \"match_score\": 73}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 4 лет, навыки AWS, Docker, Linux, Terraform, зарплата 140707-186766, локация Регионы, формат Да, занятость Частичная. Вакансия: DevOps Engineer, опыт 4-5 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 170202-211837, локация Екатеринбург, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): AWS, Docker, Terraform. Недостающие навыки: ArgoCD, GitLab CI, Kubernetes.\", \"experience_match\": \"Опыт кандидата 4 лет находится в диапазоне вакансии (4-5 лет).\", \"salary_match\": \"Ожидания кандидата (140707-186766 руб.) пересекаются с вилкой вакансии (170202-211837 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: AWS, Docker, Terraform\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, GitLab CI, Kubernetes\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 75}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 6 лет, навыки CSS, HTML, JavaScript, React, Redux, TypeScript, Vue, зарплата 95154-136124, локация Москва, формат Гибрид, занятость Частичная. Вакансия: Frontend Developer, опыт 5-6 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 239776-280736, локация Москва, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/6): CSS, React, Redux, TypeScript. Недостающие навыки: Next.js, REST API.\", \"experience_match\": \"Опыт кандидата 6 лет находится в диапазоне вакансии (5-6 лет).\", \"salary_match\": \"Ожидания кандидата (95154-136124 руб.) не пересекаются с вилкой вакансии (239776-280736 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: CSS, React, Redux, TypeScript\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Next.js, REST API\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 74}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 8 лет, навыки CSS, HTML, JavaScript, Next.js, React, Vue, зарплата 112737-149449, локация Санкт-Петербург, формат Нет, занятость Полная. Вакансия: Frontend Developer, опыт 4-6 лет, навыки CSS, REST API, Redux, TypeScript, зарплата 157114-217371, локация Москва, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/4): CSS. Недостающие навыки: REST API, Redux, TypeScript.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (112737-149449 руб.) не пересекаются с вилкой вакансии (157114-217371 руб.).\", \"location_match\": \"Локация не совпадает (Санкт-Петербург vs Москва).\", \"strengths\": [\"Совпавшие навыки: CSS\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: REST API, Redux, TypeScript\", \"Несовпадение по локации/формату\"], \"recommendation\": \"consider\"}, \"confidence\": 0.75, \"match_score\": 50}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 6 лет, навыки CSS, HTML, JavaScript, React, Redux, TypeScript, Vue, зарплата 147441-183767, локация Казань, формат Нет, занятость Полная. Вакансия: Frontend Developer, опыт 5-7 лет, навыки CSS, Next.js, REST API, Redux, TypeScript, зарплата 157391-212119, локация Казань, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): CSS, Redux, TypeScript. Недостающие навыки: Next.js, REST API.\", \"experience_match\": \"Опыт кандидата 6 лет находится в диапазоне вакансии (5-7 лет).\", \"salary_match\": \"Ожидания кандидата (147441-183767 руб.) пересекаются с вилкой вакансии (157391-212119 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: CSS, Redux, TypeScript\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Next.js, REST API\"], \"recommendation\": \"hire\"}, \"confidence\": 0.87, \"match_score\": 80}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 7 лет, навыки Django, FastAPI, PostgreSQL, Python, Redis, зарплата 104690-152155, локация Екатеринбург, формат Да, занятость Частичная. Вакансия: Python Backend Developer, опыт 2-3 лет, навыки Docker, Kafka, PostgreSQL, Python, Redis, зарплата 222871-272227, локация Екатеринбург, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): PostgreSQL, Python, Redis. Недостающие навыки: Docker, Kafka.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (3 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (104690-152155 руб.) не пересекаются с вилкой вакансии (222871-272227 руб.).\", \"location_match\": \"Локация совпадает: Екатеринбург.\", \"strengths\": [\"Совпавшие навыки: PostgreSQL, Python, Redis\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Docker, Kafka\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 70}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 8 лет, навыки HTML, JavaScript, Next.js, React, Redux, TypeScript, Vue, зарплата 99612-150353, локация Санкт-Петербург, формат Нет, занятость Частичная. Вакансия: Frontend Developer, опыт 5-7 лет, навыки CSS, Next.js, REST API, React, Redux, зарплата 211091-313331, локация Санкт-Петербург, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): Next.js, React, Redux. Недостающие навыки: CSS, REST API.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (7 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (99612-150353 руб.) не пересекаются с вилкой вакансии (211091-313331 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: Next.js, React, Redux\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: CSS, REST API\"], \"recommendation\": \"consider\"}, \"confidence\": 0.84, \"match_score\": 72}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 8 лет, навыки AWS, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 270683-320073, локация Москва, формат Нет, занятость Полная. Вакансия: DevOps Engineer, опыт 6-8 лет, навыки AWS, ArgoCD, Docker, GitLab CI, зарплата 333311-412007, локация Екатеринбург, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): AWS, GitLab CI. Недостающие навыки: ArgoCD, Docker.\", \"experience_match\": \"Опыт кандидата 8 лет находится в диапазоне вакансии (6-8 лет).\", \"salary_match\": \"Ожидания кандидата (270683-320073 руб.) не пересекаются с вилкой вакансии (333311-412007 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: AWS, GitLab CI\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, Docker\"], \"recommendation\": \"consider\"}, \"confidence\": 0.82, \"match_score\": 67}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 5 лет, навыки Airflow, Docker, MLflow, NumPy, PyTorch, зарплата 209336-293092, локация Москва, формат Да, занятость Частичная. Вакансия: ML Engineer, опыт 3-5 лет, навыки Albumentations, Docker, MLOps, OpenCV, Python, зарплата 157738-253748, локация Москва, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/5): Docker. Недостающие навыки: Albumentations, MLOps, OpenCV, Python.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (3-5 лет).\", \"salary_match\": \"Ожидания кандидата (209336-293092 руб.) пересекаются с вилкой вакансии (157738-253748 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: Docker\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV, Python\"], \"recommendation\": \"consider\"}, \"confidence\": 0.81, \"match_score\": 64}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 7 лет, навыки AsyncIO, Django, FastAPI, Redis, зарплата 177814-234561, локация Казань, формат Да, занятость Полная. Вакансия: Python Backend Developer, опыт 6-8 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 218229-319922, локация Казань, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/6): FastAPI, Redis. Недостающие навыки: Docker, Kafka, PostgreSQL, Python.\", \"experience_match\": \"Опыт кандидата 7 лет находится в диапазоне вакансии (6-8 лет).\", \"salary_match\": \"Ожидания кандидата (177814-234561 руб.) пересекаются с вилкой вакансии (218229-319922 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: FastAPI, Redis\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Docker, Kafka, PostgreSQL, Python\"], \"recommendation\": \"consider\"}, \"confidence\": 0.82, \"match_score\": 68}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 9 лет, навыки AWS, Docker, Linux, Terraform, зарплата 260008-333346, локация Москва, формат Гибрид, занятость Частичная. Вакансия: DevOps Engineer, опыт 6-7 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 294797-372487, локация Екатеринбург, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): AWS, Docker, Terraform. Недостающие навыки: ArgoCD, GitLab CI, Kubernetes.\", \"experience_match\": \"Опыт кандидата 9 лет выше верхней границы (7 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (260008-333346 руб.) пересекаются с вилкой вакансии (294797-372487 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: AWS, Docker, Terraform\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, GitLab CI, Kubernetes\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 74}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 8 лет, навыки Django, Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 278420-348901, локация Москва, формат Да, занятость Полная. Вакансия: Python Backend Developer, опыт 2-5 лет, навыки Docker, Kafka, PostgreSQL, Python, Redis, зарплата 150863-227917, локация Новосибирск, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): Docker, PostgreSQL, Python, Redis. Недостающие навыки: Kafka.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (278420-348901 руб.) не пересекаются с вилкой вакансии (150863-227917 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker, PostgreSQL, Python, Redis\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Kafka\"], \"recommendation\": \"consider\"}, \"confidence\": 0.87, \"match_score\": 79}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 5 лет, навыки CSS, HTML, Next.js, Redux, TypeScript, Vue, зарплата 152064-241427, локация Санкт-Петербург, формат Да, занятость Частичная. Вакансия: Data Analyst, опыт 5-6 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 184977-256357, локация Казань, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/5): нет. Недостающие навыки: A/B testing, Product metrics, Python, SQL, Tableau.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (5-6 лет).\", \"salary_match\": \"Ожидания кандидата (152064-241427 руб.) пересекаются с вилкой вакансии (184977-256357 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, Python, SQL, Tableau\"], \"recommendation\": \"consider\"}, \"confidence\": 0.77, \"match_score\": 55}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 4 лет, навыки Airflow, Docker, MLflow, NumPy, Pandas, PyTorch, Python, зарплата 212214-296384, локация Регионы, формат Да, занятость Полная. Вакансия: ML Engineer, опыт 3-4 лет, навыки Albumentations, Docker, MLOps, Python, зарплата 199866-288596, локация Регионы, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): Docker, Python. Недостающие навыки: Albumentations, MLOps.\", \"experience_match\": \"Опыт кандидата 4 лет находится в диапазоне вакансии (3-4 лет).\", \"salary_match\": \"Ожидания кандидата (212214-296384 руб.) пересекаются с вилкой вакансии (199866-288596 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: Docker, Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 75}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 2 лет, навыки Docker, MLflow, NumPy, Pandas, PyTorch, Python, зарплата 137159-198206, локация Регионы, формат Да, занятость Частичная. Вакансия: ML Engineer, опыт 2-3 лет, навыки Albumentations, Docker, PyTorch, Python, зарплата 146463-252759, локация Регионы, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): Docker, PyTorch, Python. Недостающие навыки: Albumentations.\", \"experience_match\": \"Опыт кандидата 2 лет находится в диапазоне вакансии (2-3 лет).\", \"salary_match\": \"Ожидания кандидата (137159-198206 руб.) пересекаются с вилкой вакансии (146463-252759 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: Docker, PyTorch, Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations\"], \"recommendation\": \"hire\"}, \"confidence\": 0.9, \"match_score\": 88}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 5 лет, навыки AWS, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 246719-294557, локация Новосибирск, формат Гибрид, занятость Полная. Вакансия: DevOps Engineer, опыт 2-4 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 236023-342717, локация Новосибирск, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/6): AWS, GitLab CI, Kubernetes, Terraform. Недостающие навыки: ArgoCD, Docker.\", \"experience_match\": \"Опыт кандидата 5 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (246719-294557 руб.) пересекаются с вилкой вакансии (236023-342717 руб.).\", \"location_match\": \"Локация совпадает: Новосибирск.\", \"strengths\": [\"Совпавшие навыки: AWS, GitLab CI, Kubernetes, Terraform\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, Docker\"], \"recommendation\": \"hire\"}, \"confidence\": 0.87, \"match_score\": 81}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 3 лет, навыки Pandas, Power BI, Python, SQL, зарплата 228782-306978, локация Казань, формат Гибрид, занятость Частичная. Вакансия: Frontend Developer, опыт 2-4 лет, навыки CSS, REST API, Redux, TypeScript, зарплата 180089-227018, локация Казань, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: CSS, REST API, Redux, TypeScript.\", \"experience_match\": \"Опыт кандидата 3 лет находится в диапазоне вакансии (2-4 лет).\", \"salary_match\": \"Ожидания кандидата (228782-306978 руб.) не пересекаются с вилкой вакансии (180089-227018 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: CSS, REST API, Redux, TypeScript\"], \"recommendation\": \"reject\"}, \"confidence\": 0.73, \"match_score\": 44}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 2 лет, навыки CSS, HTML, JavaScript, Next.js, React, Redux, TypeScript, зарплата 134756-178238, локация Регионы, формат Нет, занятость Частичная. Вакансия: Frontend Developer, опыт 6-8 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 188440-252436, локация Регионы, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (5/6): CSS, Next.js, React, Redux, TypeScript. Недостающие навыки: REST API.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (6-8 лет).\", \"salary_match\": \"Ожидания кандидата (134756-178238 руб.) не пересекаются с вилкой вакансии (188440-252436 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: CSS, Next.js, React, Redux, TypeScript\"], \"weaknesses\": [\"Недостающие навыки: REST API\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.82, \"match_score\": 67}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 3 лет, навыки AsyncIO, Django, FastAPI, PostgreSQL, Redis, зарплата 118252-171127, локация Екатеринбург, формат Да, занятость Полная. Вакансия: Python Backend Developer, опыт 4-6 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 167295-284641, локация Екатеринбург, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): FastAPI, PostgreSQL, Redis. Недостающие навыки: Docker, Kafka, Python.\", \"experience_match\": \"Опыт кандидата 3 лет ниже диапазона вакансии (4-6 лет).\", \"salary_match\": \"Ожидания кандидата (118252-171127 руб.) пересекаются с вилкой вакансии (167295-284641 руб.).\", \"location_match\": \"Локация совпадает: Екатеринбург.\", \"strengths\": [\"Совпавшие навыки: FastAPI, PostgreSQL, Redis\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Docker, Kafka, Python\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 71}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 6 лет, навыки AWS, Docker, Kubernetes, Linux, Prometheus, зарплата 128468-181974, локация Регионы, формат Да, занятость Частичная. Вакансия: DevOps Engineer, опыт 3-6 лет, навыки ArgoCD, Docker, Kubernetes, Terraform, зарплата 268644-322076, локация Регионы, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): Docker, Kubernetes. Недостающие навыки: ArgoCD, Terraform.\", \"experience_match\": \"Опыт кандидата 6 лет находится в диапазоне вакансии (3-6 лет).\", \"salary_match\": \"Ожидания кандидата (128468-181974 руб.) не пересекаются с вилкой вакансии (268644-322076 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: Docker, Kubernetes\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, Terraform\"], \"recommendation\": \"consider\"}, \"confidence\": 0.82, \"match_score\": 67}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 2 лет, навыки Docker, GitLab CI, Prometheus, Terraform, зарплата 267545-317457, локация Москва, формат Гибрид, занятость Полная. Вакансия: Frontend Developer, опыт 3-4 лет, навыки CSS, Next.js, REST API, React, TypeScript, зарплата 172329-244859, локация Регионы, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/5): нет. Недостающие навыки: CSS, Next.js, REST API, React, TypeScript.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (3-4 лет).\", \"salary_match\": \"Ожидания кандидата (267545-317457 руб.) не пересекаются с вилкой вакансии (172329-244859 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [], \"weaknesses\": [\"Недостающие навыки: CSS, Next.js, REST API, React, TypeScript\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.69, \"match_score\": 36}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 6 лет, навыки Airflow, Docker, MLflow, NumPy, Pandas, PyTorch, TensorFlow, зарплата 134774-201340, локация Казань, формат Нет, занятость Полная. Вакансия: ML Engineer, опыт 1-3 лет, навыки Albumentations, Docker, MLOps, OpenCV, Python, зарплата 241335-318578, локация Санкт-Петербург, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/5): Docker. Недостающие навыки: Albumentations, MLOps, OpenCV, Python.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (3 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (134774-201340 руб.) не пересекаются с вилкой вакансии (241335-318578 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV, Python\"], \"recommendation\": \"consider\"}, \"confidence\": 0.76, \"match_score\": 52}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 4 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 278314-340777, локация Санкт-Петербург, формат Гибрид, занятость Частичная. Вакансия: Data Analyst, опыт 3-6 лет, навыки A/B testing, Product metrics, SQL, Tableau, зарплата 222839-289349, локация Москва, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: A/B testing, Product metrics, SQL, Tableau.\", \"experience_match\": \"Опыт кандидата 4 лет находится в диапазоне вакансии (3-6 лет).\", \"salary_match\": \"Ожидания кандидата (278314-340777 руб.) пересекаются с вилкой вакансии (222839-289349 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, SQL, Tableau\"], \"recommendation\": \"consider\"}, \"confidence\": 0.77, \"match_score\": 55}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 8 лет, навыки Power BI, Python, SQL, Statistics, Tableau, зарплата 144812-229853, локация Регионы, формат Нет, занятость Полная. Вакансия: Data Analyst, опыт 4-6 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 182695-291022, локация Регионы, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): Python, SQL, Tableau. Недостающие навыки: A/B testing, Product metrics.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (144812-229853 руб.) пересекаются с вилкой вакансии (182695-291022 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: Python, SQL, Tableau\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics\"], \"recommendation\": \"consider\"}, \"confidence\": 0.86, \"match_score\": 78}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 4 лет, навыки HTML, JavaScript, Next.js, Redux, TypeScript, Vue, зарплата 172995-217123, локация Казань, формат Нет, занятость Полная. Вакансия: Frontend Developer, опыт 2-3 лет, навыки Next.js, REST API, React, TypeScript, зарплата 219811-284790, локация Казань, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): Next.js, TypeScript. Недостающие навыки: REST API, React.\", \"experience_match\": \"Опыт кандидата 4 лет выше верхней границы (3 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (172995-217123 руб.) не пересекаются с вилкой вакансии (219811-284790 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: Next.js, TypeScript\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: REST API, React\"], \"recommendation\": \"consider\"}, \"confidence\": 0.81, \"match_score\": 66}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 6 лет, навыки AsyncIO, Django, FastAPI, PostgreSQL, Python, зарплата 118440-198312, локация Санкт-Петербург, формат Нет, занятость Полная. Вакансия: Python Backend Developer, опыт 6-8 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Redis, зарплата 151300-243061, локация Екатеринбург, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): FastAPI, PostgreSQL. Недостающие навыки: Docker, Kafka, Redis.\", \"experience_match\": \"Опыт кандидата 6 лет находится в диапазоне вакансии (6-8 лет).\", \"salary_match\": \"Ожидания кандидата (118440-198312 руб.) пересекаются с вилкой вакансии (151300-243061 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: FastAPI, PostgreSQL\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Docker, Kafka, Redis\"], \"recommendation\": \"consider\"}, \"confidence\": 0.84, \"match_score\": 73}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 7 лет, навыки Airflow, Docker, MLflow, NumPy, Pandas, TensorFlow, зарплата 236810-291485, локация Санкт-Петербург, формат Гибрид, занятость Частичная. Вакансия: DevOps Engineer, опыт 1-4 лет, навыки AWS, ArgoCD, GitLab CI, Kubernetes, Terraform, зарплата 263667-383075, локация Регионы, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/5): нет. Недостающие навыки: AWS, ArgoCD, GitLab CI, Kubernetes, Terraform.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (236810-291485 руб.) пересекаются с вилкой вакансии (263667-383075 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: AWS, ArgoCD, GitLab CI, Kubernetes, Terraform\"], \"recommendation\": \"consider\"}, \"confidence\": 0.75, \"match_score\": 51}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 6 лет, навыки AsyncIO, Docker, FastAPI, PostgreSQL, Redis, зарплата 198657-252748, локация Новосибирск, формат Да, занятость Полная. Вакансия: Python Backend Developer, опыт 3-4 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, зарплата 167681-259257, локация Екатеринбург, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): Docker, FastAPI, PostgreSQL. Недостающие навыки: Kafka, Python.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (198657-252748 руб.) пересекаются с вилкой вакансии (167681-259257 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker, FastAPI, PostgreSQL\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Kafka, Python\"], \"recommendation\": \"hire\"}, \"confidence\": 0.87, \"match_score\": 80}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 7 лет, навыки A/B testing, Pandas, Power BI, Python, Tableau, зарплата 189397-223744, локация Москва, формат Нет, занятость Частичная. Вакансия: Data Analyst, опыт 5-6 лет, навыки A/B testing, Python, SQL, Tableau, зарплата 132240-191319, локация Москва, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): A/B testing, Python, Tableau. Недостающие навыки: SQL.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (189397-223744 руб.) пересекаются с вилкой вакансии (132240-191319 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: A/B testing, Python, Tableau\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: SQL\"], \"recommendation\": \"hire\"}, \"confidence\": 0.9, \"match_score\": 87}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 4 лет, навыки AWS, Docker, Kubernetes, Linux, Prometheus, Terraform, зарплата 216482-260592, локация Екатеринбург, формат Нет, занятость Полная. Вакансия: Frontend Developer, опыт 2-3 лет, навыки CSS, Next.js, REST API, Redux, зарплата 319285-384291, локация Екатеринбург, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: CSS, Next.js, REST API, Redux.\", \"experience_match\": \"Опыт кандидата 4 лет выше верхней границы (3 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (216482-260592 руб.) не пересекаются с вилкой вакансии (319285-384291 руб.).\", \"location_match\": \"Локация совпадает: Екатеринбург.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: CSS, Next.js, REST API, Redux\"], \"recommendation\": \"reject\"}, \"confidence\": 0.72, \"match_score\": 43}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 8 лет, навыки CSS, HTML, JavaScript, React, Redux, TypeScript, Vue, зарплата 197656-263810, локация Москва, формат Гибрид, занятость Частичная. Вакансия: Frontend Developer, опыт 3-5 лет, навыки CSS, Next.js, REST API, Redux, TypeScript, зарплата 128529-191567, локация Санкт-Петербург, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): CSS, Redux, TypeScript. Недостающие навыки: Next.js, REST API.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (197656-263810 руб.) не пересекаются с вилкой вакансии (128529-191567 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: CSS, Redux, TypeScript\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Next.js, REST API\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 70}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 10 лет, навыки AWS, Docker, GitLab CI, Linux, Terraform, зарплата 137496-213243, локация Екатеринбург, формат Гибрид, занятость Частичная. Вакансия: DevOps Engineer, опыт 4-7 лет, навыки AWS, ArgoCD, Kubernetes, Terraform, зарплата 339375-407851, локация Екатеринбург, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): AWS, Terraform. Недостающие навыки: ArgoCD, Kubernetes.\", \"experience_match\": \"Опыт кандидата 10 лет выше верхней границы (7 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (137496-213243 руб.) не пересекаются с вилкой вакансии (339375-407851 руб.).\", \"location_match\": \"Локация совпадает: Екатеринбург.\", \"strengths\": [\"Совпавшие навыки: AWS, Terraform\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, Kubernetes\"], \"recommendation\": \"consider\"}, \"confidence\": 0.81, \"match_score\": 66}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 5 лет, навыки AWS, Docker, Kubernetes, Prometheus, Terraform, зарплата 122634-167518, локация Регионы, формат Да, занятость Полная. Вакансия: DevOps Engineer, опыт 2-5 лет, навыки AWS, Docker, GitLab CI, Terraform, зарплата 258282-368565, локация Регионы, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): AWS, Docker, Terraform. Недостающие навыки: GitLab CI.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (2-5 лет).\", \"salary_match\": \"Ожидания кандидата (122634-167518 руб.) не пересекаются с вилкой вакансии (258282-368565 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: AWS, Docker, Terraform\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: GitLab CI\"], \"recommendation\": \"hire\"}, \"confidence\": 0.87, \"match_score\": 80}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 2 лет, навыки AWS, GitLab CI, Prometheus, Terraform, зарплата 204635-291086, локация Санкт-Петербург, формат Да, занятость Частичная. Вакансия: DevOps Engineer, опыт 3-4 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 330249-427354, локация Санкт-Петербург, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): AWS, GitLab CI, Terraform. Недостающие навыки: ArgoCD, Docker, Kubernetes.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (3-4 лет).\", \"salary_match\": \"Ожидания кандидата (204635-291086 руб.) не пересекаются с вилкой вакансии (330249-427354 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: AWS, GitLab CI, Terraform\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, Docker, Kubernetes\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.78, \"match_score\": 58}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 5 лет, навыки Pandas, Power BI, Python, Tableau, зарплата 187057-238433, локация Санкт-Петербург, формат Нет, занятость Полная. Вакансия: Frontend Developer, опыт 5-7 лет, навыки CSS, Next.js, REST API, React, Redux, зарплата 215486-294238, локация Регионы, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/5): нет. Недостающие навыки: CSS, Next.js, REST API, React, Redux.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (5-7 лет).\", \"salary_match\": \"Ожидания кандидата (187057-238433 руб.) пересекаются с вилкой вакансии (215486-294238 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: CSS, Next.js, REST API, React, Redux\"], \"recommendation\": \"consider\"}, \"confidence\": 0.76, \"match_score\": 53}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 8 лет, навыки HTML, JavaScript, Next.js, React, Redux, TypeScript, Vue, зарплата 190922-222735, локация Москва, формат Да, занятость Частичная. Вакансия: Frontend Developer, опыт 4-6 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 106926-202096, локация Казань, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/6): Next.js, React, Redux, TypeScript. Недостающие навыки: CSS, REST API.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (190922-222735 руб.) пересекаются с вилкой вакансии (106926-202096 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Next.js, React, Redux, TypeScript\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: CSS, REST API\"], \"recommendation\": \"hire\"}, \"confidence\": 0.87, \"match_score\": 81}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 9 лет, навыки Django, Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 186341-269065, локация Казань, формат Да, занятость Полная. Вакансия: Python Backend Developer, опыт 5-7 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Redis, зарплата 250427-331979, локация Казань, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): Docker, FastAPI, PostgreSQL, Redis. Недостающие навыки: Kafka.\", \"experience_match\": \"Опыт кандидата 9 лет выше верхней границы (7 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (186341-269065 руб.) пересекаются с вилкой вакансии (250427-331979 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: Docker, FastAPI, PostgreSQL, Redis\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Kafka\"], \"recommendation\": \"hire\"}, \"confidence\": 0.9, \"match_score\": 87}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 6 лет, навыки MLflow, NumPy, Pandas, TensorFlow, зарплата 197012-244917, локация Регионы, формат Нет, занятость Полная. Вакансия: ML Engineer, опыт 1-4 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, Python, зарплата 172630-260867, локация Регионы, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/6): нет. Недостающие навыки: Albumentations, Docker, MLOps, OpenCV, PyTorch, Python.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (197012-244917 руб.) пересекаются с вилкой вакансии (172630-260867 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, Docker, MLOps, OpenCV, PyTorch\"], \"recommendation\": \"consider\"}, \"confidence\": 0.75, \"match_score\": 51}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 2 лет, навыки AsyncIO, Django, Docker, FastAPI, PostgreSQL, Redis, зарплата 157585-192012, локация Санкт-Петербург, формат Нет, занятость Полная. Вакансия: Python Backend Developer, опыт 5-6 лет, навыки Docker, Kafka, PostgreSQL, Redis, зарплата 220315-289200, локация Новосибирск, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): Docker, PostgreSQL, Redis. Недостающие навыки: Kafka.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (5-6 лет).\", \"salary_match\": \"Ожидания кандидата (157585-192012 руб.) не пересекаются с вилкой вакансии (220315-289200 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker, PostgreSQL, Redis\"], \"weaknesses\": [\"Недостающие навыки: Kafka\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.8, \"match_score\": 63}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 10 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Prometheus, зарплата 218974-263138, локация Санкт-Петербург, формат Да, занятость Частичная. Вакансия: DevOps Engineer, опыт 3-6 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 241248-308742, локация Регионы, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/6): AWS, Docker, GitLab CI, Kubernetes. Недостающие навыки: ArgoCD, Terraform.\", \"experience_match\": \"Опыт кандидата 10 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (218974-263138 руб.) пересекаются с вилкой вакансии (241248-308742 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: AWS, Docker, GitLab CI, Kubernetes\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, Terraform\"], \"recommendation\": \"hire\"}, \"confidence\": 0.87, \"match_score\": 81}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 4 лет, навыки AWS, Docker, Linux, Prometheus, Terraform, зарплата 248586-326393, локация Москва, формат Да, занятость Полная. Вакансия: DevOps Engineer, опыт 3-6 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Kubernetes, зарплата 197292-274260, локация Москва, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): AWS, Docker. Недостающие навыки: ArgoCD, GitLab CI, Kubernetes.\", \"experience_match\": \"Опыт кандидата 4 лет находится в диапазоне вакансии (3-6 лет).\", \"salary_match\": \"Ожидания кандидата (248586-326393 руб.) пересекаются с вилкой вакансии (197292-274260 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: AWS, Docker\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, GitLab CI, Kubernetes\"], \"recommendation\": \"consider\"}, \"confidence\": 0.84, \"match_score\": 73}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 8 лет, навыки AsyncIO, Docker, FastAPI, PostgreSQL, зарплата 248207-286811, локация Казань, формат Нет, занятость Частичная. Вакансия: Python Backend Developer, опыт 4-7 лет, навыки FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 214121-266718, локация Казань, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): FastAPI, PostgreSQL. Недостающие навыки: Kafka, Python, Redis.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (7 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (248207-286811 руб.) пересекаются с вилкой вакансии (214121-266718 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: FastAPI, PostgreSQL\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Kafka, Python, Redis\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 71}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 4 лет, навыки CSS, HTML, Next.js, React, Redux, TypeScript, Vue, зарплата 199845-264400, локация Казань, формат Да, занятость Частичная. Вакансия: Frontend Developer, опыт 6-7 лет, навыки CSS, React, Redux, TypeScript, зарплата 237541-288547, локация Казань, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/4): CSS, React, Redux, TypeScript. Недостающие навыки: нет.\", \"experience_match\": \"Опыт кандидата 4 лет ниже диапазона вакансии (6-7 лет).\", \"salary_match\": \"Ожидания кандидата (199845-264400 руб.) пересекаются с вилкой вакансии (237541-288547 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: CSS, React, Redux, TypeScript\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостаточный опыт\"], \"recommendation\": \"hire\"}, \"confidence\": 0.91, \"match_score\": 89}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 7 лет, навыки CSS, HTML, JavaScript, Next.js, React, Redux, TypeScript, зарплата 101348-155806, локация Москва, формат Гибрид, занятость Полная. Вакансия: Frontend Developer, опыт 3-5 лет, навыки CSS, Next.js, React, Redux, TypeScript, зарплата 234355-349755, локация Санкт-Петербург, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (5/5): CSS, Next.js, React, Redux, TypeScript. Недостающие навыки: нет.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (101348-155806 руб.) не пересекаются с вилкой вакансии (234355-349755 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: CSS, Next.js, React, Redux, TypeScript\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [], \"recommendation\": \"hire\"}, \"confidence\": 0.9, \"match_score\": 88}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 1 лет, навыки Airflow, Docker, MLflow, NumPy, PyTorch, TensorFlow, зарплата 213368-274777, локация Регионы, формат Гибрид, занятость Частичная. Вакансия: ML Engineer, опыт 4-7 лет, навыки MLOps, OpenCV, PyTorch, Python, зарплата 165689-264991, локация Регионы, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/4): PyTorch. Недостающие навыки: MLOps, OpenCV, Python.\", \"experience_match\": \"Опыт кандидата 1 лет ниже диапазона вакансии (4-7 лет).\", \"salary_match\": \"Ожидания кандидата (213368-274777 руб.) пересекаются с вилкой вакансии (165689-264991 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: PyTorch\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: MLOps, OpenCV, Python\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.73, \"match_score\": 45}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 6 лет, навыки Airflow, Docker, NumPy, Pandas, PyTorch, Python, TensorFlow, зарплата 191976-234416, локация Регионы, формат Нет, занятость Полная. Вакансия: ML Engineer, опыт 5-8 лет, навыки Albumentations, Docker, MLOps, OpenCV, Python, зарплата 181770-287078, локация Регионы, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): Docker, Python. Недостающие навыки: Albumentations, MLOps, OpenCV.\", \"experience_match\": \"Опыт кандидата 6 лет находится в диапазоне вакансии (5-8 лет).\", \"salary_match\": \"Ожидания кандидата (191976-234416 руб.) пересекаются с вилкой вакансии (181770-287078 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: Docker, Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV\"], \"recommendation\": \"consider\"}, \"confidence\": 0.84, \"match_score\": 73}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 5 лет, навыки AsyncIO, Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 122917-172880, локация Новосибирск, формат Гибрид, занятость Полная. Вакансия: Python Backend Developer, опыт 6-9 лет, навыки Docker, FastAPI, Kafka, Python, Redis, зарплата 268962-362275, локация Новосибирск, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): Docker, FastAPI, Python, Redis. Недостающие навыки: Kafka.\", \"experience_match\": \"Опыт кандидата 5 лет ниже диапазона вакансии (6-9 лет).\", \"salary_match\": \"Ожидания кандидата (122917-172880 руб.) не пересекаются с вилкой вакансии (268962-362275 руб.).\", \"location_match\": \"Локация совпадает: Новосибирск.\", \"strengths\": [\"Совпавшие навыки: Docker, FastAPI, Python, Redis\"], \"weaknesses\": [\"Недостающие навыки: Kafka\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.86, \"match_score\": 78}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 5 лет, навыки AsyncIO, Django, FastAPI, PostgreSQL, Python, Redis, зарплата 202522-232624, локация Санкт-Петербург, формат Да, занятость Полная. Вакансия: Python Backend Developer, опыт 3-5 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 281168-367412, локация Санкт-Петербург, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/6): FastAPI, PostgreSQL, Python, Redis. Недостающие навыки: Docker, Kafka.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (3-5 лет).\", \"salary_match\": \"Ожидания кандидата (202522-232624 руб.) не пересекаются с вилкой вакансии (281168-367412 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: FastAPI, PostgreSQL, Python, Redis\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Docker, Kafka\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 74}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 6 лет, навыки CSS, HTML, JavaScript, Next.js, Redux, TypeScript, Vue, зарплата 219995-278176, локация Регионы, формат Нет, занятость Частичная. Вакансия: Python Backend Developer, опыт 3-4 лет, навыки Docker, FastAPI, Kafka, Python, Redis, зарплата 163394-276235, локация Казань, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/5): нет. Недостающие навыки: Docker, FastAPI, Kafka, Python, Redis.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (219995-278176 руб.) пересекаются с вилкой вакансии (163394-276235 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Docker, FastAPI, Kafka, Python, Redis\"], \"recommendation\": \"consider\"}, \"confidence\": 0.75, \"match_score\": 51}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 8 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 122064-158792, локация Екатеринбург, формат Гибрид, занятость Частичная. Вакансия: DevOps Engineer, опыт 2-4 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Kubernetes, зарплата 315258-411853, локация Регионы, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): AWS, Docker, GitLab CI, Kubernetes. Недостающие навыки: ArgoCD.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (122064-158792 руб.) не пересекаются с вилкой вакансии (315258-411853 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: AWS, Docker, GitLab CI, Kubernetes\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD\"], \"recommendation\": \"consider\"}, \"confidence\": 0.87, \"match_score\": 79}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 7 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 250272-321916, локация Санкт-Петербург, формат Да, занятость Частичная. Вакансия: DevOps Engineer, опыт 5-6 лет, навыки AWS, ArgoCD, GitLab CI, Kubernetes, Terraform, зарплата 169744-279976, локация Санкт-Петербург, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): AWS, GitLab CI, Kubernetes, Terraform. Недостающие навыки: ArgoCD.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (250272-321916 руб.) пересекаются с вилкой вакансии (169744-279976 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: AWS, GitLab CI, Kubernetes, Terraform\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD\"], \"recommendation\": \"hire\"}, \"confidence\": 0.9, \"match_score\": 87}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 5 лет, навыки CSS, HTML, Next.js, Redux, Vue, зарплата 162122-247332, локация Казань, формат Гибрид, занятость Полная. Вакансия: ML Engineer, опыт 6-8 лет, навыки Albumentations, MLOps, OpenCV, Python, зарплата 155772-272469, локация Регионы, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: Albumentations, MLOps, OpenCV, Python.\", \"experience_match\": \"Опыт кандидата 5 лет ниже диапазона вакансии (6-8 лет).\", \"salary_match\": \"Ожидания кандидата (162122-247332 руб.) пересекаются с вилкой вакансии (155772-272469 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV, Python\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.74, \"match_score\": 48}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 5 лет, навыки CSS, HTML, JavaScript, Next.js, Redux, TypeScript, Vue, зарплата 173615-242892, локация Санкт-Петербург, формат Да, занятость Частичная. Вакансия: Frontend Developer, опыт 1-2 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 114360-155367, локация Санкт-Петербург, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/6): CSS, Next.js, Redux, TypeScript. Недостающие навыки: REST API, React.\", \"experience_match\": \"Опыт кандидата 5 лет выше верхней границы (2 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (173615-242892 руб.) не пересекаются с вилкой вакансии (114360-155367 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: CSS, Next.js, Redux, TypeScript\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: REST API, React\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 75}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 2 лет, навыки AsyncIO, Django, Docker, PostgreSQL, Python, Redis, зарплата 191740-269010, локация Новосибирск, формат Гибрид, занятость Частичная. Вакансия: Python Backend Developer, опыт 2-5 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, зарплата 195345-305437, локация Новосибирск, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): Docker, PostgreSQL. Недостающие навыки: FastAPI, Kafka.\", \"experience_match\": \"Опыт кандидата 2 лет находится в диапазоне вакансии (2-5 лет).\", \"salary_match\": \"Ожидания кандидата (191740-269010 руб.) пересекаются с вилкой вакансии (195345-305437 руб.).\", \"location_match\": \"Локация совпадает: Новосибирск.\", \"strengths\": [\"Совпавшие навыки: Docker, PostgreSQL\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: FastAPI, Kafka\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 75}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 2 лет, навыки AsyncIO, Django, FastAPI, PostgreSQL, зарплата 151552-211386, локация Казань, формат Гибрид, занятость Частичная. Вакансия: Python Backend Developer, опыт 2-3 лет, навыки Docker, Kafka, Python, Redis, зарплата 271985-385050, локация Казань, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: Docker, Kafka, Python, Redis.\", \"experience_match\": \"Опыт кандидата 2 лет находится в диапазоне вакансии (2-3 лет).\", \"salary_match\": \"Ожидания кандидата (151552-211386 руб.) не пересекаются с вилкой вакансии (271985-385050 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Docker, Kafka, Python, Redis\"], \"recommendation\": \"reject\"}, \"confidence\": 0.73, \"match_score\": 44}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 4 лет, навыки Docker, PyTorch, Python, TensorFlow, зарплата 171079-222611, локация Москва, формат Да, занятость Частичная. Вакансия: ML Engineer, опыт 5-6 лет, навыки Albumentations, MLOps, PyTorch, Python, зарплата 178680-296796, локация Москва, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): PyTorch, Python. Недостающие навыки: Albumentations, MLOps.\", \"experience_match\": \"Опыт кандидата 4 лет ниже диапазона вакансии (5-6 лет).\", \"salary_match\": \"Ожидания кандидата (171079-222611 руб.) пересекаются с вилкой вакансии (178680-296796 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: PyTorch, Python\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 70}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 3 лет, навыки A/B testing, Pandas, Power BI, Python, SQL, Statistics, Tableau, зарплата 151027-215308, локация Санкт-Петербург, формат Гибрид, занятость Частичная. Вакансия: Data Analyst, опыт 2-3 лет, навыки A/B testing, Product metrics, Python, SQL, зарплата 228074-322079, локация Москва, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): A/B testing, Python, SQL. Недостающие навыки: Product metrics.\", \"experience_match\": \"Опыт кандидата 3 лет находится в диапазоне вакансии (2-3 лет).\", \"salary_match\": \"Ожидания кандидата (151027-215308 руб.) не пересекаются с вилкой вакансии (228074-322079 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: A/B testing, Python, SQL\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Product metrics\"], \"recommendation\": \"consider\"}, \"confidence\": 0.86, \"match_score\": 78}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 3 лет, навыки Docker, MLflow, Pandas, PyTorch, Python, зарплата 181787-243679, локация Санкт-Петербург, формат Нет, занятость Полная. Вакансия: ML Engineer, опыт 1-3 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, Python, зарплата 255059-324371, локация Санкт-Петербург, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): Docker, PyTorch, Python. Недостающие навыки: Albumentations, MLOps, OpenCV.\", \"experience_match\": \"Опыт кандидата 3 лет находится в диапазоне вакансии (1-3 лет).\", \"salary_match\": \"Ожидания кандидата (181787-243679 руб.) не пересекаются с вилкой вакансии (255059-324371 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: Docker, PyTorch, Python\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 69}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 2 лет, навыки CSS, JavaScript, TypeScript, Vue, зарплата 120724-171550, локация Казань, формат Нет, занятость Полная. Вакансия: Frontend Developer, опыт 3-4 лет, навыки CSS, REST API, React, Redux, TypeScript, зарплата 227365-326432, локация Казань, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): CSS, TypeScript. Недостающие навыки: REST API, React, Redux.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (3-4 лет).\", \"salary_match\": \"Ожидания кандидата (120724-171550 руб.) не пересекаются с вилкой вакансии (227365-326432 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: CSS, TypeScript\"], \"weaknesses\": [\"Недостающие навыки: REST API, React, Redux\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.77, \"match_score\": 54}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 4 лет, навыки Airflow, Docker, MLflow, PyTorch, Python, TensorFlow, зарплата 152805-210610, локация Санкт-Петербург, формат Нет, занятость Частичная. Вакансия: DevOps Engineer, опыт 2-4 лет, навыки ArgoCD, GitLab CI, Kubernetes, Terraform, зарплата 211061-281598, локация Санкт-Петербург, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: ArgoCD, GitLab CI, Kubernetes, Terraform.\", \"experience_match\": \"Опыт кандидата 4 лет находится в диапазоне вакансии (2-4 лет).\", \"salary_match\": \"Ожидания кандидата (152805-210610 руб.) не пересекаются с вилкой вакансии (211061-281598 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, GitLab CI, Kubernetes, Terraform\"], \"recommendation\": \"reject\"}, \"confidence\": 0.73, \"match_score\": 44}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 2 лет, навыки Django, Docker, FastAPI, PostgreSQL, Redis, зарплата 203193-277041, локация Екатеринбург, формат Гибрид, занятость Полная. Вакансия: Python Backend Developer, опыт 5-8 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 126721-195037, локация Екатеринбург, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/6): Docker, FastAPI, PostgreSQL, Redis. Недостающие навыки: Kafka, Python.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (5-8 лет).\", \"salary_match\": \"Ожидания кандидата (203193-277041 руб.) не пересекаются с вилкой вакансии (126721-195037 руб.).\", \"location_match\": \"Локация совпадает: Екатеринбург.\", \"strengths\": [\"Совпавшие навыки: Docker, FastAPI, PostgreSQL, Redis\"], \"weaknesses\": [\"Недостающие навыки: Kafka, Python\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.79, \"match_score\": 59}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 6 лет, навыки AsyncIO, Django, FastAPI, PostgreSQL, зарплата 219742-295961, локация Новосибирск, формат Нет, занятость Полная. Вакансия: Frontend Developer, опыт 6-7 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 213574-282447, локация Екатеринбург, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/6): нет. Недостающие навыки: CSS, Next.js, REST API, React, Redux, TypeScript.\", \"experience_match\": \"Опыт кандидата 6 лет находится в диапазоне вакансии (6-7 лет).\", \"salary_match\": \"Ожидания кандидата (219742-295961 руб.) пересекаются с вилкой вакансии (213574-282447 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: CSS, Next.js, REST API, React, Redux\"], \"recommendation\": \"consider\"}, \"confidence\": 0.76, \"match_score\": 53}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 1 лет, навыки Airflow, Docker, MLflow, NumPy, TensorFlow, зарплата 164741-234743, локация Санкт-Петербург, формат Нет, занятость Полная. Вакансия: ML Engineer, опыт 4-5 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, зарплата 299089-348919, локация Санкт-Петербург, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/5): Docker. Недостающие навыки: Albumentations, MLOps, OpenCV, PyTorch.\", \"experience_match\": \"Опыт кандидата 1 лет ниже диапазона вакансии (4-5 лет).\", \"salary_match\": \"Ожидания кандидата (164741-234743 руб.) не пересекаются с вилкой вакансии (299089-348919 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: Docker\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV, PyTorch\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.7, \"match_score\": 37}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 2 лет, навыки AsyncIO, Docker, PostgreSQL, Python, Redis, зарплата 177368-229739, локация Екатеринбург, формат Нет, занятость Частичная. Вакансия: DevOps Engineer, опыт 2-5 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Terraform, зарплата 223821-338390, локация Санкт-Петербург, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/5): Docker. Недостающие навыки: AWS, GitLab CI, Kubernetes, Terraform.\", \"experience_match\": \"Опыт кандидата 2 лет находится в диапазоне вакансии (2-5 лет).\", \"salary_match\": \"Ожидания кандидата (177368-229739 руб.) пересекаются с вилкой вакансии (223821-338390 руб.).\", \"location_match\": \"Локация не совпадает (Екатеринбург vs Санкт-Петербург).\", \"strengths\": [\"Совпавшие навыки: Docker\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: AWS, GitLab CI, Kubernetes, Terraform\", \"Несовпадение по локации/формату\"], \"recommendation\": \"consider\"}, \"confidence\": 0.78, \"match_score\": 58}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 3 лет, навыки CSS, HTML, JavaScript, Next.js, TypeScript, Vue, зарплата 185307-263243, локация Санкт-Петербург, формат Да, занятость Частичная. Вакансия: Frontend Developer, опыт 4-6 лет, навыки CSS, Next.js, REST API, React, Redux, зарплата 170020-228888, локация Санкт-Петербург, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): CSS, Next.js. Недостающие навыки: REST API, React, Redux.\", \"experience_match\": \"Опыт кандидата 3 лет ниже диапазона вакансии (4-6 лет).\", \"salary_match\": \"Ожидания кандидата (185307-263243 руб.) пересекаются с вилкой вакансии (170020-228888 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: CSS, Next.js\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: REST API, React, Redux\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.81, \"match_score\": 66}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 4 лет, навыки HTML, JavaScript, Next.js, Redux, Vue, зарплата 133844-204460, локация Санкт-Петербург, формат Да, занятость Частичная. Вакансия: Python Backend Developer, опыт 6-8 лет, навыки FastAPI, Kafka, Python, Redis, зарплата 120811-168665, локация Казань, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: FastAPI, Kafka, Python, Redis.\", \"experience_match\": \"Опыт кандидата 4 лет ниже диапазона вакансии (6-8 лет).\", \"salary_match\": \"Ожидания кандидата (133844-204460 руб.) пересекаются с вилкой вакансии (120811-168665 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: FastAPI, Kafka, Python, Redis\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.73, \"match_score\": 44}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 7 лет, навыки CSS, HTML, JavaScript, Next.js, React, TypeScript, Vue, зарплата 74907-145521, локация Казань, формат Да, занятость Частичная. Вакансия: Frontend Developer, опыт 6-7 лет, навыки CSS, REST API, React, Redux, TypeScript, зарплата 195440-266738, локация Регионы, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): CSS, React, TypeScript. Недостающие навыки: REST API, Redux.\", \"experience_match\": \"Опыт кандидата 7 лет находится в диапазоне вакансии (6-7 лет).\", \"salary_match\": \"Ожидания кандидата (74907-145521 руб.) не пересекаются с вилкой вакансии (195440-266738 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: CSS, React, TypeScript\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: REST API, Redux\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 71}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 8 лет, навыки CSS, HTML, Next.js, React, Vue, зарплата 111148-198383, локация Казань, формат Нет, занятость Полная. Вакансия: Frontend Developer, опыт 3-5 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 146563-238998, локация Казань, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): CSS, Next.js, React. Недостающие навыки: REST API, Redux, TypeScript.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (111148-198383 руб.) пересекаются с вилкой вакансии (146563-238998 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: CSS, Next.js, React\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: REST API, Redux, TypeScript\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 76}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 6 лет, навыки A/B testing, Python, SQL, Statistics, Tableau, зарплата 141546-202865, локация Москва, формат Гибрид, занятость Полная. Вакансия: Python Backend Developer, опыт 1-4 лет, навыки Docker, FastAPI, Kafka, Python, Redis, зарплата 237190-290403, локация Санкт-Петербург, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/5): Python. Недостающие навыки: Docker, FastAPI, Kafka, Redis.\", \"experience_match\": \"Опыт кандидата 6 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (141546-202865 руб.) не пересекаются с вилкой вакансии (237190-290403 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Python\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Docker, FastAPI, Kafka, Redis\"], \"recommendation\": \"consider\"}, \"confidence\": 0.77, \"match_score\": 54}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 10 лет, навыки AWS, GitLab CI, Kubernetes, Prometheus, зарплата 267556-353912, локация Москва, формат Гибрид, занятость Полная. Вакансия: DevOps Engineer, опыт 6-7 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 289083-397901, локация Новосибирск, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): AWS, GitLab CI, Kubernetes. Недостающие навыки: ArgoCD, Docker, Terraform.\", \"experience_match\": \"Опыт кандидата 10 лет выше верхней границы (7 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (267556-353912 руб.) пересекаются с вилкой вакансии (289083-397901 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: AWS, GitLab CI, Kubernetes\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, Docker, Terraform\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 74}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 9 лет, навыки AsyncIO, Django, Docker, PostgreSQL, зарплата 138100-186354, локация Москва, формат Да, занятость Полная. Вакансия: DevOps Engineer, опыт 3-5 лет, навыки ArgoCD, GitLab CI, Kubernetes, Terraform, зарплата 241275-314017, локация Санкт-Петербург, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: ArgoCD, GitLab CI, Kubernetes, Terraform.\", \"experience_match\": \"Опыт кандидата 9 лет выше верхней границы (5 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (138100-186354 руб.) не пересекаются с вилкой вакансии (241275-314017 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, GitLab CI, Kubernetes, Terraform\"], \"recommendation\": \"reject\"}, \"confidence\": 0.73, \"match_score\": 45}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 2 лет, навыки Docker, MLflow, PyTorch, Python, TensorFlow, зарплата 253768-287295, локация Москва, формат Гибрид, занятость Полная. Вакансия: ML Engineer, опыт 1-3 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, Python, зарплата 146911-238948, локация Москва, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): Docker, PyTorch, Python. Недостающие навыки: Albumentations, MLOps, OpenCV.\", \"experience_match\": \"Опыт кандидата 2 лет находится в диапазоне вакансии (1-3 лет).\", \"salary_match\": \"Ожидания кандидата (253768-287295 руб.) не пересекаются с вилкой вакансии (146911-238948 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: Docker, PyTorch, Python\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV\"], \"recommendation\": \"consider\"}, \"confidence\": 0.82, \"match_score\": 67}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 4 лет, навыки Airflow, Docker, Pandas, Python, зарплата 220551-302141, локация Регионы, формат Нет, занятость Частичная. Вакансия: Frontend Developer, опыт 2-5 лет, навыки Next.js, REST API, Redux, TypeScript, зарплата 199816-257903, локация Санкт-Петербург, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: Next.js, REST API, Redux, TypeScript.\", \"experience_match\": \"Опыт кандидата 4 лет находится в диапазоне вакансии (2-5 лет).\", \"salary_match\": \"Ожидания кандидата (220551-302141 руб.) пересекаются с вилкой вакансии (199816-257903 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Next.js, REST API, Redux, TypeScript\"], \"recommendation\": \"consider\"}, \"confidence\": 0.76, \"match_score\": 53}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 8 лет, навыки Airflow, MLflow, NumPy, Pandas, PyTorch, Python, TensorFlow, зарплата 284073-345186, локация Санкт-Петербург, формат Нет, занятость Полная. Вакансия: ML Engineer, опыт 2-3 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, Python, зарплата 265377-368125, локация Санкт-Петербург, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/6): PyTorch, Python. Недостающие навыки: Albumentations, Docker, MLOps, OpenCV.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (3 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (284073-345186 руб.) пересекаются с вилкой вакансии (265377-368125 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: PyTorch, Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, Docker, MLOps, OpenCV\"], \"recommendation\": \"consider\"}, \"confidence\": 0.82, \"match_score\": 68}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 7 лет, навыки Airflow, Docker, MLflow, Pandas, PyTorch, Python, TensorFlow, зарплата 247938-320987, локация Санкт-Петербург, формат Нет, занятость Полная. Вакансия: Python Backend Developer, опыт 2-3 лет, навыки Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 205211-319925, локация Регионы, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): Docker, Python. Недостающие навыки: FastAPI, PostgreSQL, Redis.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (3 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (247938-320987 руб.) пересекаются с вилкой вакансии (205211-319925 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker, Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: FastAPI, PostgreSQL, Redis\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 69}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 6 лет, навыки Airflow, Docker, NumPy, PyTorch, TensorFlow, зарплата 275419-350444, локация Регионы, формат Гибрид, занятость Частичная. Вакансия: ML Engineer, опыт 5-7 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, Python, зарплата 184688-229502, локация Регионы, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/6): Docker, PyTorch. Недостающие навыки: Albumentations, MLOps, OpenCV, Python.\", \"experience_match\": \"Опыт кандидата 6 лет находится в диапазоне вакансии (5-7 лет).\", \"salary_match\": \"Ожидания кандидата (275419-350444 руб.) не пересекаются с вилкой вакансии (184688-229502 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: Docker, PyTorch\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV, Python\"], \"recommendation\": \"consider\"}, \"confidence\": 0.79, \"match_score\": 61}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 8 лет, навыки HTML, Next.js, React, Redux, TypeScript, Vue, зарплата 130866-191087, локация Казань, формат Нет, занятость Полная. Вакансия: Frontend Developer, опыт 5-6 лет, навыки CSS, Next.js, REST API, React, Redux, TypeScript, зарплата 166578-266814, локация Казань, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/6): Next.js, React, Redux, TypeScript. Недостающие навыки: CSS, REST API.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (6 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (130866-191087 руб.) пересекаются с вилкой вакансии (166578-266814 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: Next.js, React, Redux, TypeScript\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: CSS, REST API\"], \"recommendation\": \"hire\"}, \"confidence\": 0.88, \"match_score\": 83}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 5 лет, навыки AWS, GitLab CI, Linux, Prometheus, зарплата 299746-353819, локация Москва, формат Да, занятость Частичная. Вакансия: DevOps Engineer, опыт 4-5 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Terraform, зарплата 292728-379740, локация Москва, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): AWS, GitLab CI. Недостающие навыки: Docker, Kubernetes, Terraform.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (4-5 лет).\", \"salary_match\": \"Ожидания кандидата (299746-353819 руб.) пересекаются с вилкой вакансии (292728-379740 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Совпавшие навыки: AWS, GitLab CI\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Docker, Kubernetes, Terraform\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 71}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 3 лет, навыки CSS, HTML, Next.js, TypeScript, Vue, зарплата 153542-241222, локация Казань, формат Гибрид, занятость Частичная. Вакансия: Frontend Developer, опыт 6-7 лет, навыки CSS, Next.js, REST API, React, TypeScript, зарплата 208538-302441, локация Казань, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): CSS, Next.js, TypeScript. Недостающие навыки: REST API, React.\", \"experience_match\": \"Опыт кандидата 3 лет ниже диапазона вакансии (6-7 лет).\", \"salary_match\": \"Ожидания кандидата (153542-241222 руб.) пересекаются с вилкой вакансии (208538-302441 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: CSS, Next.js, TypeScript\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: REST API, React\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.82, \"match_score\": 67}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 6 лет, навыки AsyncIO, Django, Docker, PostgreSQL, Python, зарплата 234572-324403, локация Санкт-Петербург, формат Гибрид, занятость Полная. Вакансия: Python Backend Developer, опыт 5-6 лет, навыки Kafka, PostgreSQL, Python, Redis, зарплата 230382-302166, локация Москва, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): PostgreSQL, Python. Недостающие навыки: Kafka, Redis.\", \"experience_match\": \"Опыт кандидата 6 лет находится в диапазоне вакансии (5-6 лет).\", \"salary_match\": \"Ожидания кандидата (234572-324403 руб.) пересекаются с вилкой вакансии (230382-302166 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: PostgreSQL, Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Kafka, Redis\"], \"recommendation\": \"consider\"}, \"confidence\": 0.86, \"match_score\": 77}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 5 лет, навыки AsyncIO, Django, Docker, PostgreSQL, зарплата 152531-195637, локация Новосибирск, формат Да, занятость Полная. Вакансия: Python Backend Developer, опыт 3-5 лет, навыки FastAPI, Kafka, Python, Redis, зарплата 143956-186058, локация Новосибирск, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: FastAPI, Kafka, Python, Redis.\", \"experience_match\": \"Опыт кандидата 5 лет находится в диапазоне вакансии (3-5 лет).\", \"salary_match\": \"Ожидания кандидата (152531-195637 руб.) пересекаются с вилкой вакансии (143956-186058 руб.).\", \"location_match\": \"Локация совпадает: Новосибирск.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: FastAPI, Kafka, Python, Redis\"], \"recommendation\": \"consider\"}, \"confidence\": 0.77, \"match_score\": 55}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 1 лет, навыки Airflow, Docker, NumPy, PyTorch, Python, зарплата 245789-309326, локация Казань, формат Да, занятость Полная. Вакансия: Python Backend Developer, опыт 5-6 лет, навыки Docker, Kafka, PostgreSQL, Python, зарплата 179936-248287, локация Москва, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): Docker, Python. Недостающие навыки: Kafka, PostgreSQL.\", \"experience_match\": \"Опыт кандидата 1 лет ниже диапазона вакансии (5-6 лет).\", \"salary_match\": \"Ожидания кандидата (245789-309326 руб.) пересекаются с вилкой вакансии (179936-248287 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker, Python\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Kafka, PostgreSQL\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.78, \"match_score\": 57}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 5 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 181645-235836, локация Регионы, формат Нет, занятость Полная. Вакансия: Data Analyst, опыт 6-7 лет, навыки A/B testing, Product metrics, Python, SQL, Tableau, зарплата 244969-329513, локация Регионы, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/5): нет. Недостающие навыки: A/B testing, Product metrics, Python, SQL, Tableau.\", \"experience_match\": \"Опыт кандидата 5 лет ниже диапазона вакансии (6-7 лет).\", \"salary_match\": \"Ожидания кандидата (181645-235836 руб.) не пересекаются с вилкой вакансии (244969-329513 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [], \"weaknesses\": [\"Недостающие навыки: A/B testing, Product metrics, Python, SQL, Tableau\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.72, \"match_score\": 42}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 5 лет, навыки Docker, MLflow, NumPy, TensorFlow, зарплата 187724-243268, локация Регионы, формат Да, занятость Частичная. Вакансия: ML Engineer, опыт 1-3 лет, навыки Docker, OpenCV, PyTorch, Python, зарплата 276773-371988, локация Регионы, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/4): Docker. Недостающие навыки: OpenCV, PyTorch, Python.\", \"experience_match\": \"Опыт кандидата 5 лет выше верхней границы (3 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (187724-243268 руб.) не пересекаются с вилкой вакансии (276773-371988 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: Docker\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: OpenCV, PyTorch, Python\"], \"recommendation\": \"consider\"}, \"confidence\": 0.77, \"match_score\": 54}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 5 лет, навыки AsyncIO, Django, Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 270685-354291, локация Екатеринбург, формат Нет, занятость Частичная. Вакансия: ML Engineer, опыт 2-4 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, Python, зарплата 246370-302256, локация Казань, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/6): Docker, Python. Недостающие навыки: Albumentations, MLOps, OpenCV, PyTorch.\", \"experience_match\": \"Опыт кандидата 5 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (270685-354291 руб.) пересекаются с вилкой вакансии (246370-302256 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker, Python\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV, PyTorch\"], \"recommendation\": \"consider\"}, \"confidence\": 0.82, \"match_score\": 68}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 4 лет, навыки A/B testing, Pandas, Power BI, Python, SQL, Statistics, Tableau, зарплата 117661-168146, локация Регионы, формат Гибрид, занятость Полная. Вакансия: Python Backend Developer, опыт 3-6 лет, навыки Docker, FastAPI, Kafka, Redis, зарплата 193706-278974, локация Санкт-Петербург, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: Docker, FastAPI, Kafka, Redis.\", \"experience_match\": \"Опыт кандидата 4 лет находится в диапазоне вакансии (3-6 лет).\", \"salary_match\": \"Ожидания кандидата (117661-168146 руб.) не пересекаются с вилкой вакансии (193706-278974 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: Docker, FastAPI, Kafka, Redis\"], \"recommendation\": \"reject\"}, \"confidence\": 0.73, \"match_score\": 44}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 1 лет, навыки CSS, HTML, React, Vue, зарплата 74947-105506, локация Регионы, формат Да, занятость Частичная. Вакансия: Frontend Developer, опыт 5-7 лет, навыки CSS, Next.js, React, Redux, TypeScript, зарплата 159042-241297, локация Регионы, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/5): CSS, React. Недостающие навыки: Next.js, Redux, TypeScript.\", \"experience_match\": \"Опыт кандидата 1 лет ниже диапазона вакансии (5-7 лет).\", \"salary_match\": \"Ожидания кандидата (74947-105506 руб.) не пересекаются с вилкой вакансии (159042-241297 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: CSS, React\"], \"weaknesses\": [\"Недостающие навыки: Next.js, Redux, TypeScript\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.73, \"match_score\": 44}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 2 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 265139-328645, локация Регионы, формат Да, занятость Частичная. Вакансия: DevOps Engineer, опыт 7-9 лет, навыки AWS, ArgoCD, Docker, Kubernetes, Terraform, зарплата 295836-410634, локация Регионы, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (4/5): AWS, Docker, Kubernetes, Terraform. Недостающие навыки: ArgoCD.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (7-9 лет).\", \"salary_match\": \"Ожидания кандидата (265139-328645 руб.) пересекаются с вилкой вакансии (295836-410634 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: AWS, Docker, Kubernetes, Terraform\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 71}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 1 лет, навыки Airflow, MLflow, Pandas, PyTorch, зарплата 276691-307037, локация Регионы, формат Да, занятость Частичная. Вакансия: ML Engineer, опыт 2-4 лет, навыки Docker, MLOps, OpenCV, PyTorch, зарплата 163878-222926, локация Регионы, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/4): PyTorch. Недостающие навыки: Docker, MLOps, OpenCV.\", \"experience_match\": \"Опыт кандидата 1 лет ниже диапазона вакансии (2-4 лет).\", \"salary_match\": \"Ожидания кандидата (276691-307037 руб.) не пересекаются с вилкой вакансии (163878-222926 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: PyTorch\"], \"weaknesses\": [\"Недостающие навыки: Docker, MLOps, OpenCV\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.72, \"match_score\": 43}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 2 лет, навыки Airflow, Docker, NumPy, TensorFlow, зарплата 132575-190523, локация Регионы, формат Нет, занятость Полная. Вакансия: ML Engineer, опыт 4-5 лет, навыки Docker, MLOps, OpenCV, PyTorch, Python, зарплата 242738-306197, локация Регионы, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/5): Docker. Недостающие навыки: MLOps, OpenCV, PyTorch, Python.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (4-5 лет).\", \"salary_match\": \"Ожидания кандидата (132575-190523 руб.) не пересекаются с вилкой вакансии (242738-306197 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: Docker\"], \"weaknesses\": [\"Недостающие навыки: MLOps, OpenCV, PyTorch, Python\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.72, \"match_score\": 43}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 10 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 266313-339849, локация Екатеринбург, формат Да, занятость Полная. Вакансия: Python Backend Developer, опыт 4-7 лет, навыки Docker, FastAPI, PostgreSQL, Python, Redis, зарплата 192626-272425, локация Новосибирск, формат Нет, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/5): Docker. Недостающие навыки: FastAPI, PostgreSQL, Python, Redis.\", \"experience_match\": \"Опыт кандидата 10 лет выше верхней границы (7 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (266313-339849 руб.) пересекаются с вилкой вакансии (192626-272425 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: FastAPI, PostgreSQL, Python, Redis\"], \"recommendation\": \"consider\"}, \"confidence\": 0.8, \"match_score\": 62}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 3 лет, навыки Airflow, Docker, MLflow, NumPy, зарплата 288555-364376, локация Москва, формат Да, занятость Частичная. Вакансия: DevOps Engineer, опыт 5-7 лет, навыки ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 151723-219837, локация Санкт-Петербург, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/5): Docker. Недостающие навыки: ArgoCD, GitLab CI, Kubernetes, Terraform.\", \"experience_match\": \"Опыт кандидата 3 лет ниже диапазона вакансии (5-7 лет).\", \"salary_match\": \"Ожидания кандидата (288555-364376 руб.) не пересекаются с вилкой вакансии (151723-219837 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: Docker\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, GitLab CI, Kubernetes, Terraform\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.72, \"match_score\": 43}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 9 лет, навыки AWS, Docker, Linux, Terraform, зарплата 302790-356266, локация Екатеринбург, формат Нет, занятость Частичная. Вакансия: DevOps Engineer, опыт 7-10 лет, навыки AWS, ArgoCD, Docker, GitLab CI, Kubernetes, Terraform, зарплата 166601-276649, локация Екатеринбург, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): AWS, Docker, Terraform. Недостающие навыки: ArgoCD, GitLab CI, Kubernetes.\", \"experience_match\": \"Опыт кандидата 9 лет находится в диапазоне вакансии (7-10 лет).\", \"salary_match\": \"Ожидания кандидата (302790-356266 руб.) не пересекаются с вилкой вакансии (166601-276649 руб.).\", \"location_match\": \"Локация совпадает: Екатеринбург.\", \"strengths\": [\"Совпавшие навыки: AWS, Docker, Terraform\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD, GitLab CI, Kubernetes\"], \"recommendation\": \"consider\"}, \"confidence\": 0.83, \"match_score\": 69}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 8 лет, навыки CSS, HTML, Next.js, TypeScript, Vue, зарплата 123252-171722, локация Санкт-Петербург, формат Да, занятость Полная. Вакансия: Frontend Developer, опыт 1-3 лет, навыки CSS, REST API, React, TypeScript, зарплата 211805-310109, локация Казань, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (2/4): CSS, TypeScript. Недостающие навыки: REST API, React.\", \"experience_match\": \"Опыт кандидата 8 лет выше верхней границы (3 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (123252-171722 руб.) не пересекаются с вилкой вакансии (211805-310109 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: CSS, TypeScript\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: REST API, React\"], \"recommendation\": \"consider\"}, \"confidence\": 0.81, \"match_score\": 66}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 2 лет, навыки Airflow, Docker, NumPy, Pandas, PyTorch, Python, TensorFlow, зарплата 118849-182361, локация Регионы, формат Гибрид, занятость Частичная. Вакансия: ML Engineer, опыт 4-5 лет, навыки Albumentations, Docker, MLOps, OpenCV, PyTorch, Python, зарплата 206339-323261, локация Регионы, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): Docker, PyTorch, Python. Недостающие навыки: Albumentations, MLOps, OpenCV.\", \"experience_match\": \"Опыт кандидата 2 лет ниже диапазона вакансии (4-5 лет).\", \"salary_match\": \"Ожидания кандидата (118849-182361 руб.) не пересекаются с вилкой вакансии (206339-323261 руб.).\", \"location_match\": \"Локация совпадает: Регионы.\", \"strengths\": [\"Совпавшие навыки: Docker, PyTorch, Python\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, MLOps, OpenCV\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.77, \"match_score\": 56}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Data Analyst, опыт 3 лет, навыки A/B testing, Pandas, Power BI, Python, SQL, Statistics, Tableau, зарплата 206469-255545, локация Санкт-Петербург, формат Гибрид, занятость Частичная. Вакансия: ML Engineer, опыт 4-7 лет, навыки Albumentations, Docker, MLOps, PyTorch, Python, зарплата 147964-215458, локация Санкт-Петербург, формат Да, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (1/5): Python. Недостающие навыки: Albumentations, Docker, MLOps, PyTorch.\", \"experience_match\": \"Опыт кандидата 3 лет ниже диапазона вакансии (4-7 лет).\", \"salary_match\": \"Ожидания кандидата (206469-255545 руб.) пересекаются с вилкой вакансии (147964-215458 руб.).\", \"location_match\": \"Локация совпадает: Санкт-Петербург.\", \"strengths\": [\"Совпавшие навыки: Python\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: Albumentations, Docker, MLOps, PyTorch\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.77, \"match_score\": 55}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 4 лет, навыки AsyncIO, Django, Docker, PostgreSQL, Python, зарплата 108719-142627, локация Екатеринбург, формат Нет, занятость Полная. Вакансия: Python Backend Developer, опыт 4-5 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, Redis, зарплата 163495-243814, локация Екатеринбург, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/6): Docker, PostgreSQL, Python. Недостающие навыки: FastAPI, Kafka, Redis.\", \"experience_match\": \"Опыт кандидата 4 лет находится в диапазоне вакансии (4-5 лет).\", \"salary_match\": \"Ожидания кандидата (108719-142627 руб.) не пересекаются с вилкой вакансии (163495-243814 руб.).\", \"location_match\": \"Локация совпадает: Екатеринбург.\", \"strengths\": [\"Совпавшие навыки: Docker, PostgreSQL, Python\", \"Опыт соответствует минимальным требованиям\"], \"weaknesses\": [\"Недостающие навыки: FastAPI, Kafka, Redis\"], \"recommendation\": \"consider\"}, \"confidence\": 0.82, \"match_score\": 67}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 5 лет, навыки AsyncIO, Django, Docker, FastAPI, Python, Redis, зарплата 166038-215644, локация Казань, формат Гибрид, занятость Частичная. Вакансия: Python Backend Developer, опыт 6-7 лет, навыки Docker, FastAPI, Kafka, PostgreSQL, Python, зарплата 246648-325123, локация Казань, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/5): Docker, FastAPI, Python. Недостающие навыки: Kafka, PostgreSQL.\", \"experience_match\": \"Опыт кандидата 5 лет ниже диапазона вакансии (6-7 лет).\", \"salary_match\": \"Ожидания кандидата (166038-215644 руб.) не пересекаются с вилкой вакансии (246648-325123 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: Docker, FastAPI, Python\"], \"weaknesses\": [\"Недостающие навыки: Kafka, PostgreSQL\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.82, \"match_score\": 67}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Frontend Developer, опыт 3 лет, навыки CSS, HTML, Next.js, React, Redux, TypeScript, зарплата 75106-158749, локация Казань, формат Нет, занятость Частичная. Вакансия: Frontend Developer, опыт 4-6 лет, навыки Next.js, REST API, React, TypeScript, зарплата 153008-220350, локация Казань, формат Да, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): Next.js, React, TypeScript. Недостающие навыки: REST API.\", \"experience_match\": \"Опыт кандидата 3 лет ниже диапазона вакансии (4-6 лет).\", \"salary_match\": \"Ожидания кандидата (75106-158749 руб.) пересекаются с вилкой вакансии (153008-220350 руб.).\", \"location_match\": \"Локация совпадает: Казань.\", \"strengths\": [\"Совпавшие навыки: Next.js, React, TypeScript\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: REST API\", \"Недостаточный опыт\"], \"recommendation\": \"hire\"}, \"confidence\": 0.88, \"match_score\": 82}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 4 лет, навыки AWS, Docker, GitLab CI, Kubernetes, Linux, Prometheus, Terraform, зарплата 181207-224293, локация Санкт-Петербург, формат Да, занятость Полная. Вакансия: DevOps Engineer, опыт 5-6 лет, навыки AWS, ArgoCD, GitLab CI, Terraform, зарплата 329662-418659, локация Москва, формат Гибрид, занятость Полная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): AWS, GitLab CI, Terraform. Недостающие навыки: ArgoCD.\", \"experience_match\": \"Опыт кандидата 4 лет ниже диапазона вакансии (5-6 лет).\", \"salary_match\": \"Ожидания кандидата (181207-224293 руб.) не пересекаются с вилкой вакансии (329662-418659 руб.).\", \"location_match\": \"Локация отличается, но удаленный/гибридный формат совместим.\", \"strengths\": [\"Совпавшие навыки: AWS, GitLab CI, Terraform\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD\", \"Недостаточный опыт\"], \"recommendation\": \"consider\"}, \"confidence\": 0.85, \"match_score\": 75}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: DevOps Engineer, опыт 7 лет, навыки AWS, Docker, GitLab CI, Linux, Prometheus, Terraform, зарплата 213505-256809, локация Новосибирск, формат Гибрид, занятость Частичная. Вакансия: DevOps Engineer, опыт 2-4 лет, навыки AWS, ArgoCD, Docker, GitLab CI, зарплата 189479-257398, локация Новосибирск, формат Нет, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (3/4): AWS, Docker, GitLab CI. Недостающие навыки: ArgoCD.\", \"experience_match\": \"Опыт кандидата 7 лет выше верхней границы (4 лет), но это допустимо.\", \"salary_match\": \"Ожидания кандидата (213505-256809 руб.) пересекаются с вилкой вакансии (189479-257398 руб.).\", \"location_match\": \"Локация совпадает: Новосибирск.\", \"strengths\": [\"Совпавшие навыки: AWS, Docker, GitLab CI\", \"Опыт соответствует минимальным требованиям\", \"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: ArgoCD\"], \"recommendation\": \"hire\"}, \"confidence\": 0.9, \"match_score\": 87}"}]}
{"messages": [{"role": "user", "content": "Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: ML Engineer, опыт 4 лет, навыки MLflow, NumPy, Pandas, PyTorch, TensorFlow, зарплата 211736-272516, локация Москва, формат Гибрид, занятость Частичная. Вакансия: Frontend Developer, опыт 5-7 лет, навыки REST API, React, Redux, TypeScript, зарплата 259182-305268, локация Москва, формат Гибрид, занятость Частичная."}, {"role": "assistant", "content": "{\"analysis\": {\"skills_match\": \"Совпавшие навыки (0/4): нет. Недостающие навыки: REST API, React, Redux, TypeScript.\", \"experience_match\": \"Опыт кандидата 4 лет ниже диапазона вакансии (5-7 лет).\", \"salary_match\": \"Ожидания кандидата (211736-272516 руб.) пересекаются с вилкой вакансии (259182-305268 руб.).\", \"location_match\": \"Локация совпадает: Москва.\", \"strengths\": [\"Зарплатные ожидания в рабочем диапазоне\"], \"weaknesses\": [\"Недостающие навыки: REST API, React, Redux, TypeScript\", \"Недостаточный опыт\"], \"recommendation\": \"reject\"}, \"confidence\": 0.75, \"match_score\": 49}"}]}

# Fully in-notebook dataset generation (no file paths)
from collections import defaultdict

RANDOM_SEED = 42
TARGET_SAMPLES = 600
TRAIN_RATIO = 0.9
random.seed(RANDOM_SEED)

PROFILES = {
    "Python Backend Developer": {
        "candidate_skills": ["Python", "FastAPI", "Django", "PostgreSQL", "Redis", "Docker", "AsyncIO"],
        "vacancy_skills": ["Python", "FastAPI", "PostgreSQL", "Docker", "Kafka", "Redis"],
        "candidate_exp": (2, 9),
        "vacancy_exp_min": (2, 6),
        "vacancy_exp_max_delta": (1, 3),
        "candidate_salary": (90000, 320000),
        "vacancy_salary": (120000, 340000),
        "locations": ["Москва", "Санкт-Петербург", "Казань", "Новосибирск", "Екатеринбург"],
    },
    "ML Engineer": {
        "candidate_skills": ["Python", "PyTorch", "TensorFlow", "Pandas", "NumPy", "MLflow", "Docker", "Airflow"],
        "vacancy_skills": ["Python", "PyTorch", "Albumentations", "OpenCV", "MLOps", "Docker"],
        "candidate_exp": (1, 8),
        "vacancy_exp_min": (1, 5),
        "vacancy_exp_max_delta": (1, 3),
        "candidate_salary": (110000, 340000),
        "vacancy_salary": (140000, 360000),
        "locations": ["Москва", "Санкт-Петербург", "Казань", "Регионы"],
    },
    "Frontend Developer": {
        "candidate_skills": ["JavaScript", "TypeScript", "React", "Redux", "CSS", "HTML", "Next.js", "Vue"],
        "vacancy_skills": ["React", "TypeScript", "Redux", "Next.js", "CSS", "REST API"],
        "candidate_exp": (1, 8),
        "vacancy_exp_min": (1, 6),
        "vacancy_exp_max_delta": (1, 2),
        "candidate_salary": (70000, 260000),
        "vacancy_salary": (100000, 290000),
        "locations": ["Москва", "Санкт-Петербург", "Казань", "Регионы"],
    },
    "DevOps Engineer": {
        "candidate_skills": ["Docker", "Kubernetes", "Terraform", "AWS", "GitLab CI", "Linux", "Prometheus"],
        "vacancy_skills": ["Docker", "Kubernetes", "Terraform", "AWS", "GitLab CI", "ArgoCD"],
        "candidate_exp": (2, 10),
        "vacancy_exp_min": (2, 7),
        "vacancy_exp_max_delta": (1, 3),
        "candidate_salary": (120000, 360000),
        "vacancy_salary": (150000, 390000),
        "locations": ["Москва", "Санкт-Петербург", "Новосибирск", "Екатеринбург", "Регионы"],
    },
    "Data Analyst": {
        "candidate_skills": ["SQL", "Python", "Pandas", "Power BI", "Tableau", "A/B testing", "Statistics"],
        "vacancy_skills": ["SQL", "Python", "A/B testing", "Product metrics", "Tableau"],
        "candidate_exp": (1, 9),
        "vacancy_exp_min": (1, 6),
        "vacancy_exp_max_delta": (1, 3),
        "candidate_salary": (80000, 280000),
        "vacancy_salary": (110000, 300000),
        "locations": ["Москва", "Санкт-Петербург", "Казань", "Регионы"],
    },
}

REMOTE_OPTIONS = ["Да", "Нет", "Гибрид"]
EMPLOYMENT_OPTIONS = ["Полная", "Частичная"]


def choose_skills(base_skills, min_n=4, max_n=7):
    k = random.randint(min_n, min(max_n, len(base_skills)))
    return sorted(random.sample(base_skills, k=k))


def overlap(a, b):
    return sorted(set(a) & set(b))


def build_pair(profile_name, hard_negative=False):
    p = PROFILES[profile_name]
    candidate_title = profile_name
    vacancy_title = profile_name

    c_skills = choose_skills(p["candidate_skills"], 4, 7)
    v_skills = choose_skills(p["vacancy_skills"], 4, 6)

    if hard_negative:
        other_profile = random.choice([x for x in PROFILES if x != profile_name])
        v_skills = choose_skills(PROFILES[other_profile]["vacancy_skills"], 4, 6)
        vacancy_title = other_profile

    c_exp = random.randint(*p["candidate_exp"])
    v_min = random.randint(*p["vacancy_exp_min"])
    v_max = v_min + random.randint(*p["vacancy_exp_max_delta"])

    c_salary_min = random.randint(p["candidate_salary"][0], p["candidate_salary"][1] - 40000)
    c_salary_max = c_salary_min + random.randint(30000, 90000)
    v_salary_min = random.randint(p["vacancy_salary"][0], p["vacancy_salary"][1] - 50000)
    v_salary_max = v_salary_min + random.randint(40000, 120000)

    c_location = random.choice(p["locations"])
    if hard_negative and random.random() < 0.7:
        v_location = random.choice([x for x in p["locations"] if x != c_location]) if len(p["locations"]) > 1 else c_location
    else:
        v_location = c_location if random.random() < 0.7 else random.choice(p["locations"])

    c_remote = random.choice(REMOTE_OPTIONS)
    v_remote = random.choice(REMOTE_OPTIONS)
    c_employment = random.choice(EMPLOYMENT_OPTIONS)
    v_employment = random.choice(EMPLOYMENT_OPTIONS)

    common = overlap(c_skills, v_skills)
    missing = sorted(set(v_skills) - set(c_skills))
    required_count = len(v_skills) if v_skills else 1
    skill_ratio = len(common) / required_count

    if c_exp < v_min:
        exp_score = max(0.0, c_exp / max(1, v_min))
        exp_text = f"Опыт кандидата {c_exp} лет ниже диапазона вакансии ({v_min}-{v_max} лет)."
    elif c_exp > v_max:
        exp_score = 0.95
        exp_text = f"Опыт кандидата {c_exp} лет выше верхней границы ({v_max} лет), но это допустимо."
    else:
        exp_score = 1.0
        exp_text = f"Опыт кандидата {c_exp} лет находится в диапазоне вакансии ({v_min}-{v_max} лет)."

    intersect = not (c_salary_max < v_salary_min or c_salary_min > v_salary_max)
    salary_score = 1.0 if intersect else 0.45
    salary_text = (
        f"Ожидания кандидата ({c_salary_min}-{c_salary_max} руб.) "
        f"{'пересекаются' if intersect else 'не пересекаются'} "
        f"с вилкой вакансии ({v_salary_min}-{v_salary_max} руб.)."
    )

    same_location = c_location == v_location
    remote_ok = v_remote in {"Да", "Гибрид"} or c_remote in {"Да", "Гибрид"}
    location_score = 1.0 if same_location or remote_ok else 0.4
    if same_location:
        location_text = f"Локация совпадает: {c_location}."
    elif remote_ok:
        location_text = "Локация отличается, но удаленный/гибридный формат совместим."
    else:
        location_text = f"Локация не совпадает ({c_location} vs {v_location})."

    employment_score = 1.0 if c_employment == v_employment else 0.6
    score = int((skill_ratio * 0.45 + exp_score * 0.25 + salary_score * 0.15 + location_score * 0.1 + employment_score * 0.05) * 100)
    score = max(10, min(96, score))

    if score >= 80:
        rec = "hire"
    elif score >= 50:
        rec = "consider"
    else:
        rec = "reject"

    strengths = []
    weaknesses = []
    if common:
        strengths.append(f"Совпавшие навыки: {', '.join(common[:5])}")
    if c_exp >= v_min:
        strengths.append("Опыт соответствует минимальным требованиям")
    if intersect:
        strengths.append("Зарплатные ожидания в рабочем диапазоне")

    if missing:
        weaknesses.append(f"Недостающие навыки: {', '.join(missing[:5])}")
    if c_exp < v_min:
        weaknesses.append("Недостаточный опыт")
    if not (same_location or remote_ok):
        weaknesses.append("Несовпадение по локации/формату")

    user_prompt = (
        "Оцени соответствие кандидата вакансии и верни только JSON. "
        f"Кандидат: {candidate_title}, опыт {c_exp} лет, навыки {', '.join(c_skills)}, "
        f"зарплата {c_salary_min}-{c_salary_max}, локация {c_location}, формат {c_remote}, занятость {c_employment}. "
        f"Вакансия: {vacancy_title}, опыт {v_min}-{v_max} лет, навыки {', '.join(v_skills)}, "
        f"зарплата {v_salary_min}-{v_salary_max}, локация {v_location}, формат {v_remote}, занятость {v_employment}."
    )

    assistant_obj = {
        "analysis": {
            "skills_match": f"Совпавшие навыки ({len(common)}/{required_count}): {', '.join(common) if common else 'нет'}. Недостающие навыки: {', '.join(missing) if missing else 'нет'}.",
            "experience_match": exp_text,
            "salary_match": salary_text,
            "location_match": location_text,
            "strengths": strengths,
            "weaknesses": weaknesses,
            "recommendation": rec,
        },
        "confidence": round(0.55 + score / 250, 2),
        "match_score": score,
    }

    return {
        "messages": [
            {"role": "user", "content": user_prompt},
            {"role": "assistant", "content": json.dumps(assistant_obj, ensure_ascii=False)},
        ]
    }


samples = []
profile_names = list(PROFILES.keys())
for i in range(TARGET_SAMPLES):
    profile = random.choice(profile_names)
    hard_negative = (i % 4 == 0)
    samples.append(build_pair(profile, hard_negative=hard_negative))

random.shuffle(samples)
train_size = int(len(samples) * TRAIN_RATIO)
train_samples = samples[:train_size]
val_samples = samples[train_size:]

def to_chat_text(item):
    msgs = item["messages"]
    user_text = next(m["content"] for m in msgs if m["role"] == "user")
    assistant_text = next(m["content"] for m in msgs if m["role"] == "assistant")
    return (
        "<|start_header_id|>user<|end_header_id|>\n\n"
        + user_text
        + "<|eot_id|>"
        + "<|start_header_id|>assistant<|end_header_id|>\n\n"
        + assistant_text
        + "<|eot_id|>"
    )

train_texts = [to_chat_text(x) for x in train_samples]
val_texts = [to_chat_text(x) for x in val_samples]

train_dataset = Dataset.from_dict({"text": train_texts})
val_dataset = Dataset.from_dict({"text": val_texts})

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print("Sample preview:\n", train_dataset[0]["text"][:300])

Train samples: 540
Val samples: 60
Sample preview:
 <|start_header_id|>user<|end_header_id|>

Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 2 лет, навыки AsyncIO, Django, FastAPI, PostgreSQL, зарплата 114935-199320, локация Казань, формат Нет, занятость Полная. Вакансия: Python Backend Developer, 


In [4]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("Model + LoRA prepared")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.4.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Model + LoRA prepared


In [5]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq

num_samples = len(train_dataset)
effective_batch = 2 * 4
steps_per_epoch = max(1, num_samples // effective_batch)
max_steps = max(120, steps_per_epoch * 3)
warmup_steps = max(10, int(max_steps * 0.05))

print(f"steps_per_epoch={steps_per_epoch}, max_steps={max_steps}, warmup={warmup_steps}")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,  # Убедись, что val_dataset определён
    dataset_text_field="text",
    max_seq_length=1024,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    args=TrainingArguments(
        output_dir="outputs_full_dataset",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        max_steps=max_steps,
        warmup_steps=warmup_steps,
        learning_rate=1e-4,
        fp16=True,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        logging_steps=10,
        eval_strategy="steps",  # 🔥 Исправлено: evaluation_strategy → eval_strategy
        eval_steps=max(20, max_steps // 10),
        save_strategy="steps",
        save_steps=max(40, max_steps // 5),
        save_total_limit=2,
        report_to="none",
        seed=42,
    ),
)

print("🔥 Запуск обучения...")
trainer.train()
print("✅ Обучение завершено!")

steps_per_epoch=67, max_steps=201, warmup=10


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/540 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/60 [00:00<?, ? examples/s]

🔥 Запуск обучения...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 540 | Num Epochs = 3 | Total steps = 201
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
20,1.257574,0.786034
40,0.356579,0.308814
60,0.266418,0.261784
80,0.250956,0.247210
100,0.244784,0.240237
120,0.233346,0.231204
140,0.225347,0.220685
160,0.210701,0.211126
180,0.210752,0.206275
200,0.203393,0.205577


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

✅ Обучение завершено!


In [7]:
sample_prompt = (
    "Оцени соответствие кандидата вакансии и верни только JSON. "
    "Кандидат: Python Backend Developer, опыт 6 лет, навыки Python, FastAPI, PostgreSQL, Docker, Redis, "
    "зарплата 170000-240000, локация Москва, формат Гибрид, занятость Полная. "
    "Вакансия: Senior Python Developer, опыт 5-7 лет, навыки Python, FastAPI, PostgreSQL, Docker, Kafka, "
    "зарплата 190000-280000, локация Москва, формат Гибрид, занятость Полная."
)

FastLanguageModel.for_inference(model)
inputs = tokenizer(
    "<|start_header_id|>user<|end_header_id|>\n\n" + sample_prompt + "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n",
    return_tensors="pt"
).to(model.device)

out = model.generate(
    **inputs,
    max_new_tokens=220,
    temperature=0.1,
    do_sample=True,
)

print(tokenizer.decode(out[0], skip_special_tokens=False))

Both `max_new_tokens` (=220) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Оцени соответствие кандидата вакансии и верни только JSON. Кандидат: Python Backend Developer, опыт 6 лет, навыки Python, FastAPI, PostgreSQL, Docker, Redis, зарплата 170000-240000, локация Москва, формат Гибрид, занятость Полная. Вакансия: Senior Python Developer, опыт 5-7 лет, навыки Python, FastAPI, PostgreSQL, Docker, Kafka, зарплата 190000-280000, локация Москва, формат Гибрид, занятость Полная.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{"analysis": {"skills_match": "Совпавшие навыки (4/5): Python, FastAPI, PostgreSQL, Docker. Недостающие навыки: Kafka.", "experience_match": "Опыт кандидата 6 лет находится в диапазоне вакансии (5-7 лет).", "salary_match": "Ожидания кандидата (170000-240000 руб.) пересекаются с вилкой вакансии (190000-280000 руб.).", "location_match": "Локация совпадает: Москва.", "strengths": ["Совпавшие навыки: Python, FastAPI, PostgreSQL, Docker", "Опыт соответствует минимальным требованиям

In [8]:
# Optional: export to GGUF for Ollama
EXPORT_DIR = "ollama_export_full"
os.makedirs(EXPORT_DIR, exist_ok=True)
model.save_pretrained_gguf(EXPORT_DIR, tokenizer, quantization_method="q4_k_m")
print("GGUF export completed:", EXPORT_DIR)

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in ollama_export_full/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [04:25<00:00, 132.85s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:21<00:00, 70.56s/it]


Unsloth: Merge process complete. Saved to `/content/ollama_export_full`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['ollama_export_full_gguf/llama-3.2-3b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['ollama_export_full_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model ollama_export_full_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to ollama_export_full_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f ollama_export_full_gguf/Modelfile
GGUF export completed: ollama_export_full
